In [ ]:
module load apps/binapps/anaconda3/2022.10
micromamba env create -f scrna_env.yaml -y
conda activate scrna

## 人
### GSE104589


In [ ]:
# ==============================================
# GSE104589 表达矩阵处理 【适配config.py 规范版】
# 自动跳过样本元数据文件 | 仅处理基因表达矩阵 | 适配所有txt/txt.gz
# 路径统一调用config | 分类输出至processed/output
# 新增：输出 Ensembl ID 列，正确映射 symbol
# ==============================================
import pandas as pd
import numpy as np
from mygene import MyGeneInfo
import os
import glob

# ===================== 核心：从config导入标准化路径 =====================
from config import (
    HUMAN_LOOSE_VS_STABLE_GSE104589,
    DATA_PROCESSED,
    RESULT_DIR
)

# ===================== 自动创建GSE专属子文件夹 =====================
PROCESSED_GSE_DIR = os.path.join(DATA_PROCESSED, "GSE104589")
OUTPUT_GSE_DIR = os.path.join(RESULT_DIR, "GSE104589")
os.makedirs(PROCESSED_GSE_DIR, exist_ok=True)
os.makedirs(OUTPUT_GSE_DIR, exist_ok=True)

RAW_DATA_DIR = os.path.join(HUMAN_LOOSE_VS_STABLE_GSE104589, "GSE104589_RAW")
mg = MyGeneInfo()

# 9个样本匹配
SAMPLE_MAP = {
    "GSM2803868": "GSM2803868_Sample_1",
    "GSM2803869": "GSM2803869_Sample_2",
    "GSM2803870": "GSM2803870_Sample_3",
    "GSM2803871": "GSM2803871_Sample_4",
    "GSM2803872": "GSM2803872_Sample_5",
    "GSM2803873": "GSM2803873_Sample_6",
    "GSM3670779": "GSM3670779_Sample_7",
    "GSM3670780": "GSM3670780_Sample_8",
    "GSM3670781": "GSM3670781_Sample_9",
}

print("开始处理文件，自动跳过样本元数据，仅提取基因表达矩阵...")
expr_df = None

# ---------------------- 全自动读取：跳过错误文件，只留表达矩阵 ----------------------
for gsm, file_prefix in SAMPLE_MAP.items():
    pattern = os.path.join(RAW_DATA_DIR, f"{file_prefix}.txt*")
    matches = glob.glob(pattern)
    if not matches:
        print(f"⚠️ 未找到文件：{file_prefix}.txt*，跳过")
        continue
    file_path = matches[0]
    print(f"读取文件：{os.path.basename(file_path)}")

    compression = 'gzip' if file_path.endswith('.gz') else None
    df = pd.read_csv(
        file_path,
        sep="\t",
        compression=compression,
        header=0,
        encoding="latin-1",
        encoding_errors="replace",
        low_memory=False
    )
    df.columns = df.columns.str.strip()

    # 跳过样本元数据文件
    if "Sampletype" in df.columns or "Macrophage" in df.columns:
        print(f"⏭️  跳过样本元数据文件：{file_prefix}")
        continue

    # 仅处理基因表达文件（含tracking_id/FPKM）
    if "tracking_id" in df.columns and "FPKM" in df.columns:
        df = df.set_index("tracking_id")
        col = df["FPKM"].rename(gsm)

        if expr_df is None:
            expr_df = col.to_frame()
        else:
            expr_df = expr_df.join(col, how="inner")
        print(f"✅ 成功提取表达矩阵：{gsm}")
    else:
        print(f"⚠️ 文件 {file_prefix} 缺少 tracking_id 或 FPKM 列，跳过")

# ---------------------- 数据清洗与过滤 ----------------------
expr_df = expr_df.astype(np.float64).fillna(0)
# 过滤低表达基因（50%以上样本有表达）
threshold = expr_df.shape[1] * 0.5
df_filter = expr_df.loc[(expr_df > 0).sum(axis=1) >= threshold]

# ---------------------- 基因ID转换（新增 Ensembl ID） ----------------------
try:
    res = mg.querymany(
        df_filter.index.tolist(),
        scopes="refseq",
        species="human",
        fields=["symbol", "ensembl", "entrezgene"],
        returnall=True
    )
    
    symbol_map = {}
    ensembl_map = {}
    ensembl_map = {}
    for item in res.get('out', []):
        q = item.get('query')
        if not q:
            continue
        # 处理 symbol
        sym = item.get('symbol')
        if isinstance(sym, list):
            sym = sym[0] if sym else np.nan
        symbol_map[q] = sym if sym else np.nan
        
        # 处理 ensembl（可能是列表、字典或字符串）
        ensg_data = item.get('ensembl')
        if ensg_data is None:
            ensg = np.nan
        elif isinstance(ensg_data, list):
            if len(ensg_data) > 0 and isinstance(ensg_data[0], dict):
                ensg = ensg_data[0].get('gene', np.nan)
            else:
                ensg = np.nan
        elif isinstance(ensg_data, dict):
            ensg = ensg_data.get('gene', np.nan)
        elif isinstance(ensg_data, str):
            ensg = ensg_data
        else:
            ensg = np.nan
        ensembl_map[q] = ensg
    df_filter["symbol"] = df_filter.index.map(symbol_map)
    df_filter["ensembl_id"] = df_filter.index.map(ensembl_map)
    
    # 丢弃 symbol 为空的行
    df_filter = df_filter.dropna(subset=["symbol"])
    
    # 按 symbol 合并
    sample_cols = [c for c in df_filter.columns if c not in ['symbol', 'ensembl_id']]
    
    def first_valid(series):
        valid = series.dropna()
        return valid.iloc[0] if not valid.empty else np.nan
    
    agg_dict = {col: 'mean' for col in sample_cols}
    agg_dict['ensembl_id'] = first_valid
    
    df_final = df_filter.groupby("symbol").agg(agg_dict)
    df_final = df_final[['ensembl_id'] + sample_cols]
    
except Exception as e:
    df_final = df_filter
    print(f"⚠️ ID转换失败，保留原始基因ID。错误信息：{e}")

# ---------------------- 保存文件 ----------------------
# 1. 表达矩阵（行名 = symbol，包含 ensembl_id 列）
matrix_path = os.path.join(PROCESSED_GSE_DIR, "GSE104589_原始表达矩阵.csv")
df_final.to_csv(matrix_path, index=True, index_label="symbol", encoding="utf-8-sig")
print(f"✅ 表达矩阵已保存，形状：{df_final.shape}，包含 ensembl_id 列")

# 2. 样本分组文件（基于样本列名）
group_map = {
    "GSM2803868": "Control",
    "GSM2803869": "Control",
    "GSM2803870": "Control",
    "GSM2803871": "UHMWPE",
    "GSM2803872": "UHMWPE",
    "GSM2803873": "UHMWPE",
    "GSM3670779": "VE-UHMWPE",
    "GSM3670780": "VE-UHMWPE",
    "GSM3670781": "VE-UHMWPE"
}
# 提取样本列（排除 ensembl_id）
sample_cols_final = [c for c in df_final.columns if c != 'ensembl_id']
group_df = pd.DataFrame({
    "sample_id": sample_cols_final,
    "group": [group_map[i] for i in sample_cols_final],
    "group_num": [0 if group_map[i] == "Control" else 1 if group_map[i] == "UHMWPE" else 2 for i in sample_cols_final]
})
group_path = os.path.join(PROCESSED_GSE_DIR, "GSE104589_样本分组信息.csv")
group_df.to_csv(group_path, index=False, encoding="utf-8-sig")

# ---------------------- 完成 ----------------------
print("\n🎉🎉🎉 全部处理完成！")
print(f"📦 表达矩阵形状：{df_final.shape}")
print(f"📂 处理后数据路径：{PROCESSED_GSE_DIR}")
print(f"📂 分析结果路径：{OUTPUT_GSE_DIR}")
print("✅ 输出文件：")
print("   - GSE104589_原始表达矩阵.csv (行名=symbol, 列=ensembl_id + 样本)")
print("   - GSE104589_样本分组信息.csv")
print("⚠️  矩阵为原始FPKM，直接用于差异分析，禁止标准化！")

In [ ]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings("ignore")

from config import DATA_PROCESSED, RESULT_DIR

GSE_ID = "GSE104589"
PROCESSED_GSE_DIR = os.path.join(DATA_PROCESSED, GSE_ID)
OUTPUT_GSE_DIR = os.path.join(RESULT_DIR, GSE_ID)
os.makedirs(PROCESSED_GSE_DIR, exist_ok=True)
os.makedirs(OUTPUT_GSE_DIR, exist_ok=True)

import rpy2.robjects as ro
from rpy2.robjects import pandas2ri
from rpy2.robjects.conversion import localconverter
from rpy2.robjects.packages import importr, isinstalled

# 确保 limma 已安装
utils = importr('utils')
if not isinstalled('limma'):
    print("正在安装 limma ...")
    ro.r('if (!requireNamespace("BiocManager", quietly=TRUE)) install.packages("BiocManager")')
    ro.r('BiocManager::install("limma", update=FALSE, ask=FALSE)')

def diff_analysis_limma_ensembl(
    expr_csv: str,
    group_csv: str,
    control_group: str,
    treat_group: str,
    min_exp: float = 1,
    gse_id: str = GSE_ID,
    output_dir: str = OUTPUT_GSE_DIR
):
    # ===================== 1. 清空R环境 =====================
    ro.r('rm(list = ls())')
    ro.r('gc()')
    print("🔄 已清空R工作环境")

    # ===================== 2. 读取数据 =====================
    original_df = pd.read_csv(expr_csv)
    gene_dict = original_df.drop_duplicates('ensembl_id').set_index('ensembl_id')['symbol'].to_dict()
    
    expr = pd.read_csv(expr_csv, index_col=1)
    group = pd.read_csv(group_csv).copy()

    # 筛选样本
    target_samples = group[group["group"].isin([control_group, treat_group])]["sample_id"].tolist()
    sample_cols = [c for c in expr.columns if c != 'symbol' and c in target_samples]
    expr = expr[sample_cols].copy()
    group = group[group["sample_id"].isin(sample_cols)].copy()

    control_samples = group[group["group"] == control_group]["sample_id"].tolist()
    treat_samples = group[group["group"] == treat_group]["sample_id"].tolist()
    print(f"✅ 对照组：{control_group} ({len(control_samples)}个样本)")
    print(f"✅ 处理组：{treat_group} ({len(treat_samples)}个样本)")

    # 低表达过滤 + 删除空值基因ID
    expr = expr[expr.mean(axis=1) >= min_exp].copy()
    expr = expr[expr.index.notna()]
    print(f"📊 过滤后基因数：{expr.shape[0]}")

    # 强制转换为字符串，杜绝float类型
    gene_ids = [str(x) for x in expr.index.tolist()]
    sample_names = [str(x) for x in expr.columns.tolist()]

    # log2转换
    log_mat = np.log2(expr + 1).values

    # 设计矩阵
    design = pd.DataFrame({
        "Intercept": 1,
        "Treat": [1 if x in treat_samples else 0 for x in expr.columns]
    }, index=expr.columns)

    # ===================== 3. 传递数据 =====================
    ro.globalenv['expr_vals'] = ro.FloatVector(log_mat.flatten(order='F'))
    ro.globalenv['n_row'] = len(gene_ids)
    ro.globalenv['n_col'] = len(sample_names)
    ro.globalenv['gene_names'] = ro.StrVector(gene_ids)
    ro.globalenv['samp_names'] = ro.StrVector(sample_names)
    
    with localconverter(ro.default_converter + pandas2ri.converter):
        ro.globalenv['design_df'] = design

    # ===================== 4. R 核心代码 =====================
    r_code = '''
    library(limma)
    mat <- matrix(expr_vals, nrow = n_row, ncol = n_col, byrow = FALSE)
    rownames(mat) <- gene_names
    colnames(mat) <- samp_names
    cat("🐛 R中基因ID前5个：", head(rownames(mat)), "\\n")
    
    design_mat <- as.matrix(design_df)
    fit <- lmFit(mat, design_mat)
    cont <- makeContrasts(Group = Treat, levels = design_mat)
    fit2 <- contrasts.fit(fit, cont)
    fit2 <- eBayes(fit2)
    res <- topTable(fit2, number = Inf, sort.by = "none")
    res$ensembl_id <- rownames(res)
    res
    '''

    # 执行R代码
    res_r = ro.r(r_code)
    with localconverter(ro.default_converter + pandas2ri.converter):
        res_df = pandas2ri.rpy2py(res_r)

    # ===================== 5. 结果合并 =====================
    print("🐍 Python基因ID前5个：", gene_ids[:5])
    print("🐛 R返回基因ID前5个：", res_df['ensembl_id'].head().tolist())

    # 匹配symbol
    res_df['symbol'] = res_df['ensembl_id'].map(gene_dict).fillna('')

    # 计算均值
    expr['ctrl_mean'] = expr[control_samples].mean(axis=1)
    expr['treat_mean'] = expr[treat_samples].mean(axis=1)

    # 合并数据
    res_df = res_df.set_index('ensembl_id')
    common = expr.index.intersection(res_df.index)
    print(f"✅ 共同基因数：{len(common)}")

    if len(common) == 0:
        print("❌ 错误")
        return pd.DataFrame()

    # 生成最终结果
    result = pd.DataFrame({
        'ensembl_id': common,
        'control_mean': expr.loc[common, 'ctrl_mean'],
        'treat_mean': expr.loc[common, 'treat_mean'],
        'log2FoldChange': res_df.loc[common, 'logFC'],
        'P_value': res_df.loc[common, 'P.Value'],
        'FDR': res_df.loc[common, 'adj.P.Val'],
        'symbol': res_df.loc[common, 'symbol']
    })

    # ===================== 【修复核心】修正np.where语法错误 =====================
    result["regulate"] = np.where(
        (result["log2FoldChange"] > 1) & (result["FDR"] < 0.05), "Up",
        np.where((result["log2FoldChange"] < -1) & (result["FDR"] < 0.05), "Down", "No")
    )

    # 保存文件
    prefix = f"{gse_id}_{control_group}_vs_{treat_group}"
    result.to_csv(os.path.join(output_dir, f"{prefix}_差异结果.csv"), index=False, encoding='utf-8-sig')
    result[result.regulate=='Up'].to_csv(os.path.join(output_dir, f"{prefix}_上调.csv"), index=False, encoding='utf-8-sig')
    result[result.regulate=='Down'].to_csv(os.path.join(output_dir, f"{prefix}_下调.csv"), index=False, encoding='utf-8-sig')

    print(f"🎉 完成！上调：{sum(result.regulate=='Up')}，下调：{sum(result.regulate=='Down')}")
    print("-"*60)
    return result


if __name__ == "__main__":
    EXPR_PATH = os.path.join(PROCESSED_GSE_DIR, f"{GSE_ID}_原始表达矩阵.csv")
    GROUP_PATH = os.path.join(PROCESSED_GSE_DIR, f"{GSE_ID}_样本分组信息.csv")

    diff_analysis_limma_ensembl(EXPR_PATH, GROUP_PATH, "Control", "UHMWPE")
    diff_analysis_limma_ensembl(EXPR_PATH, GROUP_PATH, "Control", "VE-UHMWPE")

In [ ]:
# -*- coding: utf-8 -*-
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Patch

# ===================== 导入config路径配置 =====================
from config import DATA_PROCESSED, RESULT_DIR

# ===================== 基础配置 =====================
GSE_ID = "GSE104589"
# 输入输出路径（与差异分析完全一致）
PROCESSED_GSE_DIR = os.path.join(DATA_PROCESSED, GSE_ID)
OUTPUT_GSE_DIR = os.path.join(RESULT_DIR, GSE_ID)
os.makedirs(OUTPUT_GSE_DIR, exist_ok=True)

# 差异阈值
FC_THRESH = 1
FDR_THRESH = 0.05
# 火山图标注Top基因数量
TOP_GENES = 10

# 绘图配置
plt.rcParams['font.sans-serif'] = ['DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 300

# ===================== 火山图：显示 symbol 基因名 =====================
def plot_volcano(control, treat):
    file_prefix = f"{GSE_ID}_{control}_vs_{treat}"
    input_file = os.path.join(OUTPUT_GSE_DIR, f"{file_prefix}_差异结果.csv")
    save_name = f"{file_prefix}_火山图"
    title_name = f"Volcano Plot ({control} vs {treat})"
    
    # 读取数据（适配你的列名）
    df = pd.read_csv(input_file)
    df['FDR'] = df['FDR'].astype(float).replace(0, 1e-300)
    df['-log10(FDR)'] = -np.log10(df['FDR'])
    
    # 配色
    colors = {'Up': '#FF4C4C', 'Down': '#4C6EFF', 'No': '#B8B8B8'}
    
    plt.figure(figsize=(6, 5), dpi=300)
    sns.scatterplot(
        x='log2FoldChange', y='-log10(FDR)', hue='regulate',
        data=df, palette=colors, s=8, alpha=0.8, legend='full'
    )
    
    # 阈值线
    plt.axvline(x=FC_THRESH, color='gray', linestyle='--', lw=0.8)
    plt.axvline(x=-FC_THRESH, color='gray', linestyle='--', lw=0.8)
    plt.axhline(y=-np.log10(FDR_THRESH), color='gray', linestyle='--', lw=0.8)
    
    # 标注Top差异基因的 symbol
    up_genes = df[df['regulate'] == 'Up'].sort_values('log2FoldChange', ascending=False).head(TOP_GENES)
    down_genes = df[df['regulate'] == 'Down'].sort_values('log2FoldChange').head(TOP_GENES)
    
    for _, row in up_genes.iterrows():
        plt.annotate(row['symbol'], (row['log2FoldChange'], row['-log10(FDR)']),
                    fontsize=6, color='red', xytext=(5, 5), textcoords='offset points')
    for _, row in down_genes.iterrows():
        plt.annotate(row['symbol'], (row['log2FoldChange'], row['-log10(FDR)']),
                    fontsize=6, color='blue', xytext=(5, 5), textcoords='offset points')
    
    # 图表样式
    plt.title(title_name, fontsize=12, pad=15)
    plt.xlabel('log2(Fold Change)', fontsize=10)
    plt.ylabel('-log10(FDR)', fontsize=10)
    max_fc = df['log2FoldChange'].abs().max()
    xlim = max(6, max_fc + 0.5)
    plt.xlim(-xlim, xlim)
    plt.legend(title='Gene Type', loc='upper right')
    plt.tight_layout()
    
    # 保存
    plt.savefig(os.path.join(OUTPUT_GSE_DIR, f"{save_name}.png"), dpi=300, bbox_inches='tight')
    plt.close()
    
    up_num = len(df[df['regulate'] == 'Up'])
    down_num = len(df[df['regulate'] == 'Down'])
    print(f"✅ {title_name} | 上调:{up_num} | 下调:{down_num}")

# ===================== 热图：纵轴用 symbol 显示基因名 =====================
cmap = LinearSegmentedColormap.from_list('custom', ['#3A77AF', 'white', '#FF3D3D'])

def plot_heatmap(control, treat):
    file_prefix = f"{GSE_ID}_{control}_vs_{treat}"
    deg_file = os.path.join(OUTPUT_GSE_DIR, f"{file_prefix}_差异结果.csv")
    expr_path = os.path.join(PROCESSED_GSE_DIR, f"{GSE_ID}_原始表达矩阵.csv")
    group_path = os.path.join(PROCESSED_GSE_DIR, f"{GSE_ID}_样本分组信息.csv")
    
    # 读取数据 → 【核心修复】表达矩阵以 ensembl_id 为索引（匹配你的文件格式）
    deg = pd.read_csv(deg_file)
    # 你的表达矩阵：symbol,ensembl_id,GSMxxx → index_col="ensembl_id"
    expr = pd.read_csv(expr_path, index_col="ensembl_id")  
    group = pd.read_csv(group_path)
    
    # 筛选显著差异基因
    sig_deg = deg[(deg['FDR'] < FDR_THRESH) & (abs(deg['log2FoldChange']) > FC_THRESH)].copy()
    if len(sig_deg) == 0:
        print(f"❌ {control} vs {treat} 无显著差异基因")
        return
    print(f"✅ {control} vs {treat} 提取到 {len(sig_deg)} 个显著差异基因")
    
    # 获取显著基因的 ID 和名称映射
    sig_ensembl_list = sig_deg['ensembl_id'].tolist()
    symbol_map = dict(zip(sig_deg['ensembl_id'], sig_deg['symbol']))
    
    # 【修复完成】用 ensembl_id 索引表达矩阵，不会再报 KeyError
    heatmap_data = expr.loc[sig_ensembl_list, :].copy()
    # 行名替换为 symbol（图片显示基因名）
    heatmap_data.index = heatmap_data.index.map(symbol_map)
    
    # 移除表达矩阵中的 symbol 列（仅保留表达量）
    if 'symbol' in heatmap_data.columns:
        heatmap_data = heatmap_data.drop('symbol', axis=1)
    
    # 数据标准化
    heatmap_data = heatmap_data.apply(lambda x: (x - x.mean()) / x.std(), axis=1)
    
    # 样本分组颜色
    group_dict = dict(zip(group['sample_id'], group['group']))
    group_colors = {'Control': '#8DD3C7', 'UHMWPE': '#FFB347', 'VE-UHMWPE': '#BEBADA'}
    col_colors = [group_colors[group_dict[s]] for s in heatmap_data.columns]
    
    # 绘制热图
    g = sns.clustermap(
        heatmap_data,
        cmap=cmap,
        row_cluster=True, col_cluster=True,
        col_colors=col_colors,
        linewidths=0.05,
        figsize=(12, 10),
        dendrogram_ratio=(0.12, 0.1),
        cbar_kws={"label": "Z-score"}
    )
    
    # 基因名显示设置
    if len(sig_deg) > 100:
        g.ax_heatmap.set_yticklabels([])
    else:
        plt.setp(g.ax_heatmap.get_yticklabels(), fontsize=8)
    
    # 标题+图例
    g.fig.suptitle(f'{GSE_ID} DEG Heatmap ({control} vs {treat})', y=1.02, fontsize=14)
    legend_elements = [
        Patch(facecolor=group_colors['Control'], label='Control'),
        Patch(facecolor=group_colors['UHMWPE'], label='UHMWPE'),
        Patch(facecolor=group_colors['VE-UHMWPE'], label='VE-UHMWPE')
    ]
    g.ax_col_dendrogram.legend(handles=legend_elements, loc='upper right', bbox_to_anchor=(1.2, 1))
    
    # 保存
    save_path = os.path.join(OUTPUT_GSE_DIR, f"{file_prefix}_热图.png")
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"🎉 热图已保存：{save_path}")

# ===================== 一键运行 =====================
if __name__ == "__main__":
    print("🚀 开始绘制可视化图表...\n")
    CONTROL = "Control"
    TREAT_LIST = ["UHMWPE", "VE-UHMWPE"]
    
    for treat in TREAT_LIST:
        plot_volcano(CONTROL, treat)
        plot_heatmap(CONTROL, treat)
    
    print(f"\n🎉 所有图表绘制完成！")
    print(f"📂 保存路径：{OUTPUT_GSE_DIR}")


### GSE149315 

In [ ]:
# ==============================================
# GSE149315 circRNA表达矩阵处理（适配config.py 规范版）
# 使用 limma 标准流程（适用于芯片数据）
# 输出原始未标准化矩阵 | 支持Excel输入
# 路径统一调用config | 分类输出至processed/output
# 阈值：标准严格阈值 |log2FC|>1 + FDR<0.05
# ==============================================
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings("ignore")

# ---------------------- 核心：从config导入标准化路径 ----------------------
from config import (
    HUMAN_LOOSE_VS_STABLE_GSE149315,  # GSE149315原始数据路径
    DATA_PROCESSED,                   # 处理后数据总目录
    RESULT_DIR                        # 分析结果输出总目录
)

# ===================== 自动创建GSE专属子文件夹（核心修改）=====================
GSE_ID = "GSE149315"
# 处理后数据存放：data/processed/GSE149315（原始矩阵+分组信息）
PROCESSED_GSE_DIR = os.path.join(DATA_PROCESSED, GSE_ID)
# 差异分析结果存放：output/GSE149315（差异结果+上调/下调circRNA）
OUTPUT_GSE_DIR = os.path.join(RESULT_DIR, GSE_ID)

# 自动创建文件夹（不存在则新建，无需手动操作）
os.makedirs(PROCESSED_GSE_DIR, exist_ok=True)
os.makedirs(OUTPUT_GSE_DIR, exist_ok=True)

# 原始数据路径（直接用config定义的路径，拼接Excel文件名）
RAW_DATA_PATH = os.path.join(HUMAN_LOOSE_VS_STABLE_GSE149315, "GSE149315_Expression_circRNA.xlsx")
# ==================================================================

# ---------------------- rpy2 相关导入 ----------------------
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri
from rpy2.robjects.conversion import localconverter
from rpy2.robjects.packages import importr, isinstalled

# 确保 limma 已安装（自动安装）
utils = importr('utils')
if not isinstalled('limma'):
    print("正在安装 limma ...")
    utils.install_packages('BiocManager')
    ro.r('BiocManager::install("limma", update=FALSE, ask=FALSE)')

# ---------------------- 1. 读取并整理circRNA表达数据 ----------------------
print("开始读取circRNA表达数据...")
expr_df = pd.read_excel(RAW_DATA_PATH, index_col="circRNA_ID")

sample_cols = expr_df.columns.tolist()
print(f"检测到样本：{sample_cols}")
print(f"原始数据形状：{expr_df.shape}（circRNA数量 × 样本数量）")

# ---------------------- 2. 样本分组定义 ----------------------
group_map = {
    "T1": "Loose", "T2": "Loose", "T3": "Loose",
    "N1": "Stable", "N2": "Stable", "N3": "Stable"
}
group_num_map = {"Stable": 0, "Loose": 1}

if not all(col in group_map for col in sample_cols):
    raise ValueError("部分样本未定义分组！请检查group_map与样本列是否匹配")

# ---------------------- 3. 数据清洗与低表达过滤 ----------------------
print("开始数据清洗与低表达过滤...")
expr_df = expr_df.astype(np.float64).fillna(0)

# 芯片数据过滤：保留至少在50%样本中表达值 > 0 的circRNA
sample_count = expr_df.shape[1]
threshold = sample_count * 0.5
df_filter = expr_df.loc[(expr_df > 0).sum(axis=1) >= threshold]

print(f"过滤后数据形状：{df_filter.shape}（保留的circRNA数量 × 样本数量）")
print(f"过滤掉低表达circRNA数量：{expr_df.shape[0] - df_filter.shape[0]}")

df_final = df_filter.copy()
print("保留circRNA ID作为行名，直接用于差异分析")

# ---------------------- 4. 准备设计矩阵 ----------------------
group_labels = [group_map[col] for col in df_final.columns]
# 设计矩阵：截距 + 分组（Loose vs Stable）
design_matrix = pd.DataFrame({
    'Intercept': 1,
    'Loose_vs_Stable': [1 if g == 'Loose' else 0 for g in group_labels]
}).astype(float)

# ---------------------- 5. 差异分析（标准 limma 流程，无 trend） ----------------------
print("\n开始差异分析（limma 标准流程，适用于芯片数据）...")

with localconverter(ro.default_converter + pandas2ri.converter):
    # 将数据传入 R 全局环境（自动转换为 R 对象）
    ro.globalenv['expr_mat'] = df_final
    ro.globalenv['design_mat'] = design_matrix
    
    # 执行 R 代码
    ro.r('''
        library(limma)
        # 线性拟合
        fit <- lmFit(expr_mat, design_mat)
        # 经验贝叶斯收缩（标准方法，不设 trend）
        fit <- eBayes(fit)
        # 提取对比组结果（coef=2 对应 Loose_vs_Stable）
        res <- topTable(fit, coef=2, number=nrow(expr_mat), sort.by="none")
    ''')
    
    # 获取结果（自动转换为 pandas DataFrame）
    res_df = ro.globalenv['res']

# 提取所需列
log2fc = res_df['logFC'].values
p_values = res_df['P.Value'].values
fdr = res_df['adj.P.Val'].values

# 整合差异分析结果
diff_result = pd.DataFrame({
    "circRNA_ID": df_final.index,
    "log2FC": log2fc,
    "p_value": p_values,
    "FDR": fdr
}).set_index("circRNA_ID")

# ---------------------- 6. 筛选显著上调/下调circRNA ----------------------
up_circ = diff_result[(diff_result["log2FC"] > 1) & (diff_result["FDR"] < 0.05)]
down_circ = diff_result[(diff_result["log2FC"] < -1) & (diff_result["FDR"] < 0.05)]

print(f"\n📊 显著差异circRNA统计（标准严格阈值 |log2FC|>1 + FDR<0.05）：")
print(f"✅ 上调circRNA数量：{len(up_circ)}")
print(f"✅ 下调circRNA数量：{len(down_circ)}")

# ---------------------- 7. 保存所有结果文件（严格分类存放）----------------------
print("\n保存表达矩阵、分组信息、差异结果、上调/下调circRNA...")

# 7.1 原始表达矩阵 → 存入 processed/GSE149315
matrix_path = os.path.join(PROCESSED_GSE_DIR, f"{GSE_ID}_circRNA_原始表达矩阵.csv")
df_final.to_csv(matrix_path, index=True, encoding="utf-8-sig")

# 7.2 样本分组信息 → 存入 processed/GSE149315
group_path = os.path.join(PROCESSED_GSE_DIR, f"{GSE_ID}_样本分组信息.csv")
group_df = pd.DataFrame({
    "sample_id": df_final.columns,
    "group": [group_map[col] for col in df_final.columns],
    "group_num": [group_num_map[group_map[col]] for col in df_final.columns]
})
group_df.to_csv(group_path, index=False, encoding="utf-8-sig")

# 7.3 完整差异分析结果 → 存入 output/GSE149315
diff_path = os.path.join(OUTPUT_GSE_DIR, f"{GSE_ID}_circRNA_差异分析结果.csv")
diff_result.to_csv(diff_path, index=True, encoding="utf-8-sig")
print(f"✅ 完整差异分析结果已保存：{diff_path}")

# 7.4 显著上调circRNA → 存入 output/GSE149315
up_path = os.path.join(OUTPUT_GSE_DIR, f"{GSE_ID}_显著上调circRNA.csv")
up_circ.to_csv(up_path, encoding="utf-8-sig")

# 7.5 显著下调circRNA → 存入 output/GSE149315
down_path = os.path.join(OUTPUT_GSE_DIR, f"{GSE_ID}_显著下调circRNA.csv")
down_circ.to_csv(down_path, encoding="utf-8-sig")

# ---------------------- 完成 ----------------------
print("\n🎉🎉🎉 全部处理完成（使用 limma 标准流程，适用于芯片数据）！")
print(f"📦 处理后数据（原始矩阵/分组）保存路径：{PROCESSED_GSE_DIR}")
print(f"📦 差异分析结果（上调/下调）保存路径：{OUTPUT_GSE_DIR}")
print(f"📄 文件列表：")
print(f"   1. 原始表达矩阵：{GSE_ID}_circRNA_原始表达矩阵.csv")
print(f"   2. 样本分组信息：{GSE_ID}_样本分组信息.csv")
print(f"   3. 完整差异结果：{GSE_ID}_circRNA_差异分析结果.csv")
print(f"   4. 显著上调circRNA：{GSE_ID}_显著上调circRNA.csv")
print(f"   5. 显著下调circRNA：{GSE_ID}_显著下调circRNA.csv")
print("📊 差异分析方法：limma (linear model + empirical Bayes), 适用于芯片表达数据")

In [ ]:
# ==============================================
# GSE149315 circRNA 火山图 + 热图绘制
# 基于已有的差异分析结果文件 | 适配 config.py 路径规范
# ==============================================
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from adjustText import adjust_text

# ===================== 导入 config 路径配置（核心修改） =====================
from config import DATA_PROCESSED, RESULT_DIR

# ===================== 路径配置（自动生成GSE专属目录） =====================
GSE_ID = "GSE149315"
# 输入：处理后数据（原始表达矩阵+分组信息）
PROCESSED_GSE_DIR = os.path.join(DATA_PROCESSED, GSE_ID)
# 输入/输出：差异结果+可视化图片
OUTPUT_GSE_DIR = os.path.join(RESULT_DIR, GSE_ID)

# 自动创建文件夹
os.makedirs(OUTPUT_GSE_DIR, exist_ok=True)

# 定义文件路径（严格分类读取）
diff_file = os.path.join(OUTPUT_GSE_DIR, f"{GSE_ID}_circRNA_差异分析结果.csv")       # 差异结果
expr_file = os.path.join(PROCESSED_GSE_DIR, f"{GSE_ID}_circRNA_原始表达矩阵.csv")   # 表达矩阵
group_file = os.path.join(PROCESSED_GSE_DIR, f"{GSE_ID}_样本分组信息.csv")           # 分组信息

# 阈值参数
FC_THRESH = 1.0
FDR_THRESH = 0.05

# ===================== 1. 读取数据 =====================
print("读取差异分析结果...")
diff_result = pd.read_csv(diff_file, index_col=0)

print("读取原始表达矩阵...")
expr_df = pd.read_csv(expr_file, index_col=0)

print("读取样本分组信息...")
group_df = pd.read_csv(group_file)

# 确保样本顺序一致
expr_df = expr_df[group_df["sample_id"]]

# 构建分组字典
group_map = dict(zip(group_df["sample_id"], group_df["group"]))
group_num_map = {"Stable": 0, "Loose": 1}

# ===================== 2. 定义显著性标签 =====================
diff_result["is_significant"] = (abs(diff_result["log2FC"]) > FC_THRESH) & (diff_result["FDR"] < FDR_THRESH)
significant_circ = diff_result["is_significant"].sum()
print(f"显著差异circRNA数量：{significant_circ}")

# ===================== 3. 绘制火山图 =====================
print("\n绘制火山图...")
plt.figure(figsize=(10, 6))
plt.rcParams["font.sans-serif"] = ["Arial"]

non_sig = diff_result[~diff_result["is_significant"]]
sig_up = diff_result[(diff_result["is_significant"]) & (diff_result["log2FC"] > 0)]
sig_down = diff_result[(diff_result["is_significant"]) & (diff_result["log2FC"] < 0)]

plt.scatter(non_sig["log2FC"], -np.log10(non_sig["FDR"]), c="gray", alpha=0.5, s=20, label="NS")
plt.scatter(sig_up["log2FC"], -np.log10(sig_up["FDR"]), c="red", alpha=0.8, s=30, label=f"Up ({len(sig_up)})")
plt.scatter(sig_down["log2FC"], -np.log10(sig_down["FDR"]), c="blue", alpha=0.8, s=30, label=f"Down ({len(sig_down)})")

# 阈值线
plt.axvline(x=FC_THRESH, color="black", linestyle="--", alpha=0.5)
plt.axvline(x=-FC_THRESH, color="black", linestyle="--", alpha=0.5)
plt.axhline(y=-np.log10(FDR_THRESH), color="black", linestyle="--", alpha=0.5)

# 标注Top10显著circRNA（按FDR排序）
top10 = diff_result.sort_values("FDR").head(10)
texts = []
for circ_id in top10.index:
    x = top10.loc[circ_id, "log2FC"]
    y = -np.log10(top10.loc[circ_id, "FDR"])
    label = circ_id if len(circ_id) <= 30 else circ_id[:27] + "..."
    texts.append(plt.text(x, y, label, fontsize=8))
adjust_text(texts, arrowprops=dict(arrowstyle="->", color="black", alpha=0.3))

plt.xlabel("log2(Fold Change)", fontsize=12)
plt.ylabel("-log10(FDR)", fontsize=12)
plt.title(f"{GSE_ID} circRNA Volcano Plot (Loose vs Stable)", fontsize=14, fontweight="bold")
plt.legend()
plt.grid(alpha=0.3)

# 保存火山图至 output/GSE149315
volcano_path = os.path.join(OUTPUT_GSE_DIR, f"{GSE_ID}_circRNA_火山图.png")
plt.tight_layout()
plt.savefig(volcano_path, dpi=300, bbox_inches="tight")
plt.close()
print(f"火山图已保存：{volcano_path}")

# ===================== 4. 绘制热图（Top50差异circRNA） =====================
if significant_circ >= 1:
    print("\n绘制热图...")
    top_n = min(50, diff_result.shape[0])
    top_circ = diff_result.sort_values("FDR").head(top_n).index
    heat_data = expr_df.loc[top_circ].copy()
    
    # Z-score 标准化
    heat_scaled = (heat_data - heat_data.mean(axis=1).values.reshape(-1,1)) / heat_data.std(axis=1).values.reshape(-1,1)
    
    # 样本分组颜色条
    sample_colors = [plt.cm.Set3(group_num_map[group_map[col]]) for col in heat_data.columns]
    
    # 绘制聚类热图
    g = sns.clustermap(heat_scaled, cmap="RdBu_r", row_cluster=True, col_cluster=False,
                       col_colors=sample_colors,
                       cbar_kws={"label": "Z-score"}, linewidths=0.1, figsize=(12, 10))
    g.ax_col_dendrogram.set_visible(False)
    g.fig.suptitle(f"Top {top_n} Differentially Expressed circRNA", y=1.02)
    
    # 保存热图至 output/GSE149315
    heatmap_path = os.path.join(OUTPUT_GSE_DIR, f"{GSE_ID}_circRNA_热图.png")
    plt.savefig(heatmap_path, dpi=300, bbox_inches="tight")
    plt.close()
    print(f"热图已保存：{heatmap_path}")
else:
    print("无显著差异circRNA，跳过热图绘制")

print("\n🎉 可视化完成！")
print(f"📂 图片保存路径：{OUTPUT_GSE_DIR}")

In [ ]:
import pandas as pd
import os
import warnings
from mygene import MyGeneInfo

warnings.filterwarnings('ignore')

# ===================== 导入 config 路径配置 =====================
from config import (
    RESULT_DIR,
    HUMAN_LOOSE_VS_STABLE_GSE149315
)

# ===================== 自动配置路径 =====================
GSE_ID = "GSE149315"
OUTPUT_GSE_DIR = os.path.join(RESULT_DIR, GSE_ID)
os.makedirs(OUTPUT_GSE_DIR, exist_ok=True)

UP_FILE = os.path.join(OUTPUT_GSE_DIR, f"{GSE_ID}_显著上调circRNA.csv")
DOWN_FILE = os.path.join(OUTPUT_GSE_DIR, f"{GSE_ID}_显著下调circRNA.csv")
OUT_UP = os.path.join(OUTPUT_GSE_DIR, f"{GSE_ID}_显著上调circRNA_GeneSymbol_Ensembl.csv")
OUT_DOWN = os.path.join(OUTPUT_GSE_DIR, f"{GSE_ID}_显著下调circRNA_GeneSymbol_Ensembl.csv")
MAP_FILE = os.path.join(HUMAN_LOOSE_VS_STABLE_GSE149315, "hsa_hg19_circRNA.txt")

# ===================== 1. 加载 circRNA → Gene Symbol 映射 =====================
print("📥 正在加载本地 circRNA 映射表...")
col_names = [
    'chrom', 'start', 'end', 'strand', 'circRNA ID',
    'genomic length', 'spliced seq length', 'samples', 'repeats',
    'annotation', 'best transcript', 'gene symbol', 'circRNA study'
]
mapping_df = pd.read_csv(MAP_FILE, sep="\t", comment='#', names=col_names, header=None)
circ_to_gene = dict(zip(mapping_df['circRNA ID'], mapping_df['gene symbol']))
print(f"✅ 成功加载 {len(circ_to_gene)} 条 circRNA→Gene 映射记录")

# ===================== 2. 获取所有唯一 Gene Symbol 并查询 Ensembl ID =====================
all_genes = set(circ_to_gene.values())
print(f"📊 共涉及 {len(all_genes)} 个唯一基因符号，正在查询 Ensembl ID ...")

mg = MyGeneInfo()
# 批量查询（一次最多 1000 个，这里分块处理，但一般不会超过）
gene_list = list(all_genes)
# 去除可能的空值或 "."
gene_list = [g for g in gene_list if g and g != '.']
# 分批查询（mygene 限制一次最多 1000）
ensembl_dict = {}
batch_size = 1000
for i in range(0, len(gene_list), batch_size):
    batch = gene_list[i:i+batch_size]
    try:
        res = mg.querymany(batch, scopes='symbol', species='human', fields='ensembl', returnall=True)
        for item in res.get('out', []):
            symbol = item.get('query')
            ensg = item.get('ensembl')
            if isinstance(ensg, dict):
                ensg = ensg.get('gene')
            elif isinstance(ensg, list):
                ensg = ensg[0] if ensg else None
            if symbol and ensg:
                ensembl_dict[symbol] = ensg
        print(f"  批次 {i//batch_size+1} 完成，已获取 {len(ensembl_dict)} 个 Ensembl ID")
    except Exception as e:
        print(f"  ⚠️ 查询失败: {e}")

print(f"✅ 成功获取 {len(ensembl_dict)} 个基因的 Ensembl ID")

# ===================== 3. 批量处理函数 =====================
def process_file(input_path, output_path):
    df = pd.read_csv(input_path)
    print(f"📂 读取文件：{input_path}")
    print(f"🔢 待转换 circRNA 数量：{len(df)}")

    circ_col = "circRNA_ID" if "circRNA_ID" in df.columns else df.columns[0]
    circ_ids = df[circ_col].tolist()

    genes = []
    ensembl_ids = []
    for idx, circ_id in enumerate(circ_ids, 1):
        gene = circ_to_gene.get(circ_id, "Not_Found")
        ensg = ensembl_dict.get(gene, "") if gene != "Not_Found" else ""
        genes.append(gene)
        ensembl_ids.append(ensg)
        if idx % 500 == 0:
            print(f"  已处理 {idx}/{len(circ_ids)} ...")

    df['gene_symbol'] = genes
    df['ensembl_id'] = ensembl_ids
    df.to_csv(output_path, index=False, encoding='utf-8')
    print(f"✅ 转换完成！文件保存至：{output_path}\n")

# ===================== 4. 主程序 =====================
if __name__ == "__main__":
    print("="*60)
    print("🚀 circRNA ID → 宿主基因 + Ensembl ID 转换工具")
    print("="*60)
    process_file(UP_FILE, OUT_UP)
    process_file(DOWN_FILE, OUT_DOWN)
    print("🎉 全部转换完成！")
    print(f"📂 结果文件保存路径：{OUTPUT_GSE_DIR}")

### GSE171542

In [ ]:
import pandas as pd
import numpy as np
from mygene import MyGeneInfo
import os
import warnings
warnings.filterwarnings('ignore')

# ---------------------- 核心：从config导入标准化路径 ----------------------
from config import (
    HUMAN_LOOSE_VS_STABLE_GSE171542,
    DATA_PROCESSED,
    RESULT_DIR
)

# ===================== 路径配置（完全不变）=====================
GSE_ID = "GSE171542"
PROCESSED_GSE_DIR = os.path.join(DATA_PROCESSED, GSE_ID)
OUTPUT_GSE_DIR = os.path.join(RESULT_DIR, GSE_ID)
os.makedirs(PROCESSED_GSE_DIR, exist_ok=True)
os.makedirs(OUTPUT_GSE_DIR, exist_ok=True)

RAW_DATA_DIR = os.path.join(HUMAN_LOOSE_VS_STABLE_GSE171542, "GSE171542_RAW")
# ==================================================================

# ---------------------- rpy2 相关导入 ----------------------
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri
from rpy2.robjects.conversion import localconverter
from rpy2.robjects.packages import importr, isinstalled

# 确保 limma 已安装
utils = importr('utils')
if not isinstalled('limma'):
    print("正在安装 limma ...")
    utils.install_packages('BiocManager')
    ro.r('BiocManager::install("limma", update=FALSE, ask=FALSE)')

mg = MyGeneInfo()

# ---------------------- 样本映射与分组（完全不变）--------------------
SAMPLE_GROUP = {
    "GSM5226831": "Control",
    "GSM5226832": "Control",
    "GSM5226833": "Control",
    "GSM5226834": "RANKL",
    "GSM5226835": "RANKL",
    "GSM5226836": "RANKL",
    "GSM5226837": "TYMP",
    "GSM5226838": "TYMP",
    "GSM5226839": "TYMP",
}

file_map = {
    "GSM5226831": "GSM5226831_MCSF1_trim.isoforms.results.txt.gz",
    "GSM5226832": "GSM5226832_MCSF2_trim.isoforms.results.txt.gz",
    "GSM5226833": "GSM5226833_MCSF4_trim.isoforms.results.txt.gz",
    "GSM5226834": "GSM5226834_RANKL1_trim.isoforms.results.txt.gz",
    "GSM5226835": "GSM5226835_RANKL2_trim.isoforms.results.txt.gz",
    "GSM5226836": "GSM5226836_RANKL3_trim.isoforms.results.txt.gz",
    "GSM5226837": "GSM5226837_TP1_trim.isoforms.results.txt.gz",
    "GSM5226838": "GSM5226838_TP2_trim.isoforms.results.txt.gz",
    "GSM5226839": "GSM5226839_TP3_trim.isoforms.results.txt.gz",
}

MIN_EXP_SAMPLES = 2
P_THRESHOLD = 0.05
LOG2FC_THRESHOLD = 1.0

# ---------------------- 1. 读取数据构建 FPKM 矩阵（索引=Ensembl ID）----------------------
print("="*50)
print("读取原始 isoforms 文件...")
expr_df = None
for gsm, group in SAMPLE_GROUP.items():
    file_path = os.path.join(RAW_DATA_DIR, file_map[gsm])
    df = pd.read_csv(file_path, sep="\t", compression="gzip", low_memory=False)
    # 按gene_id(Ensembl)汇总FPKM，索引=Ensembl ID
    gene_level = df.groupby("gene_id")["FPKM"].sum().rename(gsm)
    expr_df = gene_level.to_frame() if expr_df is None else expr_df.join(gene_level, how="inner")

print(f"原始矩阵形状：{expr_df.shape}")

# ---------------------- 2. 过滤低表达基因（保留Ensembl ID索引）----------------------
expr_filter = expr_df.loc[(expr_df > 0).sum(axis=1) >= MIN_EXP_SAMPLES]
expr_filter = expr_filter.fillna(0).astype(float)
print(f"过滤后形状：{expr_filter.shape}")

# ---------------------- 3. 基因ID转换（保留Ensembl ID，新增symbol列，不合并）---------------------
print("转换基因 ID...")
gene_ids = expr_filter.index.tolist()
res = mg.querymany(gene_ids, scopes=["ensembl.gene"], species="human", fields="symbol", returnall=False)
id_map = {item["query"]: item.get("symbol", item["query"]) for item in res}

# 【核心修改】添加symbol列，**保留Ensembl ID为索引**，不合并、不丢失
expr_filter["symbol"] = expr_filter.index.map(id_map)
# 去除无symbol的行
expr_final = expr_filter.dropna(subset=["symbol"]).copy()

print(f"最终基因数(Ensembl ID)：{expr_final.shape[0]}")

# ---------------------- 4. 分别进行 limma 差异分析（基于Ensembl ID）----------------------
control_samples = [gsm for gsm, g in SAMPLE_GROUP.items() if g == "Control"]
rankl_samples = [gsm for gsm, g in SAMPLE_GROUP.items() if g == "RANKL"]
tymp_samples = [gsm for gsm, g in SAMPLE_GROUP.items() if g == "TYMP"]

def run_limma(expr_sub, gene_symbols, group_labels, contrast_name):
    """【修改】基于Ensembl ID执行limma，返回双ID结果"""
    # 提取表达矩阵（索引=Ensembl ID）
    expr_values = expr_sub.drop(columns=["symbol"])
    expr_log2 = np.log2(expr_values + 1)
    
    # 设计矩阵
    design = pd.DataFrame({
        'Intercept': 1,
        contrast_name: [1 if g == contrast_name else 0 for g in group_labels]
    })
    
    with localconverter(ro.default_converter + pandas2ri.converter):
        ro.globalenv['expr_mat'] = expr_log2
        ro.globalenv['design_mat'] = design
        ro.r('''
            library(limma)
            fit <- lmFit(expr_mat, design_mat)
            fit <- eBayes(fit)
            res <- topTable(fit, coef=2, number=nrow(expr_mat), sort.by="none")
        ''')
        res_df = ro.globalenv['res']
    
    # 【核心修改】合并 Ensembl ID + symbol + 差异结果
    diff_result = pd.DataFrame({
        "ensembl_id": expr_values.index,        # 保留Ensembl ID
        "symbol": gene_symbols,                 # 保留symbol
        "log2FoldChange": res_df['logFC'].values,
        "pvalue": res_df['P.Value'].values,
        "padj": res_df['adj.P.Val'].values,
    })
    diff_result["abs_log2FC"] = np.abs(diff_result["log2FoldChange"])
    return diff_result

# 4.1 RANKL vs Control
print("\n分析 RANKL vs Control...")
expr_rankl = expr_final[control_samples + rankl_samples + ["symbol"]]
group_rankl = ['Control']*3 + ['RANKL']*3
diff_rankl = run_limma(expr_rankl, expr_rankl['symbol'].values, group_rankl, "RANKL")

up_rankl = diff_rankl[(diff_rankl["padj"] < P_THRESHOLD) & (diff_rankl["log2FoldChange"] > LOG2FC_THRESHOLD)]
down_rankl = diff_rankl[(diff_rankl["padj"] < P_THRESHOLD) & (diff_rankl["log2FoldChange"] < -LOG2FC_THRESHOLD)]
print(f"  上调基因：{len(up_rankl)}，下调基因：{len(down_rankl)}")

# 4.2 TYMP vs Control
print("\n分析 TYMP vs Control...")
expr_tymp = expr_final[control_samples + tymp_samples + ["symbol"]]
group_tymp = ['Control']*3 + ['TYMP']*3
diff_tymp = run_limma(expr_tymp, expr_tymp['symbol'].values, group_tymp, "TYMP")

up_tymp = diff_tymp[(diff_tymp["padj"] < P_THRESHOLD) & (diff_tymp["log2FoldChange"] > LOG2FC_THRESHOLD)]
down_tymp = diff_tymp[(diff_tymp["padj"] < P_THRESHOLD) & (diff_tymp["log2FoldChange"] < -LOG2FC_THRESHOLD)]
print(f"  上调基因：{len(up_tymp)}，下调基因：{len(down_tymp)}")

# ---------------------- 5. 保存结果（路径/文件名完全不变，内容含双ID）----------------------
# 1. 原始表达矩阵（含ensembl_id索引 + symbol列）
expr_final.to_csv(os.path.join(PROCESSED_GSE_DIR, f"{GSE_ID}_原始表达矩阵(FPKM).csv"), index=True, encoding="utf-8-sig")

# 2. 样本分组信息（不变）
group_df = pd.DataFrame({
    "sample_id": expr_final.drop(columns=["symbol"]).columns,
    "group": [SAMPLE_GROUP[col] for col in expr_final.drop(columns=["symbol"]).columns]
})
group_df.to_csv(os.path.join(PROCESSED_GSE_DIR, f"{GSE_ID}_样本分组信息.csv"), index=False, encoding="utf-8-sig")

# 3. RANKL vs Control 结果（含ensembl_id + symbol）
diff_rankl.to_csv(os.path.join(OUTPUT_GSE_DIR, f"{GSE_ID}_差异分析结果_RANKL_vs_Control.csv"), index=False, encoding="utf-8-sig")
up_rankl.sort_values("log2FoldChange", ascending=False).to_csv(os.path.join(OUTPUT_GSE_DIR, f"{GSE_ID}_上调基因_RANKL_vs_Control.csv"), index=False, encoding="utf-8-sig")
down_rankl.sort_values("log2FoldChange", ascending=True).to_csv(os.path.join(OUTPUT_GSE_DIR, f"{GSE_ID}_下调基因_RANKL_vs_Control.csv"), index=False, encoding="utf-8-sig")

# 4. TYMP vs Control 结果（含ensembl_id + symbol）
diff_tymp.to_csv(os.path.join(OUTPUT_GSE_DIR, f"{GSE_ID}_差异分析结果_TYMP_vs_Control.csv"), index=False, encoding="utf-8-sig")
up_tymp.sort_values("log2FoldChange", ascending=False).to_csv(os.path.join(OUTPUT_GSE_DIR, f"{GSE_ID}_上调基因_TYMP_vs_Control.csv"), index=False, encoding="utf-8-sig")
down_tymp.sort_values("log2FoldChange", ascending=True).to_csv(os.path.join(OUTPUT_GSE_DIR, f"{GSE_ID}_下调基因_TYMP_vs_Control.csv"), index=False, encoding="utf-8-sig")

# ---------------------- 完成 ----------------------
print("\n🎉 完成！基于 Ensembl ID 完成差异分析，结果保留双ID")
print(f"📦 处理后数据保存路径：{PROCESSED_GSE_DIR}")
print(f"📦 差异分析结果保存路径：{OUTPUT_GSE_DIR}")

In [ ]:
# ===================== 火山图 + 热图绘制代码（适配 Ensembl ID 分析结果）=====================
import matplotlib.pyplot as plt
import seaborn as sns
from adjustText import adjust_text
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

# ===================== 导入 config 路径配置 =====================
from config import DATA_PROCESSED, RESULT_DIR

# ===================== 自动配置路径（与差异分析完全一致）=====================
GSE_ID = "GSE171542"
PROCESSED_GSE_DIR = os.path.join(DATA_PROCESSED, GSE_ID)
OUTPUT_GSE_DIR = os.path.join(RESULT_DIR, GSE_ID)
os.makedirs(OUTPUT_GSE_DIR, exist_ok=True)

# 绘图参数
P_THRESHOLD = 0.05
LOG2FC_THRESHOLD = 1.0

# 定义两个对比组（文件名100%匹配差异分析输出）
contrasts = [
    {
        "name": "RANKL_vs_Control",
        "diff_file": os.path.join(OUTPUT_GSE_DIR, f"{GSE_ID}_差异分析结果_RANKL_vs_Control.csv"),
        "title": "RANKL vs Control"
    },
    {
        "name": "TYMP_vs_Control",
        "diff_file": os.path.join(OUTPUT_GSE_DIR, f"{GSE_ID}_差异分析结果_TYMP_vs_Control.csv"),
        "title": "TYMP vs Control"
    }
]

# 读取数据（适配新格式：表达矩阵索引=ensembl_id，含symbol列）
expr_file = os.path.join(PROCESSED_GSE_DIR, f"{GSE_ID}_原始表达矩阵(FPKM).csv")
group_file = os.path.join(PROCESSED_GSE_DIR, f"{GSE_ID}_样本分组信息.csv")

# 核心：表达矩阵索引为 ensembl_id
expr_final = pd.read_csv(expr_file, index_col=0)
group_df = pd.read_csv(group_file)
GROUP_MAP = dict(zip(group_df["sample_id"], group_df["group"]))

# -----------------------------------------------------------------------------
# 循环处理每个对比组
# -----------------------------------------------------------------------------
for contrast in contrasts:
    print(f"\n📌 处理对比：{contrast['title']}")
    diff_file = contrast["diff_file"]
    
    if not os.path.exists(diff_file):
        print(f"⚠️ 差异文件不存在，跳过：{diff_file}")
        continue
    
    # 读取差异结果（含 ensembl_id + symbol 双列）
    diff_result = pd.read_csv(diff_file)
    # 过滤无效值
    diff_plot = diff_result.replace([np.inf, -np.inf], np.nan).dropna(subset=["log2FoldChange", "padj"]).copy()
    
    # 分类基因（使用symbol显示，ensembl_id匹配）
    ns = diff_plot[~((diff_plot["padj"] < P_THRESHOLD) & (abs(diff_plot["log2FoldChange"]) > LOG2FC_THRESHOLD))]
    up = diff_plot[(diff_plot["padj"] < P_THRESHOLD) & (diff_plot["log2FoldChange"] > LOG2FC_THRESHOLD)]
    down = diff_plot[(diff_plot["padj"] < P_THRESHOLD) & (diff_plot["log2FoldChange"] < -LOG2FC_THRESHOLD)]
    
    print(f"  上调基因数：{len(up)}，下调基因数：{len(down)}")
    
    # -------------------------------------------------------------------------
    # 1. 火山图（直接标注 symbol 基因名）
    # -------------------------------------------------------------------------
    plt.figure(figsize=(10, 6))
    plt.scatter(ns["log2FoldChange"], -np.log10(ns["padj"]), c="gray", s=20, alpha=0.5, label="NS")
    if len(up) > 0:
        plt.scatter(up["log2FoldChange"], -np.log10(up["padj"]), c="#E63946", s=30, alpha=0.8, label=f"Up ({len(up)})")
    if len(down) > 0:
        plt.scatter(down["log2FoldChange"], -np.log10(down["padj"]), c="#457B9D", s=30, alpha=0.8, label=f"Down ({len(down)})")
    
    # 阈值线
    plt.axvline(x=LOG2FC_THRESHOLD, color="black", linestyle="--", lw=1, alpha=0.6)
    plt.axvline(x=-LOG2FC_THRESHOLD, color="black", linestyle="--", lw=1, alpha=0.6)
    plt.axhline(y=-np.log10(P_THRESHOLD), color="black", linestyle="--", lw=1, alpha=0.6)
    
    # 标注Top10差异基因（显示 symbol）
    if len(up) + len(down) > 0:
        top10 = diff_plot.sort_values("padj").head(10)
        texts = []
        for _, row in top10.iterrows():
            texts.append(plt.text(
                row["log2FoldChange"], -np.log10(row["padj"]), 
                row["symbol"][:18], fontsize=7  # 直接显示symbol
            ))
        adjust_text(texts, arrowprops=dict(arrowstyle="->", color="black", alpha=0.3, lw=0.5))
    
    # 图表样式
    plt.xlabel("log2(Fold Change)", fontsize=12)
    plt.ylabel("-log10(Adjusted P-value)", fontsize=12)
    plt.title(f"GSE171542 Volcano Plot ({contrast['title']})", fontsize=14, fontweight="bold")
    plt.legend(loc="upper right", frameon=True)
    plt.grid(alpha=0.3)
    plt.xlim(-6, 6)
    plt.ylim(0, 6)
    
    # 保存
    volcano_path = os.path.join(OUTPUT_GSE_DIR, f"{GSE_ID}_火山图_{contrast['name']}.png")
    plt.tight_layout()
    plt.savefig(volcano_path, dpi=300, bbox_inches="tight", facecolor="white")
    plt.close()
    print(f"✅ 火山图保存：{volcano_path}")
    
    # -------------------------------------------------------------------------
    # 2. 热图（用ensembl_id匹配数据，图片显示symbol）
    # -------------------------------------------------------------------------
    if len(up) + len(down) == 0:
        print("⚠️ 无差异基因，跳过热图绘制")
        continue
    
    # 获取Top差异基因（ensembl_id + symbol映射）
    top_num = min(50, len(up) + len(down))
    top_df = diff_plot.sort_values("padj").head(top_num)
    # 核心：用ensembl_id匹配表达矩阵
    top_ensembl = top_df["ensembl_id"].tolist()
    gene_id_map = dict(zip(top_df["ensembl_id"], top_df["symbol"]))
    
    # 筛选存在的基因
    available_genes = [g for g in top_ensembl if g in expr_final.index]
    if len(available_genes) == 0:
        print("⚠️ 热图：无可用基因，跳过")
        continue
    
    # 提取表达矩阵 + 转换行名为symbol（图片显示基因名）
    heat_data = expr_final.loc[available_genes].copy()
    heat_data.index = heat_data.index.map(gene_id_map)  # 关键：ID转symbol
    # 移除symbol列，仅保留样本表达量
    heat_data = heat_data.drop(columns=["symbol"])
    
    # Z-score标准化
    heat_scaled = heat_data.sub(heat_data.mean(axis=1), axis=0).div(heat_data.std(axis=1), axis=0)
    
    # 样本分组颜色
    group_colors = {"Control": "#A8DADC", "RANKL": "#FFB703", "TYMP": "#E9C46A"}
    col_colors = [group_colors[GROUP_MAP[sample]] for sample in heat_data.columns]
    
    # 绘制聚类热图
    g = sns.clustermap(
        heat_scaled,
        cmap="RdBu_r", center=0,
        row_cluster=True, col_cluster=False,
        col_colors=col_colors,
        cbar_kws={"label": "Z-score"},
        linewidths=0.2,
        figsize=(11, 10),
        dendrogram_ratio=(0.1, 0.05)
    )
    g.ax_col_dendrogram.set_visible(False)
    g.fig.suptitle(
        f"Top {len(available_genes)} Differentially Expressed Genes\n({contrast['title']})", 
        y=1.02, fontsize=14, fontweight="bold"
    )
    
    # 保存热图
    heatmap_path = os.path.join(OUTPUT_GSE_DIR, f"{GSE_ID}_热图_{contrast['name']}.png")
    plt.savefig(heatmap_path, dpi=300, bbox_inches="tight", facecolor="white")
    plt.close()
    print(f"✅ 热图保存：{heatmap_path}")

print("\n🎉 所有绘图任务完成！")
print(f"📂 图片保存路径：{OUTPUT_GSE_DIR}")

### GSE276597

In [ ]:
# ==============================================
# GSE276597 表达矩阵处理 【DESeq2 标准流程】
# 使用 DESeq2 进行差异分析（适用于原始整数 counts）
# 基于 Ensembl ID 分析 | 结果保留双ID
# 输出原始未标准化矩阵 | 支持 txt.gz 输入
# 路径统一调用config | 分类输出至processed/output
# 阈值：标准严格阈值 |log2FC|>1 + FDR<0.05
# ==============================================
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

# ---------------------- 核心：从config导入标准化路径 ----------------------
from config import (
    HUMAN_LOOSE_VS_STABLE_GSE276597,  # GSE276597原始数据根目录
    DATA_PROCESSED,                   # 处理后数据总目录
    RESULT_DIR                        # 分析结果输出总目录
)

# ===================== 自动创建GSE专属子文件夹（完全不变）=====================
GSE_ID = "GSE276597"
PROCESSED_GSE_DIR = os.path.join(DATA_PROCESSED, GSE_ID)
OUTPUT_GSE_DIR = os.path.join(RESULT_DIR, GSE_ID)

os.makedirs(PROCESSED_GSE_DIR, exist_ok=True)
os.makedirs(OUTPUT_GSE_DIR, exist_ok=True)

RAW_DATA_PATH = os.path.join(HUMAN_LOOSE_VS_STABLE_GSE276597, "GSE276597_Galaxy70-DESeq2_result_overview.txt.gz")
# ==================================================================

# ---------------------- rpy2 相关导入（完全不变）----------------------
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri
from rpy2.robjects.conversion import localconverter
from rpy2.robjects.packages import importr, isinstalled

# 确保 DESeq2 已安装
utils = importr('utils')
if not isinstalled('DESeq2'):
    print("正在安装 DESeq2 ...")
    utils.install_packages('BiocManager')
    ro.r('BiocManager::install("DESeq2", update=FALSE, ask=FALSE)')
deseq2 = importr('DESeq2')

# ---------------------- 1. 读取原始数据（完全不变）----------------------
print("="*50)
print("读取 GSE276597 原始数据...")
print("="*50)
df_raw = pd.read_csv(RAW_DATA_PATH, sep="\t", compression="infer", low_memory=False)
print("文件列名（前10个）：", list(df_raw.columns)[:10])
print(f"数据形状：{df_raw.shape}")

# ---------------------- 2. 分离注释列和样本表达列（完全不变）----------------------
anno_cols = ["Sort0", "Gene ID", "Gene name", "Gene description", "Chr", 
             "Start", "End", "Strand", "Length", "IGV coordinates"]
sample_cols = [col for col in df_raw.columns if col not in anno_cols]
print(f"\n✅ 识别到样本列：{len(sample_cols)} 个")
print(f"样本列示例：{sample_cols[:5]}...")

# ---------------------- 3. 构建原始整数表达矩阵（索引=Ensembl ID）----------------------
expr_df = df_raw.set_index("Gene ID")[sample_cols].copy()
expr_df = expr_df.apply(pd.to_numeric, errors='coerce').fillna(0).astype(int)
print(f"\n✅ 原始整数矩阵形状：{expr_df.shape}（基因 × 样本）")

# ---------------------- 4. 低表达过滤（基于原始 counts，完全不变）---------------------
print("\n执行低表达基因过滤...")
threshold = max(2, expr_df.shape[1] * 0.3)
df_filter = expr_df.loc[(expr_df > 0).sum(axis=1) >= threshold]
print(f"✅ 过滤后矩阵形状：{df_filter.shape}")

# ---------------------- 5. 基因ID映射（保留Ensembl ID，新增symbol列，不合并）---------------------
print("\n执行基因ID映射（保留Ensembl ID，不合并基因）...")
# 建立Ensembl ID -> symbol映射
id_map = df_raw.set_index("Gene ID")["Gene name"].to_dict()
# 【核心修改】保留Ensembl ID为索引，仅添加symbol列，不合并、不求和
df_filter["symbol"] = df_filter.index.map(id_map)
# 去除无symbol的行
df_final = df_filter.dropna(subset=["symbol"]).copy()

print(f"✅ 最终基因数(Ensembl ID)：{df_final.shape[0]}")

# ---------------------- 6. 样本分组（完全不变）----------------------
print("\n配置样本分组...")
GROUP_MAP = {
    "N4161":"Control","N4162":"Control","N4147":"Control","N4148":"Control","N4149":"Control",
    "N4150":"Control","N4151":"Control","N4157":"Control","N4158":"Control","N4159":"Control",
    "N4160":"Control","N4163":"Control","N4164":"Control","N4152":"Control","N4153":"Control",
    "N4154":"Control","N4155":"Control","N4156":"Control",
    "N5033":"Case","N5034":"Case","N5035":"Case","N5026":"Case","N5027":"Case",
    "N5028":"Case","N5029":"Case","N5030":"Case","N5031":"Case","N5032":"Case"
}
control_samples = [s for s, g in GROUP_MAP.items() if g == "Control"]
case_samples = [s for s, g in GROUP_MAP.items() if g == "Case"]
print(f"✅ 对照组(Control)样本数：{len(control_samples)}")
print(f"✅ 处理组(Case)样本数：{len(case_samples)}")

# ---------------------- 7. 差异分析（DESeq2 基于Ensembl ID）----------------------
print("\n开始差异分析（DESeq2，适用于原始整数 counts）...")

# 准备 colData（完全不变）
col_data = pd.DataFrame({
    "sample": df_final.drop(columns=["symbol"]).columns,
    "condition": [GROUP_MAP[col] for col in df_final.drop(columns=["symbol"]).columns]
})
col_data["condition"] = col_data["condition"].astype("category")
col_data = col_data.set_index("sample")

# 设计公式
design = ro.Formula('~ condition')

# 使用 rpy2 调用 DESeq2
with localconverter(ro.default_converter + pandas2ri.converter):
    # 【核心】输入纯表达矩阵（索引=Ensembl ID）
    ro.globalenv['counts_mat'] = df_final.drop(columns=["symbol"]).astype(int)
    ro.globalenv['col_data'] = col_data
    ro.globalenv['design'] = design
    
    ro.r('''
        library(DESeq2)
        dds <- DESeqDataSetFromMatrix(countData = counts_mat,
                                      colData = col_data,
                                      design = design)
        dds <- DESeq(dds)
        res <- results(dds, contrast = c("condition", "Case", "Control"))
        res_df <- as.data.frame(res)
    ''')
    
    res_df = ro.globalenv['res_df']

# 提取结果
log2fc = res_df['log2FoldChange'].values
p_values = res_df['pvalue'].values
padj = res_df['padj'].values

# 处理NA值
padj = np.nan_to_num(padj, nan=1.0)
p_values = np.nan_to_num(p_values, nan=1.0)

# 【核心修改】构建差异结果表（同时保留ensembl_id + symbol）
diff_result = pd.DataFrame({
    "ensembl_id": df_final.index,        # Ensembl ID
    "symbol": df_final["symbol"].values, # Gene Symbol
    "log2FoldChange": log2fc,
    "pvalue": p_values,
    "padj": padj,
    "abs_log2FC": np.abs(log2fc)
})

# ---------------------- 8. 筛选差异基因（完全不变）----------------------
P_THRESHOLD = 0.05
LOG2FC_THRESHOLD = 1.0

print(f"\n筛选差异基因（阈值：padj < {P_THRESHOLD} 且 |log2FC| > {LOG2FC_THRESHOLD}）...")
up_genes = diff_result[(diff_result["padj"] < P_THRESHOLD) & (diff_result["log2FoldChange"] > LOG2FC_THRESHOLD)]
down_genes = diff_result[(diff_result["padj"] < P_THRESHOLD) & (diff_result["log2FoldChange"] < -LOG2FC_THRESHOLD)]

print(f"✅ 上调基因数（Case>Control）：{len(up_genes)}")
print(f"✅ 下调基因数（Case<Control）：{len(down_genes)}")
print(f"✅ 总差异基因数：{len(up_genes) + len(down_genes)}")

# ---------------------- 9. 保存结果文件（路径/文件名完全不变，内容含双ID）----------------------
print("\n" + "="*50)
print("保存结果文件...")
print("="*50)

# 1. 原始表达矩阵（索引=ensembl_id + symbol列）
df_final.to_csv(os.path.join(PROCESSED_GSE_DIR, f"{GSE_ID}_原始表达矩阵.csv"), index=True, encoding="utf-8-sig")

# 2. 样本分组信息（完全不变）
group_df = pd.DataFrame({
    "sample_id": df_final.drop(columns=["symbol"]).columns, 
    "group": [GROUP_MAP[s] for s in df_final.drop(columns=["symbol"]).columns]
})
group_df.to_csv(os.path.join(PROCESSED_GSE_DIR, f"{GSE_ID}_样本分组信息.csv"), index=False, encoding="utf-8-sig")

# 3. 完整差异分析结果（含ensembl_id + symbol）
diff_result.to_csv(os.path.join(OUTPUT_GSE_DIR, f"{GSE_ID}_差异分析结果_Case_vs_Control.csv"), index=False, encoding="utf-8-sig")

# 4. 上调基因（含双ID）
up_sorted = up_genes.sort_values("log2FoldChange", ascending=False)
up_sorted.to_csv(os.path.join(OUTPUT_GSE_DIR, f"{GSE_ID}_上调基因_Case_vs_Control.csv"), index=False, encoding="utf-8-sig")

# 5. 下调基因（含双ID）
down_sorted = down_genes.sort_values("log2FoldChange", ascending=True)
down_sorted.to_csv(os.path.join(OUTPUT_GSE_DIR, f"{GSE_ID}_下调基因_Case_vs_Control.csv"), index=False, encoding="utf-8-sig")

# ---------------------- 完成 ----------------------
print("\n🎉🎉🎉 GSE276597 数据处理 + DESeq2 差异分析全部完成！")
print(f"📊 最终表达矩阵形状：{df_final.shape}（基因×样本）")
print(f"📈 上调基因数：{len(up_sorted)}，下调基因数：{len(down_sorted)}")
print(f"📂 原始矩阵/分组信息保存路径：{PROCESSED_GSE_DIR}")
print(f"📂 差异分析/上调下调基因保存路径：{OUTPUT_GSE_DIR}")

In [ ]:
# -*- coding: utf-8 -*-
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap
import os

# ===================== 导入 config 路径配置（核心修改） =====================
from config import DATA_PROCESSED, RESULT_DIR

# 字体配置
plt.rcParams['font.sans-serif'] = ['DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# ===================== 自动配置路径 =====================
GSE_ID = "GSE276597"
# 输入：处理后数据（原始表达矩阵+分组信息）
PROCESSED_GSE_DIR = os.path.join(DATA_PROCESSED, GSE_ID)
# 输入/输出：差异结果+可视化图片
OUTPUT_GSE_DIR = os.path.join(RESULT_DIR, GSE_ID)

# 自动创建输出文件夹
os.makedirs(OUTPUT_GSE_DIR, exist_ok=True)

# 阈值
FC_THRESH = 1
FDR_THRESH = 0.05

# ===================== 函数：绘制火山图（适配新格式） =====================
def plot_volcano(input_file, save_name, title_name):
    # 【修改1】新差异文件无索引列，删除 index_col=0
    df = pd.read_csv(input_file)

    # 兼容列名：padj / FDR
    if 'padj' in df.columns:
        df.rename(columns={'padj': 'FDR'}, inplace=True)
    if 'log2FoldChange' in df.columns:
        df.rename(columns={'log2FoldChange': 'log2FC'}, inplace=True)

    df['-log10(FDR)'] = -np.log10(df['FDR'].astype(float))

    def classify(row):
        if row['FDR'] < FDR_THRESH and row['log2FC'] > FC_THRESH:
            return 'Up'
        elif row['FDR'] < FDR_THRESH and row['log2FC'] < -FC_THRESH:
            return 'Down'
        else:
            return 'No'

    df['type'] = df.apply(classify, axis=1)

    plt.figure(figsize=(6, 5), dpi=300)
    colors = {'Up': '#FF4C4C', 'Down': '#4C6EFF', 'No': '#B8B8B8'}
    sns.scatterplot(x='log2FC', y='-log10(FDR)', hue='type',
                    data=df, palette=colors, s=8, alpha=0.8, legend='full')

    plt.axvline(x=FC_THRESH, color='gray', linestyle='--', lw=0.8)
    plt.axvline(x=-FC_THRESH, color='gray', linestyle='--', lw=0.8)
    plt.axhline(y=-np.log10(FDR_THRESH), color='gray', linestyle='--', lw=0.8)

    plt.title(f'{GSE_ID} {title_name}', fontsize=12, pad=15)
    plt.xlabel('log2(Fold Change)', fontsize=10)
    plt.ylabel('-log10(FDR)', fontsize=10)
    plt.xlim(-6, 6)
    plt.legend(title='Gene Type', loc='upper right')
    plt.tight_layout()

    # 保存路径不变
    save_path = os.path.join(OUTPUT_GSE_DIR, f"{save_name}.png")
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()

    up_num = len(df[df['type'] == 'Up'])
    down_num = len(df[df['type'] == 'Down'])
    print(f"✅ {title_name} 火山图完成 | 上调:{up_num} | 下调:{down_num}")

# ===================== 热图函数（适配Ensembl ID + symbol显示） =====================
def plot_heatmap(matrix_file, group_file, diff_file, save_name, top_n=50):
    # 1. 读取数据
    expr = pd.read_csv(matrix_file, index_col=0)  # 索引=Ensembl ID，含symbol列
    group = pd.read_csv(group_file)
    diff = pd.read_csv(diff_file)  # 含ensembl_id + symbol双列

    # 2. 筛选显著差异基因
    diff_sig = diff[
        (diff['padj'] < FDR_THRESH) &
        (abs(diff['log2FoldChange']) > FC_THRESH)
    ].copy()

    if len(diff_sig) == 0:
        print("⚠️ 无显著差异基因，跳过热图")
        return

    # 3. 【修改2】取Top N差异基因，建立Ensembl ID → symbol映射
    diff_sig['abs_log2FC'] = diff_sig['log2FoldChange'].abs()
    top_df = diff_sig.sort_values('abs_log2FC', ascending=False).head(top_n)
    # 获取Top基因的Ensembl ID和对应的symbol
    ensembl_to_symbol = dict(zip(top_df['ensembl_id'], top_df['symbol']))
    top_ensembl_ids = top_df['ensembl_id'].tolist()

    # 4. 【修改3】按Ensembl ID提取矩阵 + Z-score标准化
    expr_plot = expr.loc[expr.index.isin(top_ensembl_ids), :].copy()
    # 删除表达矩阵中的symbol列，仅保留样本数据
    expr_plot = expr_plot.drop(columns=['symbol'])
    expr_plot = expr_plot.T.apply(lambda x: (x - x.mean()) / x.std()).T

    # 5. 【修改4】将索引替换为symbol（图片显示基因名）
    expr_plot.index = expr_plot.index.map(ensembl_to_symbol)

    # 6. 样本分组排序（不变）
    group_order = ['Case', 'Control']
    group['group_order'] = group['group'].map({g: i for i, g in enumerate(group_order)})
    sample_order = group.sort_values('group_order')['sample_id'].tolist()
    expr_plot = expr_plot[sample_order]

    # 7. 分组颜色条（不变）
    group_dict = dict(zip(group['sample_id'], group['group']))
    group_labels = [group_dict[s] for s in sample_order]
    group_colors_map = {'Case': '#1f77b4', 'Control': '#ff7f0e'}
    col_colors = [group_colors_map[g] for g in group_labels]

    # 8. 色标配置（不变）
    cmap = sns.diverging_palette(240, 10, as_cmap=True)
    vmin, vmax = -2.5, 2.5

    # 9. 绘制热图（不变）
    g = sns.clustermap(
        expr_plot,
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
        row_cluster=True,
        col_cluster=False,
        col_colors=col_colors,
        figsize=(12, 10),
        dendrogram_ratio=(0.1, 0.05),
        cbar_pos=(0.02, 0.8, 0.03, 0.18),
        yticklabels=True,
        xticklabels=True,
        linewidths=0.1
    )

    # 10. 标题（不变）
    g.fig.suptitle(
        f'{GSE_ID} Top{top_n} Differentially Expressed Genes (Case vs Control)',
        y=1.02,
        fontsize=14,
        fontweight='bold'
    )

    # 11. 分组图例（不变）
    for g_name, color in group_colors_map.items():
        g.ax_col_dendrogram.bar(0, 0, color=color, label=g_name, linewidth=0)
    g.ax_col_dendrogram.legend(
        loc='upper right',
        bbox_to_anchor=(1.15, 1),
        fontsize=10,
        title='Group'
    )

    # 12. 标签旋转（不变）
    plt.setp(g.ax_heatmap.get_xticklabels(), rotation=45, ha='right', fontsize=8)
    plt.setp(g.ax_heatmap.get_yticklabels(), fontsize=8)

    # 保存路径不变
    save_path = os.path.join(OUTPUT_GSE_DIR, f"{save_name}.png")
    plt.tight_layout()
    g.savefig(save_path, dpi=300, bbox_inches='tight', facecolor='white')
    plt.close()

    up_num = len(diff_sig[diff_sig['log2FoldChange'] > 0])
    down_num = len(diff_sig[diff_sig['log2FoldChange'] < 0])
    print(f"✅ 热图完成：{save_name} | 上调:{up_num} | 下调:{down_num} | Top{top_n}")

# ===================== 一键运行（完全不变） =====================
if __name__ == "__main__":
    print("🚀 开始绘制 GSE276597 火山图 + 热图...\n")

    # 火山图
    plot_volcano(
        input_file=os.path.join(OUTPUT_GSE_DIR, f"{GSE_ID}_差异分析结果_Case_vs_Control.csv"),
        save_name=f"{GSE_ID}_火山图_Case_vs_Control",
        title_name="Volcano Plot (Case vs Control)"
    )

    # 热图
    plot_heatmap(
        matrix_file=os.path.join(PROCESSED_GSE_DIR, f"{GSE_ID}_原始表达矩阵.csv"),
        group_file=os.path.join(PROCESSED_GSE_DIR, f"{GSE_ID}_样本分组信息.csv"),
        diff_file=os.path.join(OUTPUT_GSE_DIR, f"{GSE_ID}_差异分析结果_Case_vs_Control.csv"),
        save_name=f"{GSE_ID}_热图_差异Top50",
        top_n=50
    )

    print(f"\n🎉 全部图片已保存至：{OUTPUT_GSE_DIR}")


    
### GSE284391



In [ ]:
# ==============================================
# HA vs OA 单细胞表达矩阵构建（GSE文件夹分离版 + 基因名规范化）
# 输出：
#   - data/processed/GSE284391/     → HA 数据（中间结果）
#   - data/processed/GSE152805/     → OA 数据（中间结果）
#   - data/processed/HA_vs_OA/      → 合并后的表达矩阵和分组信息（供差异分析使用）
# 核心修改：以 Ensembl ID 为基因索引，同时保存 ID + Symbol
# ==============================================
import pandas as pd
import numpy as np
import os
import gzip
import warnings
import scipy.io as sio
from scipy.sparse import csr_matrix, hstack, save_npz, load_npz
import gc

# 导入config路径配置
from config import (
    DATA_PROCESSED,
    HUMAN_HA_VS_OA_GSE152805,
    HUMAN_HA_VS_OA_GSE284391
)

warnings.filterwarnings('ignore')

# ===================== 定义输出路径 =====================
HA_PROCESSED_DIR = os.path.join(DATA_PROCESSED, "GSE284391")
OA_PROCESSED_DIR = os.path.join(DATA_PROCESSED, "GSE152805")
COMBINED_DIR = os.path.join(DATA_PROCESSED, "HA_vs_OA")

for d in [HA_PROCESSED_DIR, OA_PROCESSED_DIR, COMBINED_DIR]:
    os.makedirs(d, exist_ok=True)

# ==============================================================================
# 步骤1：处理 HA 数据（GSE284391）→ 以Ensembl ID为核心，保存ID+Symbol
# ==============================================================================
print("="*60)
print("步骤1：处理 GSE284391 HA 数据 -> 稀疏矩阵")
print("="*60)

MTX_PATH = os.path.join(HUMAN_HA_VS_OA_GSE284391, "GSE284391_matrix.mtx.gz")
FEATURES_PATH = os.path.join(HUMAN_HA_VS_OA_GSE284391, "GSE284391_features.tsv.gz")
BARCODES_PATH = os.path.join(HUMAN_HA_VS_OA_GSE284391, "GSE284391_barcodes.tsv.gz")

with gzip.open(MTX_PATH, 'rb') as f:
    mtx_data = sio.mmread(f)

# 读取特征：Ensembl ID + Gene Symbol（核心修改）
features = pd.read_csv(FEATURES_PATH, sep="\t", compression="gzip", header=None)
if features.shape[1] >= 3:
    features.columns = ["gene_id", "gene_symbol", "feature_type"]
else:
    features.columns = ["gene_id", "gene_symbol"]
# 数据清洗：去除空值、规范化
features = features.dropna(subset=["gene_id", "gene_symbol"]).copy()
features["gene_symbol"] = features["gene_symbol"].astype(str).str.strip().str.upper()
# 基于 Ensembl ID 去重（核心修改：不再用symbol去重）
features = features.drop_duplicates(subset=["gene_id"], keep="first")

gene_ids = features["gene_id"].tolist()
gene_symbols = features["gene_symbol"].tolist()
id2symbol_ha = dict(zip(gene_ids, gene_symbols))  # ID→Symbol映射

barcodes = pd.read_csv(BARCODES_PATH, sep="\t", compression="gzip", header=None)
cell_ids = barcodes[0].tolist()
print(f"原始 HA 基因数: {len(gene_ids)}, 细胞数: {len(cell_ids)}")

# 确定矩阵方向
if mtx_data.shape[0] == len(gene_ids) and mtx_data.shape[1] == len(cell_ids):
    ha_sparse = csr_matrix(mtx_data)
    print("方向: 基因 × 细胞")
elif mtx_data.shape[1] == len(gene_ids) and mtx_data.shape[0] == len(cell_ids):
    ha_sparse = csr_matrix(mtx_data.T)
    print("方向: 细胞 × 基因（已转置）")
else:
    raise ValueError("维度不匹配")

# 去除全零基因
nonzero_rows = ha_sparse.getnnz(axis=1) > 0
ha_sparse = ha_sparse[nonzero_rows, :]
# 过滤基因ID和Symbol
gene_ids_final = [gene_ids[i] for i in range(len(gene_ids)) if nonzero_rows[i]]
gene_symbols_final = [gene_symbols[i] for i in range(len(gene_ids)) if nonzero_rows[i]]
print(f"去零后 HA 基因数: {len(gene_ids_final)}")
if len(gene_ids_final) == 0:
    raise RuntimeError("HA 数据基因列表为空！请检查原始文件。")

# 保存文件（路径/文件名不变，内容改为ID+Symbol双列）
save_npz(os.path.join(HA_PROCESSED_DIR, "GSE284391_HA_sparse.npz"), ha_sparse)
# 保存：Ensembl ID + Symbol
gene_df_ha = pd.DataFrame({"gene_id": gene_ids_final, "gene_symbol": gene_symbols_final})
gene_df_ha.to_csv(os.path.join(HA_PROCESSED_DIR, "GSE284391_HA_genes.csv"), index=False)
pd.Series(cell_ids).to_csv(os.path.join(HA_PROCESSED_DIR, "GSE284391_HA_cells.csv"), index=False, header=False)
# ID→索引映射
ha_gene_to_idx = {gene: i for i, gene in enumerate(gene_ids_final)}
pd.Series(ha_gene_to_idx).astype(str).to_csv(os.path.join(HA_PROCESSED_DIR, "GSE284391_HA_gene_mapping.csv"))

print(f"✅ HA 数据已保存到: {HA_PROCESSED_DIR}")
print(f"   矩阵: {ha_sparse.shape[0]} 基因 × {ha_sparse.shape[1]} 细胞, 非零 {ha_sparse.nnz}")

del mtx_data, ha_sparse, gene_ids, gene_symbols, gene_ids_final, gene_symbols_final
gc.collect()

# ==============================================================================
# 步骤2：处理 OA 数据（GSE152805）→ 以Ensembl ID为核心，保存ID+Symbol
# ==============================================================================
print("\n" + "="*60)
print("步骤2：处理 GSE152805 OA 数据 -> 稀疏矩阵")
print("="*60)

matrix_files = [f for f in os.listdir(HUMAN_HA_VS_OA_GSE152805) if f.endswith(".matrix.mtx.gz")]
print(f"发现 OA matrix 文件: {matrix_files}")

oa_sparse_list = []
oa_gene_list = []      # 存储 [gene_id, gene_symbol]
oa_cell_list = []

for matrix_file in matrix_files:
    prefix = matrix_file.replace(".matrix.mtx.gz", "")
    matrix_path = os.path.join(HUMAN_HA_VS_OA_GSE152805, matrix_file)
    genes_path = os.path.join(HUMAN_HA_VS_OA_GSE152805, f"{prefix}.genes.tsv.gz")
    barcodes_path = os.path.join(HUMAN_HA_VS_OA_GSE152805, f"{prefix}.barcodes.tsv.gz")

    if not (os.path.exists(genes_path) and os.path.exists(barcodes_path)):
        print(f"  跳过 {prefix}：缺少基因或细胞文件")
        continue

    print(f"  读取 {prefix} ...")
    with gzip.open(matrix_path, 'rb') as f:
        mtx = sio.mmread(f)

    # 读取OA基因：Ensembl ID + Symbol（核心修改）
    genes = pd.read_csv(genes_path, sep="\t", compression="gzip", header=None)
    genes = genes.dropna()
    if genes.shape[1] >= 2:
        gene_ids_oa = genes[0].astype(str).str.strip().tolist()
        gene_symbols_oa = genes[1].astype(str).str.strip().str.upper().tolist()
    else:
        raise ValueError("OA基因文件必须包含 Ensembl ID 和 Symbol 两列")
    
    # 基于Ensembl ID去重
    gene_df_oa = pd.DataFrame({"gene_id": gene_ids_oa, "gene_symbol": gene_symbols_oa})
    gene_df_oa = gene_df_oa.drop_duplicates(subset=["gene_id"], keep="first")
    gene_ids_oa = gene_df_oa["gene_id"].tolist()
    gene_symbols_oa = gene_df_oa["gene_symbol"].tolist()

    # 细胞ID
    barcodes_oa = pd.read_csv(barcodes_path, sep="\t", compression="gzip", header=None)
    cell_ids_oa = [f"{prefix}_{b}" for b in barcodes_oa[0].tolist()]
    print(f"    原始基因数: {len(gene_ids_oa)}, 细胞数: {len(cell_ids_oa)}")

    # 矩阵方向
    if mtx.shape[0] == len(gene_ids_oa) and mtx.shape[1] == len(cell_ids_oa):
        sparse_mat = csr_matrix(mtx)
    elif mtx.shape[1] == len(gene_ids_oa) and mtx.shape[0] == len(cell_ids_oa):
        sparse_mat = csr_matrix(mtx.T)
    else:
        print(f"    维度不匹配，跳过")
        continue

    # 去除全零基因
    nonzero_rows = sparse_mat.getnnz(axis=1) > 0
    sparse_mat = sparse_mat[nonzero_rows, :]
    # 过滤基因
    gene_ids_oa_final = [gene_ids_oa[i] for i in range(len(gene_ids_oa)) if nonzero_rows[i]]
    gene_symbols_oa_final = [gene_symbols_oa[i] for i in range(len(gene_ids_oa)) if nonzero_rows[i]]
    print(f"    去零后基因数: {len(gene_ids_oa_final)}")

    oa_sparse_list.append(sparse_mat)
    oa_gene_list.append( (gene_ids_oa_final, gene_symbols_oa_final) )
    oa_cell_list.append(cell_ids_oa)
    print(f"      保留细胞: {len(cell_ids_oa)}")

if not oa_sparse_list:
    raise RuntimeError("没有成功读取任何 OA 样本")

# 合并 OA 样本（基于 Ensembl ID 取交集）
print("  合并 OA 样本（基因ID交集）...")
# 提取所有样本的基因ID
oa_gene_ids = [g[0] for g in oa_gene_list]
common_ids_oa = set(oa_gene_ids[0])
for g in oa_gene_ids[1:]:
    common_ids_oa &= set(g)
common_ids_oa = sorted(common_ids_oa)
print(f"    OA 共同基因ID数: {len(common_ids_oa)}")
if len(common_ids_oa) == 0:
    raise RuntimeError("OA 各样本之间无共同基因ID！")

# 匹配Symbol
id2symbol_oa = {}
for g_ids, g_syms in oa_gene_list:
    id2symbol_oa.update(dict(zip(g_ids, g_syms)))
common_symbols_oa = [id2symbol_oa[g] for g in common_ids_oa]

# 子集化矩阵
oa_sub_matrices = []
for mat, (g_ids, _) in zip(oa_sparse_list, oa_gene_list):
    gene_to_idx = {g: i for i, g in enumerate(g_ids)}
    idx = [gene_to_idx[g] for g in common_ids_oa]
    oa_sub_matrices.append(mat[idx, :])
oa_combined = hstack(oa_sub_matrices)
oa_cells_combined = [cell for cells in oa_cell_list for cell in cells]

# 保存OA数据（路径/文件名不变）
save_npz(os.path.join(OA_PROCESSED_DIR, "GSE152805_OA_sparse.npz"), oa_combined)
# 保存ID+Symbol
gene_df_oa = pd.DataFrame({"gene_id": common_ids_oa, "gene_symbol": common_symbols_oa})
gene_df_oa.to_csv(os.path.join(OA_PROCESSED_DIR, "GSE152805_OA_genes.csv"), index=False)
pd.Series(oa_cells_combined).to_csv(os.path.join(OA_PROCESSED_DIR, "GSE152805_OA_cells.csv"), index=False, header=False)
# ID→索引映射
oa_gene_to_idx = {gene: i for i, gene in enumerate(common_ids_oa)}
pd.Series(oa_gene_to_idx).astype(str).to_csv(os.path.join(OA_PROCESSED_DIR, "GSE152805_OA_gene_mapping.csv"))

print(f"✅ OA 数据已保存到: {OA_PROCESSED_DIR}")
print(f"   矩阵: {oa_combined.shape[0]} 基因 × {oa_combined.shape[1]} 细胞, 非零 {oa_combined.nnz}")

del oa_sparse_list, oa_gene_list, oa_cell_list, oa_sub_matrices, oa_combined
gc.collect()

# ==============================================================================
# 步骤3：合并 HA 和 OA（基于 Ensembl ID 交集，同时保存Symbol）
# ==============================================================================
print("\n" + "="*60)
print("步骤3：合并 HA 和 OA 稀疏矩阵（基因ID交集）")
print("="*60)

# 加载 HA 数据（读取ID+Symbol）
ha_gene_df = pd.read_csv(os.path.join(HA_PROCESSED_DIR, "GSE284391_HA_genes.csv"))
ha_gene_ids = ha_gene_df["gene_id"].tolist()
ha_id2symbol = dict(zip(ha_gene_df["gene_id"], ha_gene_df["gene_symbol"]))
ha_gene_mapping = pd.read_csv(
    os.path.join(HA_PROCESSED_DIR, "GSE284391_HA_gene_mapping.csv"),
    index_col=0, header=None, dtype=str
).to_dict()[1]
ha_cells = pd.read_csv(os.path.join(HA_PROCESSED_DIR, "GSE284391_HA_cells.csv"), header=None, dtype=str)[0].tolist()
ha_sparse = load_npz(os.path.join(HA_PROCESSED_DIR, "GSE284391_HA_sparse.npz"))
print(f"HA 基因ID数: {len(ha_gene_ids)}")

# 加载 OA 数据（读取ID+Symbol）
oa_gene_df = pd.read_csv(os.path.join(OA_PROCESSED_DIR, "GSE152805_OA_genes.csv"))
oa_gene_ids = oa_gene_df["gene_id"].tolist()
oa_id2symbol = dict(zip(oa_gene_df["gene_id"], oa_gene_df["gene_symbol"]))
oa_gene_mapping = pd.read_csv(
    os.path.join(OA_PROCESSED_DIR, "GSE152805_OA_gene_mapping.csv"),
    index_col=0, header=None, dtype=str
).to_dict()[1]
oa_cells = pd.read_csv(os.path.join(OA_PROCESSED_DIR, "GSE152805_OA_cells.csv"), header=None, dtype=str)[0].tolist()
oa_sparse = load_npz(os.path.join(OA_PROCESSED_DIR, "GSE152805_OA_sparse.npz"))
print(f"OA 基因ID数: {len(oa_gene_ids)}")

# 基于 Ensembl ID 取交集（核心修改）
common_ids = sorted(set(ha_gene_ids) & set(oa_gene_ids))
print(f"共同基因ID数: {len(common_ids)}")
if len(common_ids) == 0:
    raise RuntimeError("HA 和 OA 无共同基因ID！")
# 合并后的Symbol（统一映射）
common_symbols = [ha_id2symbol[g] for g in common_ids]

# 子集化矩阵
ha_idx = [int(ha_gene_mapping[g]) for g in common_ids]
oa_idx = [int(oa_gene_mapping[g]) for g in common_ids]
ha_sub = ha_sparse[ha_idx, :]
oa_sub = oa_sparse[oa_idx, :]

del ha_sparse, oa_sparse
gc.collect()

# 合并矩阵
combined_sparse = hstack([ha_sub, oa_sub])
combined_cells = ha_cells + oa_cells
combined_groups = ["HA"] * len(ha_cells) + ["OA"] * len(oa_cells)

# 保存合并结果（路径/文件名不变，保存ID+Symbol）
save_npz(os.path.join(COMBINED_DIR, "HA_vs_OA_combined_sparse.npz"), combined_sparse)
# 核心输出：Ensembl ID + Symbol 双列
combined_gene_df = pd.DataFrame({"gene_id": common_ids, "gene_symbol": common_symbols})
combined_gene_df.to_csv(os.path.join(COMBINED_DIR, "HA_vs_OA_combined_genes.csv"), index=False)
pd.Series(combined_cells).to_csv(os.path.join(COMBINED_DIR, "HA_vs_OA_combined_cells.csv"), index=False, header=False)
group_df = pd.DataFrame({"cell_id": combined_cells, "group": combined_groups})
group_df.to_csv(os.path.join(COMBINED_DIR, "HA_vs_OA_group_info.csv"), index=False)

print(f"\n🎉 合并完成！")
print(f"📁 合并矩阵保存在: {COMBINED_DIR}")
print(f"   形状: {combined_sparse.shape[0]} 基因 × {combined_sparse.shape[1]} 细胞")
print(f"   HA 细胞: {len(ha_cells)}, OA 细胞: {len(oa_cells)}")

In [ ]:
# ==============================================
# HA vs OA 单细胞差异分析 + 火山图 + 热图（适配 config.py + 双ID格式）
# 读取 data/processed/HA_vs_OA/ 合并数据（Ensembl ID+Symbol）
# 输出到 output/HA_vs_OA_diff/
# ==============================================
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.sparse import load_npz
import scanpy as sc
from matplotlib import rcParams

# 导入 config 路径配置
from config import DATA_PROCESSED, RESULT_DIR

# ===================== 路径配置 =====================
COMBINED_DIR = os.path.join(DATA_PROCESSED, "HA_vs_OA")
OUTPUT_DIR = os.path.join(RESULT_DIR, "HA_vs_OA_diff")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 设置 scanpy 输出详细进度
sc.settings.verbosity = 3

# 设置绘图风格
rcParams['figure.figsize'] = (8, 6)
rcParams['font.size'] = 10
sns.set_style("whitegrid")

# ===================== 1. 加载数据（适配双ID格式：Ensembl ID + Symbol） =====================
print("加载稀疏矩阵和元数据...")
X = load_npz(os.path.join(COMBINED_DIR, "HA_vs_OA_combined_sparse.npz"))
combined_gene_df = pd.read_csv(os.path.join(COMBINED_DIR, "HA_vs_OA_combined_genes.csv"))
gene_ids = combined_gene_df["gene_id"].tolist()
gene_symbols = combined_gene_df["gene_symbol"].tolist()
id2symbol = dict(zip(gene_ids, gene_symbols))

cells = pd.read_csv(os.path.join(COMBINED_DIR, "HA_vs_OA_combined_cells.csv"), header=None)[0].tolist()
group_info = pd.read_csv(os.path.join(COMBINED_DIR, "HA_vs_OA_group_info.csv"))

# 确保顺序一致
group_info = group_info.set_index("cell_id").loc[cells]
print(f"矩阵形状: {X.shape} 基因 × 细胞")
print(f"HA 细胞数: {(group_info['group'] == 'HA').sum()}")
print(f"OA 细胞数: {(group_info['group'] == 'OA').sum()}")

# ===================== 2. 构建 AnnData 对象 =====================
adata = sc.AnnData(
    X=X.T.astype(np.float32),
    obs=group_info,
    var=combined_gene_df.set_index("gene_id")
)
print("AnnData 构建完成")

# ===================== 3. 质量控制与标准化 =====================
sc.pp.filter_cells(adata, min_genes=200)
sc.pp.filter_genes(adata, min_cells=3)

sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

# ===================== 4. 差异分析 =====================
print("开始差异分析...")
sc.tl.rank_genes_groups(adata, groupby='group', groups=['OA'], reference='HA',
                        method='wilcoxon', use_raw=False, n_jobs=4)

# 提取结果
de_df = sc.get.rank_genes_groups_df(adata, group='OA')
print("原始列名:", de_df.columns.tolist())

# 重命名列并添加基因Symbol
if 'logfoldchanges' in de_df.columns:
    de_df = de_df.rename(columns={
        'names': 'gene_id',
        'logfoldchanges': 'log2FC',
        'pvals': 'p_val',
        'pvals_adj': 'p_val_adj'
    })
    if 'scores' in de_df.columns:
        de_df = de_df.drop(columns=['scores'])
else:
    de_df.columns = ['gene_id', 'scores', 'log2FC', 'p_val', 'p_val_adj']
    de_df = de_df.drop(columns=['scores'])

de_df["gene_symbol"] = de_df["gene_id"].map(id2symbol)
de_df = de_df[["gene_id", "gene_symbol", "log2FC", "p_val", "p_val_adj"]]
de_df = de_df.sort_values('p_val_adj')

de_df.to_csv(os.path.join(OUTPUT_DIR, "HA_vs_OA_diff_all.csv"), index=False)
print(f"差异分析完成，共 {len(de_df)} 个基因")

# 筛选显著差异基因
sig_up = de_df[(de_df['log2FC'] > 1) & (de_df['p_val_adj'] < 0.05)]
sig_down = de_df[(de_df['log2FC'] < -1) & (de_df['p_val_adj'] < 0.05)]
sig_all = pd.concat([sig_up, sig_down])
print(f"显著上调基因: {len(sig_up)}")
print(f"显著下调基因: {len(sig_down)}")

# 保存显著基因
sig_up.to_csv(os.path.join(OUTPUT_DIR, "HA_vs_OA_upregulated_genes.csv"), index=False)
sig_down.to_csv(os.path.join(OUTPUT_DIR, "HA_vs_OA_downregulated_genes.csv"), index=False)
sig_all.to_csv(os.path.join(OUTPUT_DIR, "HA_vs_OA_significant_genes.csv"), index=False)

# ===================== 5. 火山图 =====================
fig, ax = plt.subplots(figsize=(10, 8))
ax.scatter(de_df['log2FC'], -np.log10(de_df['p_val_adj']),
           c='lightgray', alpha=0.5, s=5, label='Not significant')
ax.scatter(sig_up['log2FC'], -np.log10(sig_up['p_val_adj']),
           c='red', alpha=0.8, s=20, label=f'Upregulated ({len(sig_up)})')
ax.scatter(sig_down['log2FC'], -np.log10(sig_down['p_val_adj']),
           c='blue', alpha=0.8, s=20, label=f'Downregulated ({len(sig_down)})')
ax.axhline(-np.log10(0.05), linestyle='--', color='black', alpha=0.5)
ax.axvline(-1, linestyle='--', color='black', alpha=0.5)
ax.axvline(1, linestyle='--', color='black', alpha=0.5)
ax.set_xlabel('log2 Fold Change (OA vs HA)', fontsize=12)
ax.set_ylabel('-log10(Adjusted P-value)', fontsize=12)
ax.set_title('Volcano Plot: OA vs HA', fontsize=14)
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "Volcano_plot.png"), dpi=300)
plt.close()
print("火山图已保存")

# ===================== 6. 热图（修复：seaborn手动绘制，显示基因Symbol） =====================
print("绘制热图...")
# 获取Top基因ID
top_up_ids = sig_up.nlargest(20, 'log2FC')['gene_id'].tolist()
top_down_ids = sig_down.nsmallest(20, 'log2FC')['gene_id'].tolist()
top_gene_ids = top_up_ids + top_down_ids

# 过滤掉QC中被删除的基因
existing_genes = adata.var_names.tolist()
top_gene_ids_filtered = [g for g in top_gene_ids if g in existing_genes]
top_gene_symbols = [id2symbol[g] for g in top_gene_ids_filtered]
print(f"热图展示基因数: {len(top_gene_ids_filtered)}")

# 下采样细胞
np.random.seed(42)
n_cells_sample = 200
ha_cells = adata.obs[adata.obs['group'] == 'HA'].index
oa_cells = adata.obs[adata.obs['group'] == 'OA'].index
if len(ha_cells) > n_cells_sample:
    ha_cells = np.random.choice(ha_cells, n_cells_sample, replace=False)
if len(oa_cells) > n_cells_sample:
    oa_cells = np.random.choice(oa_cells, n_cells_sample, replace=False)
sample_cells = np.concatenate([ha_cells, oa_cells])

# 提取表达矩阵
adata_sub = adata[sample_cells, top_gene_ids_filtered].copy()
expr_matrix = pd.DataFrame(
    adata_sub.X.toarray().T,  # 基因 × 细胞
    index=top_gene_symbols,
    columns=adata_sub.obs_names
)
# 按分组排序
group_order = adata_sub.obs.sort_values('group').index
expr_matrix = expr_matrix[group_order]
group_labels = adata_sub.obs.loc[group_order, 'group']

# 手动绘制热图（seaborn，无scanpy兼容问题）
plt.figure(figsize=(16, 10))
g = sns.clustermap(
    expr_matrix,
    cmap="RdBu_r",
    standard_scale=0,  # 标准化基因
    row_cluster=False,
    col_cluster=False,
    col_colors=group_labels.map({"HA": "#1f77b4", "OA": "#ff7f0e"}),
    figsize=(16, 10)
)
# 设置标题
plt.suptitle('Top 20 Up/Down-regulated Genes (OA vs HA)', y=1.02, fontsize=16)
# 保存
g.savefig(os.path.join(OUTPUT_DIR, "Heatmap_top_genes.png"), dpi=300, bbox_inches='tight')
plt.close()
print(f"热图已保存至: {os.path.join(OUTPUT_DIR, 'Heatmap_top_genes.png')}")

# 保存表达矩阵
top_expr = pd.DataFrame(adata_sub.X.toarray(),
                        index=adata_sub.obs_names,
                        columns=top_gene_symbols)
top_expr.to_csv(os.path.join(OUTPUT_DIR, "Top_genes_expression_matrix.csv"))

print("\n🎉 所有分析完成！")
print(f"差异分析结果保存在: {OUTPUT_DIR}")

In [ ]:
# ==============================================
# HA vs OA 单细胞差异分析 + 火山图 + 热图（适配 config.py + 双ID格式）
# 读取 data/processed/HA_vs_OA/ 合并数据（Ensembl ID+Symbol）
# 输出到 output/HA_vs_OA_diff/
# ==============================================
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.sparse import load_npz
import scanpy as sc
from matplotlib import rcParams

# 导入 config 路径配置
from config import DATA_PROCESSED, RESULT_DIR

# ===================== 路径配置 =====================
COMBINED_DIR = os.path.join(DATA_PROCESSED, "HA_vs_OA")
OUTPUT_DIR = os.path.join(RESULT_DIR, "HA_vs_OA_diff")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 设置 scanpy 输出详细进度
sc.settings.verbosity = 3

# 设置绘图风格
rcParams['figure.figsize'] = (8, 6)
rcParams['font.size'] = 10
sns.set_style("whitegrid")

# ===================== 1. 加载数据（适配双ID格式：Ensembl ID + Symbol） =====================
print("加载稀疏矩阵和元数据...")
X = load_npz(os.path.join(COMBINED_DIR, "HA_vs_OA_combined_sparse.npz"))
combined_gene_df = pd.read_csv(os.path.join(COMBINED_DIR, "HA_vs_OA_combined_genes.csv"))
gene_ids = combined_gene_df["gene_id"].tolist()
gene_symbols = combined_gene_df["gene_symbol"].tolist()
id2symbol = dict(zip(gene_ids, gene_symbols))

cells = pd.read_csv(os.path.join(COMBINED_DIR, "HA_vs_OA_combined_cells.csv"), header=None)[0].tolist()
group_info = pd.read_csv(os.path.join(COMBINED_DIR, "HA_vs_OA_group_info.csv"))

# 确保顺序一致
group_info = group_info.set_index("cell_id").loc[cells]
print(f"矩阵形状: {X.shape} 基因 × 细胞")
print(f"HA 细胞数: {(group_info['group'] == 'HA').sum()}")
print(f"OA 细胞数: {(group_info['group'] == 'OA').sum()}")

# ===================== 2. 构建 AnnData 对象 =====================
adata = sc.AnnData(
    X=X.T.astype(np.float32),
    obs=group_info,
    var=combined_gene_df.set_index("gene_id")
)
print("AnnData 构建完成")

# ===================== 3. 质量控制与标准化 =====================
sc.pp.filter_cells(adata, min_genes=200)
sc.pp.filter_genes(adata, min_cells=3)

sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

# ===================== 4. 差异分析 =====================
print("开始差异分析...")
sc.tl.rank_genes_groups(adata, groupby='group', groups=['OA'], reference='HA',
                        method='wilcoxon', use_raw=False, n_jobs=4)

# 提取结果
de_df = sc.get.rank_genes_groups_df(adata, group='OA')
print("原始列名:", de_df.columns.tolist())

# 重命名列并添加基因Symbol
if 'logfoldchanges' in de_df.columns:
    de_df = de_df.rename(columns={
        'names': 'gene_id',
        'logfoldchanges': 'log2FC',
        'pvals': 'p_val',
        'pvals_adj': 'p_val_adj'
    })
    if 'scores' in de_df.columns:
        de_df = de_df.drop(columns=['scores'])
else:
    de_df.columns = ['gene_id', 'scores', 'log2FC', 'p_val', 'p_val_adj']
    de_df = de_df.drop(columns=['scores'])

de_df["gene_symbol"] = de_df["gene_id"].map(id2symbol)
de_df = de_df[["gene_id", "gene_symbol", "log2FC", "p_val", "p_val_adj"]]
de_df = de_df.sort_values('p_val_adj')

de_df.to_csv(os.path.join(OUTPUT_DIR, "HA_vs_OA_diff_all.csv"), index=False)
print(f"差异分析完成，共 {len(de_df)} 个基因")

# 筛选显著差异基因
sig_up = de_df[(de_df['log2FC'] > 1) & (de_df['p_val_adj'] < 0.05)]
sig_down = de_df[(de_df['log2FC'] < -1) & (de_df['p_val_adj'] < 0.05)]
sig_all = pd.concat([sig_up, sig_down])
print(f"显著上调基因: {len(sig_up)}")
print(f"显著下调基因: {len(sig_down)}")

# 保存显著基因
sig_up.to_csv(os.path.join(OUTPUT_DIR, "HA_vs_OA_upregulated_genes.csv"), index=False)
sig_down.to_csv(os.path.join(OUTPUT_DIR, "HA_vs_OA_downregulated_genes.csv"), index=False)
sig_all.to_csv(os.path.join(OUTPUT_DIR, "HA_vs_OA_significant_genes.csv"), index=False)

# ===================== 5. 火山图（修正版） =====================
fig, ax = plt.subplots(figsize=(10, 8))

# 1. 绘制散点点 (保持你的逻辑不变)
not_sig = de_df[~((de_df['p_val_adj'] < 0.05) & (abs(de_df['log2FC']) > 1))]
up_sig = de_df[(de_df['log2FC'] > 1) & (de_df['p_val_adj'] < 0.05)]
down_sig = de_df[(de_df['log2FC'] < -1) & (de_df['p_val_adj'] < 0.05)]

# 2. 修正绘图逻辑：严格遵守 "红高蓝低"
ax.scatter(not_sig['log2FC'], -np.log10(not_sig['p_val_adj']),
           c='lightgray', alpha=0.5, s=5, label='Not significant')
# 红点表示上调 (OA高)
ax.scatter(up_sig['log2FC'], -np.log10(up_sig['p_val_adj']),
           c='red', alpha=0.8, s=20, label=f'Upregulated ({len(up_sig)})')
# 蓝点表示下调 (OA低)
ax.scatter(down_sig['log2FC'], -np.log10(down_sig['p_val_adj']),
           c='blue', alpha=0.8, s=20, label=f'Downregulated ({len(down_sig)})')

# 3. 修正图例展示 - 调整图例顺序使其符合视觉层级
handles, labels = ax.get_legend_handles_labels()
# 重新排序：Not sig (灰) -> Up (红) -> Down (蓝)
ax.legend(handles=[handles[0], handles[1], handles[2]],
          labels=['Not significant', f'Upregulated ({len(up_sig)})', f'Downregulated ({len(down_sig)})'],
          fontsize=12)

# 4. 添加标题并加粗
ax.set_title('Volcano Plot: OA vs HA', fontsize=16, fontweight='bold', pad=20)
ax.set_xlabel('log2 Fold Change (OA vs HA)', fontsize=14)
ax.set_ylabel('-log10(Adjusted P-value)', fontsize=14)

# 保存
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "Volcano_plot.png"), dpi=300)
plt.close()
print("火山图已保存")

# ===================== 6. 热图（最终修复版：符合单细胞热图行业标准） =====================
print("绘制标准单细胞差异基因热图...")
# 1. 获取Top差异基因ID（按log2FC排序）
top_up_ids = sig_up.nlargest(20, 'log2FC')['gene_id'].tolist()  # 上调（OA高），从大到小
top_down_ids = sig_down.nsmallest(20, 'log2FC')['gene_id'].tolist()  # 下调（OA低），从小到大
top_gene_ids = top_up_ids + top_down_ids  # 上调在上，下调在下

# 2. 过滤QC中被删除的基因
existing_genes = adata.var_names.tolist()
top_gene_ids_filtered = [g for g in top_gene_ids if g in existing_genes]
top_gene_symbols = [id2symbol[g] for g in top_gene_ids_filtered]
print(f"热图展示基因数: {len(top_gene_ids_filtered)}")

# 3. 下采样细胞（平衡两组细胞数）
np.random.seed(42)
n_cells_sample = 200
ha_cells = adata.obs[adata.obs['group'] == 'HA'].index
oa_cells = adata.obs[adata.obs['group'] == 'OA'].index
ha_sample = np.random.choice(ha_cells, min(n_cells_sample, len(ha_cells)), replace=False)
oa_sample = np.random.choice(oa_cells, min(n_cells_sample, len(oa_cells)), replace=False)
sample_cells = np.concatenate([ha_sample, oa_sample])

# 4. 提取表达矩阵
adata_sub = adata[sample_cells, top_gene_ids_filtered].copy()
expr_matrix = pd.DataFrame(
    adata_sub.X.toarray().T,  # 基因 × 细胞
    index=top_gene_symbols,
    columns=adata_sub.obs_names
)

# 5. 【核心修复1：正确Z-score标准化（按基因/行）】
expr_zscore = expr_matrix.apply(lambda x: (x - x.mean()) / x.std(), axis=1)

# 6. 按分组排序细胞（HA左，OA右）
group_order = adata_sub.obs.sort_values('group').index
expr_zscore = expr_zscore[group_order]
group_labels = adata_sub.obs.loc[group_order, 'group']

# 7. 【核心修复2：正确配色（红高蓝低，RdBu行业标准）】
group_color_map = {'HA': '#1f77b4', 'OA': '#ff7f0e'}
col_colors = group_labels.map(group_color_map)

# 8. 绘制标准单细胞热图
g = sns.clustermap(
    expr_zscore,
    cmap="RdBu",  # 【关键修复】红高、蓝低，符合行业标准
    vmin=-2.5, vmax=2.5,  # Z-score标准范围
    row_cluster=False,  # 保持上调/下调顺序，不聚类
    col_cluster=False,  # 保持分组顺序，不聚类
    col_colors=col_colors,  # 分组条
    figsize=(16, 10),
    yticklabels=True,  # 显示基因名
    xticklabels=False,  # 隐藏重叠的细胞ID
    cbar_pos=(0.02, 0.8, 0.03, 0.18),  # 颜色条放在右侧
    dendrogram_ratio=(0.1, 0.05),
    linewidths=0.1,
    cbar_kws={'label': 'Z-score (Expression Level)'}  # 颜色条标签
)

# 9. 添加分组图例
for group, color in group_color_map.items():
    g.ax_col_dendrogram.bar(0, 0, color=color, label=group, linewidth=0)
g.ax_col_dendrogram.legend(loc="upper right", bbox_to_anchor=(1.2, 1), fontsize=12)

# 10. 设置标题（学术规范）
g.fig.suptitle('Top 20 Up/Down-regulated Genes (OA vs HA)', y=1.02, fontsize=16, fontweight='bold')

# 11. 保存热图
g.savefig(
    os.path.join(OUTPUT_DIR, "Heatmap_top_genes_corrected.png"),
    dpi=300,
    bbox_inches='tight',
    facecolor='white'
)
plt.close()
print(f"修复后的热图已保存至: {os.path.join(OUTPUT_DIR, 'Heatmap_top_genes_corrected.png')}")

# 12. 保存标准化后的表达矩阵
top_expr = pd.DataFrame(expr_zscore.T, index=expr_zscore.columns, columns=expr_zscore.index)
top_expr.to_csv(os.path.join(OUTPUT_DIR, "Top_genes_expression_matrix_zscore.csv"))

print("\n🎉 所有分析完成！")
print(f"差异分析结果保存在: {OUTPUT_DIR}")

### 人 关键基因 韦恩图

### 人 KEGG

In [ ]:
# ==============================================
# 多个数据集差异基因整合与韦恩图（适配 config.py）
# 增强：提取交集基因的差异统计信息，并输出 top20 交集基因
# 【已修改】基于 Ensembl ID 取交集 | 同时保存 Ensembl ID + Symbol
# 输入：output/各GSE文件夹中的差异基因CSV
# 输出：output/human_core_genes/ 下的整合结果，包含交集基因的统计信息和 top20
# ==============================================
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
from matplotlib_venn import venn2
from config import RESULT_DIR

# ===================== 路径配置（完全不变） =====================
# 各数据集差异基因所在目录（output下）
GSE104589_DIR = os.path.join(RESULT_DIR, "GSE104589")
GSE149315_DIR = os.path.join(RESULT_DIR, "GSE149315")
GSE171542_DIR = os.path.join(RESULT_DIR, "GSE171542")
GSE276597_DIR = os.path.join(RESULT_DIR, "GSE276597")
HA_DIR = os.path.join(RESULT_DIR, "HA_vs_OA_diff")   # HA_vs_OA 差异分析结果目录

# 输出目录（完全不变）
OUTPUT_DIR = os.path.join(RESULT_DIR, "human_core_genes")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ===================== 通用读取函数（双ID：ensembl_id + symbol） =====================
def get_diff_genes_ensembl(up_file, down_file):
    """
    读取上调/下调差异基因，基于 ensembl_id 去重
    返回：ensembl_id集合, {ensembl_id: symbol} 字典
    """
    up = pd.read_csv(up_file)
    down = pd.read_csv(down_file)
    # 合并上下调，按ensembl_id去重
    df = pd.concat([up, down], ignore_index=True).drop_duplicates(subset="ensembl_id")
    ensembl_set = set(df["ensembl_id"].dropna().astype(str))
    id_symbol_dict = df.set_index("ensembl_id")["symbol"].to_dict()
    return ensembl_set, id_symbol_dict

# ===================== 1. GSE104589（ensembl_id + symbol）【✅ 修复核心报错】 =====================
print("="*50)
print("读取 GSE104589 差异基因...")
# 匹配你的输出文件名
# 第一组：Control vs UHMWPE
gse104589_up_UHMWPE = os.path.join(GSE104589_DIR, "GSE104589_Control_vs_UHMWPE_上调.csv")
gse104589_down_UHMWPE = os.path.join(GSE104589_DIR, "GSE104589_Control_vs_UHMWPE_下调.csv")
# 第二组：Control vs VE-UHMWPE
gse104589_up_VE_UHMWPE = os.path.join(GSE104589_DIR, "GSE104589_Control_vs_VE-UHMWPE_上调.csv")
gse104589_down_VE_UHMWPE = os.path.join(GSE104589_DIR, "GSE104589_Control_vs_VE-UHMWPE_下调.csv")

# 分别读取两组差异基因，再取并集（修复函数参数不匹配问题）
genes_uhmwpe_ens, dict_uhmwpe = get_diff_genes_ensembl(gse104589_up_UHMWPE, gse104589_down_UHMWPE)
genes_ve_uhmwpe_ens, dict_ve_uhmwpe = get_diff_genes_ensembl(gse104589_up_VE_UHMWPE, gse104589_down_VE_UHMWPE)

# 合并两组的并集 + 合并基因字典
genes_104589_ens = genes_uhmwpe_ens.union(genes_ve_uhmwpe_ens)
dict_104589 = {**dict_uhmwpe, **dict_ve_uhmwpe}

print(f"  Control vs UHMWPE 差异基因数: {len(genes_uhmwpe_ens)}")
print(f"  Control vs VE-UHMWPE 差异基因数: {len(genes_ve_uhmwpe_ens)}")
print(f"GSE104589 总差异基因数（并集）: {len(genes_104589_ens)}")

# ===================== 2. GSE149315 (circRNA宿主基因，已转双ID) =====================
print("\n读取 GSE149315 差异基因（已转换宿主基因+Ensembl）...")
gse149315_up = os.path.join(GSE149315_DIR, "GSE149315_显著上调circRNA_GeneSymbol_Ensembl.csv")
gse149315_down = os.path.join(GSE149315_DIR, "GSE149315_显著下调circRNA_GeneSymbol_Ensembl.csv")
up_149 = pd.read_csv(gse149315_up)
down_149 = pd.read_csv(gse149315_down)
df_149 = pd.concat([up_149, down_149], ignore_index=True).drop_duplicates(subset="ensembl_id")
genes_149315_ens = set(df_149["ensembl_id"].dropna().astype(str))
dict_149315 = df_149.set_index("ensembl_id")["gene_symbol"].to_dict()
print(f"GSE149315 差异基因数: {len(genes_149315_ens)}")

# ===================== 3. GSE171542（ensembl_id + symbol） =====================
print("\n读取 GSE171542 差异基因（RANKL vs Control 和 TYMP vs Control 的并集）...")
# RANKL分组
rankl_up = os.path.join(GSE171542_DIR, "GSE171542_上调基因_RANKL_vs_Control.csv")
rankl_down = os.path.join(GSE171542_DIR, "GSE171542_下调基因_RANKL_vs_Control.csv")
genes_rankl_ens, dict_rankl = get_diff_genes_ensembl(rankl_up, rankl_down)
# TYMP分组
tymp_up = os.path.join(GSE171542_DIR, "GSE171542_上调基因_TYMP_vs_Control.csv")
tymp_down = os.path.join(GSE171542_DIR, "GSE171542_下调基因_TYMP_vs_Control.csv")
genes_tymp_ens, dict_tymp = get_diff_genes_ensembl(tymp_up, tymp_down)
# 合并并集
genes_171542_ens = genes_rankl_ens.union(genes_tymp_ens)
dict_171542 = {**dict_rankl, **dict_tymp}
print(f"  RANKL vs Control 差异基因数: {len(genes_rankl_ens)}")
print(f"  TYMP vs Control 差异基因数: {len(genes_tymp_ens)}")
print(f"  GSE171542 总差异基因数（并集）: {len(genes_171542_ens)}")

# ===================== 4. GSE276597（ensembl_id + symbol） =====================
print("\n读取 GSE276597 差异基因...")
gse276597_up = os.path.join(GSE276597_DIR, "GSE276597_上调基因_Case_vs_Control.csv")
gse276597_down = os.path.join(GSE276597_DIR, "GSE276597_下调基因_Case_vs_Control.csv")
genes_276597_ens, dict_276597 = get_diff_genes_ensembl(gse276597_up, gse276597_down)
print(f"GSE276597 差异基因数: {len(genes_276597_ens)}")

# ===================== 5. HA_vs_OA（gene_id=Ensembl + gene_symbol） =====================
print("\n读取 HA_vs_OA 显著差异基因...")
ha_file = os.path.join(HA_DIR, "HA_vs_OA_significant_genes.csv")
ha_df = pd.read_csv(ha_file)
# 重命名为标准ensembl列名，统一格式
ha_df = ha_df.rename(columns={"gene_id": "ensembl_id", "gene_symbol": "symbol"})
ha_df = ha_df.drop_duplicates(subset="ensembl_id")
genes_ha_ens = set(ha_df["ensembl_id"].dropna().astype(str))
dict_ha = ha_df.set_index("ensembl_id")["symbol"].to_dict()
print(f"HA_vs_OA 显著差异基因数: {len(genes_ha_ens)}")

# ===================== 6. 四个数据集的总并集（Ensembl ID） =====================
union_four_ens = genes_104589_ens | genes_149315_ens | genes_171542_ens | genes_276597_ens
# 合并所有ID-Symbol映射
all_id_dict = {}
all_id_dict.update(dict_104589)
all_id_dict.update(dict_149315)
all_id_dict.update(dict_171542)
all_id_dict.update(dict_276597)

print(f"\n四个数据集差异基因总并集(Ensembl ID): {len(union_four_ens)}")

# 保存各数据集差异基因列表（双ID，文件名完全不变）
def save_dual_id(gene_set, id_dict, filename):
    df = pd.DataFrame({
        "ensembl_id": list(gene_set),
        "symbol": [id_dict.get(g, "") for g in gene_set]
    }).sort_values("ensembl_id")
    df.to_csv(os.path.join(OUTPUT_DIR, filename), index=False, encoding="utf-8-sig")

save_dual_id(genes_104589_ens, dict_104589, "GSE104589_diff_genes.csv")
save_dual_id(genes_149315_ens, dict_149315, "GSE149315_diff_genes.csv")
save_dual_id(genes_171542_ens, dict_171542, "GSE171542_diff_genes.csv")
save_dual_id(genes_276597_ens, dict_276597, "GSE276597_diff_genes.csv")
save_dual_id(union_four_ens, all_id_dict, "four_datasets_union_genes.csv")

# ===================== 7. 与 HA_vs_OA 取交集（Ensembl ID），保留完整统计信息 =====================
common_ens = union_four_ens & genes_ha_ens
print(f"\n四个数据集并集与 HA_vs_OA 交集基因数(Ensembl ID): {len(common_ens)}")

# 筛选交集基因（保留所有统计列 + 双ID）
common_df = ha_df[ha_df["ensembl_id"].isin(common_ens)].copy()
# 保存完整差异信息（文件名完全不变）
common_df.to_csv(os.path.join(OUTPUT_DIR, "human_core_genes_intersection_with_stats.csv"), index=False, encoding="utf-8-sig")

# 按绝对 log2FC 降序排序，取前20（双ID保留）
common_df['abs_log2FC'] = common_df['log2FC'].abs()
top20_by_fc = common_df.sort_values('abs_log2FC', ascending=False).head(20).drop(columns=['abs_log2FC'])
top20_by_fc.to_csv(os.path.join(OUTPUT_DIR, "human_core_genes_top20_by_logFC.csv"), index=False, encoding="utf-8-sig")

# 按 padj 升序排序取前20
top20_by_padj = common_df.sort_values('p_val_adj').head(20)
top20_by_padj.to_csv(os.path.join(OUTPUT_DIR, "human_core_genes_top20_by_padj.csv"), index=False, encoding="utf-8-sig")

# 保存仅基因名列表（双ID）
top20_by_fc[["ensembl_id", "symbol"]].to_csv(
    os.path.join(OUTPUT_DIR, "human_core_genes_top20_names.csv"), index=False, encoding="utf-8-sig"
)

print(f"\n✅ Top 20 交集基因（按 |log2FC| 排序）已保存")
print(f"✅ Top 20 交集基因（按 padj 排序）已保存")

# ===================== 8. 韦恩图（完全不变） =====================
plt.figure(figsize=(8, 6))
venn2(subsets=(
    len(union_four_ens - common_ens),
    len(genes_ha_ens - common_ens),
    len(common_ens)
),
set_labels=('Four datasets (union)', 'HA_vs_OA'))
plt.title('Overlap of Differential Genes', fontsize=14)
plt.savefig(os.path.join(OUTPUT_DIR, "Venn_union_four_vs_HA.png"), dpi=300, bbox_inches='tight')
plt.close()

# ===================== 完成输出（完全不变） =====================
print("\n🎉 整合完成！所有输出文件保存在：", OUTPUT_DIR)
print("文件列表：")
print("  - GSE104589_diff_genes.csv")
print("  - GSE149315_diff_genes.csv")
print("  - GSE171542_diff_genes.csv")
print("  - GSE276597_diff_genes.csv")
print("  - four_datasets_union_genes.csv")
print("  - human_core_genes_intersection_with_stats.csv    (完整差异信息)")
print("  - human_core_genes_top20_by_logFC.csv             (按 |log2FC| 排序的 top20)")
print("  - human_core_genes_top20_by_padj.csv              (按 padj 排序的 top20)")
print("  - human_core_genes_top20_names.csv                (仅 top20 双ID)")
print("  - Venn_union_four_vs_HA.png")

In [ ]:
# ==============================================
# 人类交集基因的 KEGG 富集分析（全部基因）+ PPI 边列表生成
# 【已修复】列名错误 + 变量定义 + 规范读取Ensembl ID
# ==============================================

import os
import sys
import subprocess
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import requests
from pathlib import Path

# 导入 config
from config import RESULT_DIR

# 设置输出目录
OUTPUT_DIR = Path(RESULT_DIR) / "human_core_genes"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 输入文件
GENE_STATS_FILE = OUTPUT_DIR / "human_core_genes_intersection_with_stats.csv"

# ===================== 辅助函数 =====================
def install_pip_package(package):
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])
        print(f"✅ {package} 安装成功")
        return True
    except Exception as e:
        print(f"❌ {package} 安装失败: {e}")
        return False

# ===================== 1. 读取基因列表（✅ 修复：标准读取ensembl_id）=====================
print("=" * 50)
print("读取人类全部交集基因...")
print("=" * 50)
gene_stats_df = pd.read_csv(GENE_STATS_FILE)
# 🔥 修复：强制读取你的标准列 ensembl_id
gene_list = gene_stats_df['ensembl_id'].dropna().astype(str).tolist()
print(f"待分析基因数: {len(gene_list)}")

# ===================== 2. 依赖包 =====================
try:
    from gseapy import enrichr
    gseapy_available = True
except ImportError:
    install_pip_package("gseapy")
    from gseapy import enrichr
    gseapy_available = True

# ===================== 3. clusterProfiler 富集分析 =====================
print("\n开始 KEGG 富集分析...")
use_clusterProfiler = False
result_df = pd.DataFrame()
plot_df = pd.DataFrame()  # 🔥 修复：提前定义，避免未定义报错

try:
    import rpy2.robjects as ro
    from rpy2.robjects import pandas2ri
    from rpy2.robjects.conversion import localconverter
    from rpy2.robjects.packages import importr, isinstalled
    rpy2_available = True
except ImportError:
    rpy2_available = False

if rpy2_available and isinstalled('clusterProfiler') and isinstalled('org.Hs.eg.db'):
    use_clusterProfiler = True
    ro.r('options(stringsAsFactors = FALSE)')
    clusterProfiler = importr('clusterProfiler')
    org_Hs_eg_db = ro.r['org.Hs.eg.db']

    try:
        # ID转换
        gene_vector = ro.StrVector(gene_list)
        entrez_df = clusterProfiler.bitr(gene_vector, fromType="ENSEMBL", toType="ENTREZID", OrgDb=org_Hs_eg_db)
        entrez_ids = entrez_df.rx2('ENTREZID')
        print(f"✅ 成功转换 {len(entrez_ids)} / {len(gene_list)} 个基因为 Entrez ID")

        # 富集分析（宽松阈值）
        enrich_result = clusterProfiler.enrichKEGG(gene=entrez_ids, organism='hsa', pvalueCutoff=1, qvalueCutoff=1)
        if not ro.r('is.null')(enrich_result)[0]:
            r_df = ro.r('as.data.frame')(enrich_result)
            with localconverter(ro.default_converter + pandas2ri.converter):
                result_df = pandas2ri.rpy2py(r_df)
            
            result_df.to_csv(OUTPUT_DIR / "kegg_enrichment_results_full.csv", index=False)
            print(f"✅ KEGG 富集结果已保存，共 {len(result_df)} 条通路（未过滤）")

            # 🔥 修复：列名 pvalue（无点）！！！核心bug修复
            sig_df = result_df[result_df['p.adjust'] < 0.05].copy()
            if sig_df.empty:
                print("⚠️ 未发现显著富集的通路（padj < 0.05），展示前20条通路")
                result_df_sorted = result_df.sort_values('pvalue')  # 这里修复！
                plot_df = result_df_sorted.head(20).copy()
                plot_df['Description'] = plot_df['Description'] + " (NS)"
            else:
                plot_df = sig_df.sort_values('p.adjust').head(20).copy()

    except Exception as e:
        print(f"❌ clusterProfiler 分析出错: {e}")

# ===================== 4. 可视化（✅ 修复后必运行）=====================
if not plot_df.empty:
    # 条形图
    fig, ax = plt.subplots(figsize=(10, 8))
    ax.barh(plot_df['Description'], plot_df['Count'], color='steelblue')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "kegg_barplot.png", dpi=300, bbox_inches='tight')
    plt.close()
    print("✅ KEGG 条形图已保存")

    # 气泡图
    fig, ax = plt.subplots(figsize=(12, 8))
    scatter = ax.scatter(plot_df['Count']/plot_df['Count'].max(), plot_df['Description'],
                         s=plot_df['Count']*20, c=-np.log10(plot_df['pvalue']), cmap='viridis')
    plt.colorbar(scatter, label='-log10(pvalue)')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "kegg_bubble.png", dpi=300, bbox_inches='tight')
    plt.close()
    print("✅ KEGG 气泡图已保存")
else:
    print("⚠️ 没有通路结果，跳过可视化。")

# ===================== 5. PPI 网络（无修改，正常运行）=====================
print("\n生成 PPI 网络边列表...")
def get_string_interactions(gene_list, species=9606, score_threshold=400):
    identifiers = "\n".join(gene_stats_df['symbol'].tolist()) # 🔥 用Symbol，STRING不支持Ensembl
    url = "https://string-db.org/api/tsv/network"
    params = {"identifiers": identifiers, "species": species, "required_score": score_threshold}
    response = requests.post(url, data=params)
    lines = response.text.strip().split('\n')
    header = lines[0].split('\t')
    edges = []
    for line in lines[1:]:
        row = line.split('\t')
        edges.append({'node1': row[2], 'node2': row[3], 'score': float(row[5])})
    return pd.DataFrame(edges)

ppi_edges = get_string_interactions(gene_list)
if not ppi_edges.empty:
    ppi_edges.to_csv(OUTPUT_DIR / "ppi_edges.csv", index=False)
    nodes = set(ppi_edges['node1']).union(set(ppi_edges['node2']))
    pd.DataFrame({'Gene': sorted(nodes)}).to_csv(OUTPUT_DIR / "ppi_nodes.csv", index=False)
    print(f"✅ PPI 边：{len(ppi_edges)} 条，节点：{len(nodes)} 个")

print("\n🎉 分析完成！")

### 人 PPI

In [ ]:
# ==============================================
# PPI 网络分析：基于全部交集基因的 PPI 边列表，可视化仅显示 top100 节点
# 【100% 适配上方富集+PPI生成代码】
# 输入：output/human_core_genes/ppi_edges.csv
# 输出：节点指标表，核心基因列表，网络图（top100），GraphML 文件
# ==============================================

import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from pathlib import Path
from config import RESULT_DIR

# ===================== 路径配置（与上方代码完全一致） =====================
OUTPUT_DIR = Path(RESULT_DIR) / "human_core_genes"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
EDGES_FILE = OUTPUT_DIR / "ppi_edges.csv"

# ===================== 1. 读取边列表并构建网络 =====================
print("=" * 50)
print("读取 PPI 边列表（全部节点）...")
print("=" * 50)

# 检查文件是否存在
if not EDGES_FILE.exists():
    print("❌ 错误：未找到 ppi_edges.csv 文件，请先运行上方的 PPI 生成代码！")
    exit()

edges_df = pd.read_csv(EDGES_FILE)
print(f"原始边数: {len(edges_df)}")

# 创建无向图
G = nx.Graph()
for idx, row in edges_df.iterrows():
    G.add_edge(row['node1'], row['node2'], weight=row['score'])

print(f"网络节点数: {G.number_of_nodes()}")
print(f"网络边数: {G.number_of_edges()}")

# ===================== 2. 计算拓扑指标（基于全图） =====================
print("\n" + "=" * 50)
print("计算节点拓扑指标...")
print("=" * 50)

if G.number_of_nodes() > 0:
    degree_centrality = nx.degree_centrality(G)
    betweenness_centrality = nx.betweenness_centrality(G, normalized=True)
    closeness_centrality = nx.closeness_centrality(G)
    clustering_coeff = nx.clustering(G)
    degree_raw = dict(G.degree())

    node_metrics = []
    for node in G.nodes():
        node_metrics.append({
            'Gene': node,
            'Degree': degree_raw[node],
            'Degree_Centrality': degree_centrality[node],
            'Betweenness_Centrality': betweenness_centrality[node],
            'Closeness_Centrality': closeness_centrality[node],
            'Clustering_Coefficient': clustering_coeff[node]
        })

    metrics_df = pd.DataFrame(node_metrics).sort_values('Degree_Centrality', ascending=False)
    metrics_df.to_csv(OUTPUT_DIR / "ppi_node_metrics_full.csv", index=False)
    print(f"✅ 节点指标已保存: ppi_node_metrics_full.csv")
else:
    print("⚠️ 网络无节点，跳过指标计算")
    metrics_df = pd.DataFrame()

# ===================== 3. 输出核心基因列表（Top 10 by Degree） =====================
if not metrics_df.empty:
    top_genes = metrics_df.head(10)['Gene'].tolist()
    pd.DataFrame({'Gene': top_genes}).to_csv(OUTPUT_DIR / "top10_hub_genes_full.csv", index=False)
    print(f"✅ Top 10 核心基因已保存: top10_hub_genes_full.csv")
    print("核心基因:", top_genes)
else:
    print("⚠️ 无核心基因可输出")

# ===================== 4. 提取度数最高的前100个节点，绘制子图 =====================
print("\n" + "=" * 50)
print("绘制 PPI 网络图（仅显示度数最高的前100个节点）...")
print("=" * 50)

if G.number_of_nodes() > 0:
    # 按度数降序排序节点
    sorted_degrees = sorted(G.degree, key=lambda x: x[1], reverse=True)
    top_n = min(100, G.number_of_nodes())
    top_nodes = [node for node, deg in sorted_degrees[:top_n]]
    G_sub = G.subgraph(top_nodes)

    num_nodes = G_sub.number_of_nodes()
    num_edges = G_sub.number_of_edges()
    print(f"子图节点数: {num_nodes}")
    print(f"子图边数: {num_edges}")

    # 动态调整图形参数
    fig_size = max(10, min(30, num_nodes // 10))
    node_size = max(50, min(300, 5000 // max(1, num_nodes // 20)))
    font_size = max(6, min(12, 200 // max(1, num_nodes // 20)))

    plt.figure(figsize=(fig_size, fig_size))
    pos = nx.spring_layout(G_sub, k=2 / num_nodes**0.3, iterations=100, seed=42)

    # 节点/边样式
    node_colors = [degree_centrality[node] for node in G_sub.nodes()] if num_nodes > 0 else []
    if G_sub.edges():
        max_weight = max([G_sub[u][v]['weight'] for u, v in G_sub.edges()])
        edge_widths = [G_sub[u][v]['weight'] / max_weight * 2 for u, v in G_sub.edges()]
    else:
        edge_widths = []

    # 绘图
    nx.draw_networkx_nodes(G_sub, pos, node_size=node_size, node_color=node_colors, cmap='coolwarm', alpha=0.9)
    if edge_widths:
        nx.draw_networkx_edges(G_sub, pos, width=edge_widths, alpha=0.5, edge_color='gray')

    # 智能标签：节点少全标，节点多只标Top30
    if num_nodes <= 50:
        nx.draw_networkx_labels(G_sub, pos, font_size=font_size, font_family='sans-serif')
    else:
        sub_degrees = sorted(G_sub.degree, key=lambda x: x[1], reverse=True)[:30]
        top_labels = {node: node for node, deg in sub_degrees}
        nx.draw_networkx_labels(G_sub, pos, labels=top_labels, font_size=font_size, font_family='sans-serif')

    plt.title(f'Human PPI Network (Top {top_n} hub genes, {num_edges} edges)', fontsize=14)
    plt.axis('off')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "ppi_network_top100.png", dpi=300, bbox_inches='tight')
    plt.close()
    print("✅ 网络图（top100）已保存: ppi_network_top100.png")
else:
    print("⚠️ 无网络数据，跳过绘图")

# ===================== 5. 保存完整网络文件（GraphML） =====================
if G.number_of_nodes() > 0:
    nx.write_graphml(G, OUTPUT_DIR / "ppi_network_full.graphml")
    print("✅ 完整网络文件已保存: ppi_network_full.graphml (可导入 Cytoscape)")

# ===================== 完成 =====================
print("\n🎉 PPI 分析完成！所有输出文件保存在：", OUTPUT_DIR)
print("文件列表：")
print("  - ppi_node_metrics_full.csv        (所有节点的拓扑指标)")
print("  - top10_hub_genes_full.csv         (Top10 核心基因)")
print("  - ppi_network_top100.png           (网络可视化图)")
print("  - ppi_network_full.graphml         (Cytoscape 专用文件)")



## 鼠
### GSE234863



In [ ]:
# ==============================================
# GSE234863 差异表达分析（DESeq2 标准流程）
# 适配 config.py 路径规范 | 双ID：ENSEMBL ID + Symbol
# 输出：差异结果 → output/GSE234863/，表达矩阵 → data/processed/GSE234863/
# ==============================================
import pandas as pd
import numpy as np
import os
import re
import warnings
warnings.filterwarnings('ignore')

# rpy2 相关
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri
from rpy2.robjects.conversion import localconverter
from rpy2.robjects.packages import importr, isinstalled

# 导入 config
from config import DATA_PROCESSED, RESULT_DIR, MOUSE_LOOSE_VS_STABLE_GSE234863

# 自动安装 DESeq2 依赖（若未安装）
utils = importr('utils')
if not isinstalled('BiocManager'):
    utils.install_packages('BiocManager', quiet=True)
if not isinstalled('DESeq2'):
    print("正在安装 DESeq2 ...")
    ro.r('BiocManager::install("DESeq2", update=FALSE, ask=FALSE)')

# ===================== 路径配置 =====================
# 原始数据文件
RAW_DATA_PATH = os.path.join(MOUSE_LOOSE_VS_STABLE_GSE234863, "GSE234863_all.genes.expression.annot.txt.gz")

# 输出目录
PROCESSED_DIR = os.path.join(DATA_PROCESSED, "GSE234863")
OUTPUT_DIR = os.path.join(RESULT_DIR, "GSE234863")
os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ===================== 实验设计 =====================
CONTROL_GROUP = "Naive"
CASE_GROUP    = "Ti"
P_THRESHOLD   = 0.05
LOG2FC_THRESHOLD = 1  # 相当于 2倍变化

# ===================== 1. 读取数据 + 标准化双ID（ENSEMBL ID + Symbol） =====================
print("="*50)
print("读取 GSE234863 原始数据...")
print("="*50)
df_raw = pd.read_csv(RAW_DATA_PATH, sep="\t", compression="infer", low_memory=False)
print(f"原始形状：{df_raw.shape}")

# 重命名核心ID列：id → ensembl_id（明确为ENSEMBL ID）
df_raw = df_raw.rename(columns={"id": "ensembl_id"})
# 填充缺失的Symbol（无Symbol则用ENSEMBL ID代替）
if "Symbol" in df_raw.columns:
    df_raw["Symbol"] = df_raw["Symbol"].fillna(df_raw["ensembl_id"])
else:
    df_raw["Symbol"] = df_raw["ensembl_id"]

# 识别注释列 + 计数列
anno_keywords = ["ensembl_id", "Symbol", "Description", "KEGG", "Pathway", "GO", "TF", "Chr", "Start", "End", "Strand", "Length"]
anno_cols = [col for col in df_raw.columns if any(kw in col for kw in anno_keywords)]
all_sample_cols = [col for col in df_raw.columns if col not in anno_cols]
count_cols = [col for col in all_sample_cols if col.endswith("_count")]

if len(count_cols) == 0:
    raise ValueError("未找到 _count 列，请检查数据。")

# ===================== 2. 构建计数矩阵（索引=ENSEMBL ID，保留双ID映射） =====================
# 双ID映射字典（ENSEMBL ID → Symbol，后续所有表格共用）
id_symbol_map = df_raw.set_index("ensembl_id")["Symbol"].to_dict()

# 构建DESeq2输入矩阵：索引=ENSEMBL ID
count_df = df_raw.set_index("ensembl_id")[count_cols].copy()
count_df = count_df.apply(pd.to_numeric, errors='coerce').fillna(0).astype(int)

# 原始计数矩阵（包含ENSEMBL ID + Symbol）
raw_count_matrix = df_raw[["ensembl_id", "Symbol"] + count_cols].copy()

print(f"Count 矩阵形状：{count_df.shape}（基因 × 样本）")

# ===================== 3. 样本分组 =====================
GROUP_MAP = {}
for col in count_cols:
    base = col.replace("_count", "")
    group = base.split("-")[0]
    GROUP_MAP[col] = group

control_samples = [s for s, g in GROUP_MAP.items() if g == CONTROL_GROUP]
case_samples = [s for s, g in GROUP_MAP.items() if g == CASE_GROUP]
print(f"对照组 {CONTROL_GROUP} 样本：{control_samples}")
print(f"实验组 {CASE_GROUP} 样本：{case_samples}")

sample_order = control_samples + case_samples
count_df = count_df[sample_order]
condition = [CONTROL_GROUP] * len(control_samples) + [CASE_GROUP] * len(case_samples)

# ===================== 4. DESeq2 分析（基于 ENSEMBL ID） =====================
print("\n执行 DESeq2 分析...")
with localconverter(ro.default_converter + pandas2ri.converter):
    ro.globalenv['count_matrix'] = count_df
    ro.globalenv['condition'] = condition
    ro.r('''
        library(DESeq2)
        colData <- data.frame(condition = factor(condition, levels=c("Naive","Ti")))
        dds <- DESeqDataSetFromMatrix(countData = count_matrix,
                                      colData = colData,
                                      design = ~ condition)
        dds <- DESeq(dds)
        res <- results(dds, contrast=c("condition", "Ti", "Naive"))
        res_df <- as.data.frame(res)
        norm_counts <- counts(dds, normalized=TRUE)
    ''')
    res_df = ro.globalenv['res_df']
    norm_counts = ro.globalenv['norm_counts']

# 转换回 pandas DataFrame
with localconverter(ro.default_converter + pandas2ri.converter):
    res_pd = ro.conversion.rpy2py(res_df)
    norm_counts_array = ro.conversion.rpy2py(norm_counts)

# ===================== 5. 构建归一化表达矩阵（包含双ID） =====================
gene_ids = count_df.index.tolist()
sample_names = sample_order
# 归一化矩阵 + 双ID
norm_pd = pd.DataFrame(norm_counts_array, index=gene_ids, columns=sample_names)
norm_pd = norm_pd.reset_index().rename(columns={"index": "ensembl_id"})
norm_pd["Symbol"] = norm_pd["ensembl_id"].map(id_symbol_map)
# 调整列顺序：ENSEMBL ID + Symbol + 表达量
norm_pd = norm_pd[["ensembl_id", "Symbol"] + sample_names]

# ===================== 6. 整理差异结果（强制包含双ID + 所有分析指标） =====================
res_pd = res_pd.reset_index().rename(columns={"index": "ensembl_id"})
# 添加 Symbol 列
res_pd["Symbol"] = res_pd["ensembl_id"].map(id_symbol_map)

# 规范列顺序：双ID → 差异分析指标
required_cols = [
    "ensembl_id", "Symbol", "baseMean", "log2FoldChange", 
    "lfcSE", "stat", "pvalue", "padj"
]
# 补充缺失列
for col in required_cols:
    if col not in res_pd.columns:
        res_pd[col] = np.nan
res_pd = res_pd[required_cols]

# 填充缺失值
res_pd["padj"] = res_pd["padj"].fillna(1.0)
res_pd["pvalue"] = res_pd["pvalue"].fillna(1.0)

# 筛选差异基因
sig = (res_pd["padj"] < P_THRESHOLD) & (abs(res_pd["log2FoldChange"]) > LOG2FC_THRESHOLD)
up_genes = res_pd[sig & (res_pd["log2FoldChange"] > 0)].sort_values("log2FoldChange", ascending=False)
down_genes = res_pd[sig & (res_pd["log2FoldChange"] < 0)].sort_values("log2FoldChange", ascending=True)

print(f"\n✅ 上调基因数：{len(up_genes)}")
print(f"✅ 下调基因数：{len(down_genes)}")
print(f"✅ 总差异基因数：{len(up_genes) + len(down_genes)}")

# ===================== 7. 保存所有结果（双ID标准输出） =====================
gse_id = "GSE234863"

# 1. 差异分析全结果/上调/下调基因（均包含 ENSEMBL ID + Symbol）
res_pd.to_csv(os.path.join(OUTPUT_DIR, f"{gse_id}_DESeq2_full_results.csv"), index=False, encoding="utf-8-sig")
up_genes.to_csv(os.path.join(OUTPUT_DIR, f"{gse_id}_上调基因.csv"), index=False, encoding="utf-8-sig")
down_genes.to_csv(os.path.join(OUTPUT_DIR, f"{gse_id}_下调基因.csv"), index=False, encoding="utf-8-sig")

# 2. 原始计数矩阵（包含双ID）
raw_count_matrix.to_csv(os.path.join(PROCESSED_DIR, f"{gse_id}_原始计数矩阵.csv"), index=False, encoding="utf-8-sig")
# 3. 归一化表达矩阵（包含双ID）
norm_pd.to_csv(os.path.join(PROCESSED_DIR, f"{gse_id}_归一化表达矩阵.csv"), index=False, encoding="utf-8-sig")
# 4. 样本分组信息
group_info = pd.DataFrame({"sample_id": sample_order, "group": condition})
group_info.to_csv(os.path.join(PROCESSED_DIR, f"{gse_id}_样本分组信息.csv"), index=False, encoding="utf-8-sig")

# ===================== 完成输出 =====================
print("\n🎉 DESeq2 分析完成！")
print(f"📊 分析矩阵：{count_df.shape[0]} 基因 × {count_df.shape[1]} 样本")
print(f"🎯 差异分析基于：ENSEMBL ID")
print(f"🎨 可视化推荐使用：Symbol 列")
print(f"📁 差异结果保存在：{OUTPUT_DIR}")
print(f"📁 表达矩阵保存在：{PROCESSED_DIR}")

In [ ]:
# ==============================================
# 火山图 + 基因表达热图 绘制代码（适配 DESeq2 双ID输出）
# 基于 GSE234863 DESeq2 分析结果 | 全图使用 Symbol 展示
# 标题使用英文，路径严格匹配新输出文件
# ==============================================
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap

# 导入 config
from config import DATA_PROCESSED, RESULT_DIR

# ---------------------- 全局绘图设置 ----------------------
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 300
plt.rcParams['font.size'] = 10
sns.set_style("white")

# ===================== 配置（严格匹配新分析代码） =====================
GSE_ID = "GSE234863"
CONTROL_GROUP = "Naive"
CASE_GROUP = "Ti"

# 绘图阈值
P_CUTOFF = 0.05
FC_CUTOFF = 1.0
TOP_N_HEATMAP = 50

# 路径配置（100% 匹配新输出文件名）
OUTPUT_DIR = os.path.join(RESULT_DIR, GSE_ID)
PROCESSED_DIR = os.path.join(DATA_PROCESSED, GSE_ID)

# 差异分析结果
diff_file = os.path.join(OUTPUT_DIR, f"{GSE_ID}_DESeq2_full_results.csv")
# 归一化表达矩阵（双ID：ensembl_id + Symbol）
expr_file = os.path.join(PROCESSED_DIR, f"{GSE_ID}_归一化表达矩阵.csv")
# 样本分组信息
group_file = os.path.join(PROCESSED_DIR, f"{GSE_ID}_样本分组信息.csv")

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ==============================================================================
# 一、读取数据（适配双ID格式）
# ==============================================================================
print("="*50)
print("读取 DESeq2 双ID格式结果...")
print("="*50)

# 1. 差异结果（包含 ensembl_id + Symbol）
diff_result = pd.read_csv(diff_file)
# 2. 归一化表达矩阵（重置索引为 ensembl_id，方便匹配）
expr_norm = pd.read_csv(expr_file)
expr_norm = expr_norm.set_index("ensembl_id")
# 3. 样本分组
group_df = pd.read_csv(group_file)

print(f"✅ 差异基因数：{diff_result.shape[0]}")
print(f"✅ 表达矩阵形状：{expr_norm.shape}")
print(f"✅ 样本数：{group_df.shape[0]}")

# 统一样本顺序
sample_cols = group_df['sample_id'].tolist()
expr_norm = expr_norm[sample_cols]

# ==============================================================================
# 二、火山图（英文标题 + Symbol 标注逻辑）
# ==============================================================================
print("\n绘制火山图...")

def classify_gene(row):
    if row['padj'] < P_CUTOFF and row['log2FoldChange'] > FC_CUTOFF:
        return 'Up-regulated'
    elif row['padj'] < P_CUTOFF and row['log2FoldChange'] < -FC_CUTOFF:
        return 'Down-regulated'
    else:
        return 'Not significant'

diff_result['gene_type'] = diff_result.apply(classify_gene, axis=1)

fig, ax = plt.subplots(figsize=(10, 8))
colors = {'Not significant': 'lightgray', 'Up-regulated': '#e64343', 'Down-regulated': '#436eee'}

for gene_type, color in colors.items():
    mask = diff_result['gene_type'] == gene_type
    ax.scatter(
        diff_result.loc[mask, 'log2FoldChange'],
        -np.log10(diff_result.loc[mask, 'padj']),
        c=color, s=15, alpha=0.7, label=gene_type
    )

# 阈值线
ax.axvline(x=FC_CUTOFF, color='black', linestyle='--', lw=1)
ax.axvline(x=-FC_CUTOFF, color='black', linestyle='--', lw=1)
ax.axhline(y=-np.log10(P_CUTOFF), color='black', linestyle='--', lw=1)

ax.set_xlabel('log₂(Fold Change)', fontweight='bold', fontsize=12)
ax.set_ylabel('-log₁₀(Adjusted P-value)', fontweight='bold', fontsize=12)
ax.set_title(f'{GSE_ID} Volcano Plot\n{CASE_GROUP} vs {CONTROL_GROUP}', fontweight='bold', fontsize=14)
ax.legend(loc='upper right')
ax.set_xlim(-6, 6)

volcano_path = os.path.join(OUTPUT_DIR, f"{GSE_ID}_volcano_plot.png")
plt.tight_layout()
plt.savefig(volcano_path, dpi=300, bbox_inches='tight')
plt.close()

up_count = (diff_result['gene_type'] == 'Up-regulated').sum()
down_count = (diff_result['gene_type'] == 'Down-regulated').sum()
print(f"✅ 火山图完成 | 上调:{up_count} 下调:{down_count}")

# ==============================================================================
# 三、热图（Top50 差异基因 + Symbol 展示）
# ==============================================================================
print("\n绘制基因表达热图...")

# 1. 筛选Top差异基因（按显著性排序，基于 ensembl_id）
diff_sorted = diff_result.sort_values('padj').drop_duplicates(subset='Symbol', keep='first')
top_genes = diff_sorted.head(TOP_N_HEATMAP)

# 2. 匹配表达矩阵（Ensembl ID 匹配 → Symbol 展示）
top_ens_ids = top_genes['ensembl_id'].tolist()
top_symbols = top_genes['Symbol'].tolist()

# 提取子矩阵
expr_sub = expr_norm.loc[expr_norm.index.isin(top_ens_ids)].copy()
expr_sub = expr_sub.reindex(top_ens_ids)
# 行名替换为 Symbol（图表展示用）
expr_sub.index = top_symbols

if len(expr_sub) == 0:
    print("⚠️ 无有效差异基因，跳过热图")
else:
    # Z-score 标准化
    expr_z = expr_sub.sub(expr_sub.mean(axis=1), axis=0)
    expr_z = expr_z.div(expr_sub.std(axis=1), axis=0).fillna(0)

    # 绘图
    cmap = LinearSegmentedColormap.from_list('custom_cmap', ['#3b76af', 'white', '#e37a57'])
    fig_height = max(8, len(expr_sub) * 0.3 + 2)
    fig, ax = plt.subplots(figsize=(12, fig_height))
    
    sns.heatmap(
        expr_z,
        cmap=cmap, center=0, linewidths=0.5,
        xticklabels=[col.replace('_count', '') for col in expr_z.columns],
        yticklabels=expr_z.index,  # 用 Symbol 标注基因
        cbar_kws={'label': 'Z-score'}
    )

    ax.set_title(f'{GSE_ID} Top {len(expr_sub)} DEGs\n{CASE_GROUP} vs {CONTROL_GROUP}', fontweight='bold', fontsize=16)
    plt.xticks(rotation=45, ha='right')
    
    heatmap_path = os.path.join(OUTPUT_DIR, f"{GSE_ID}_heatmap.png")
    plt.tight_layout()
    plt.savefig(heatmap_path, dpi=300, bbox_inches='tight')
    plt.close()
    print("✅ 热图绘制完成！")

print("\n🎉 所有图表绘制完成！")
print(f"保存路径：{OUTPUT_DIR}")



### GSE85051



In [ ]:
import pandas as pd
import numpy as np
import os
from pathlib import Path
import gzip
import re

# 导入 config 路径配置
from config import DATA_PROCESSED, MOUSE_HA_VS_OS_GSE85051, MOUSE_HA_VS_OS_GSE85051_RAW

# ===================== 统一预处理类：适配 GSE85051 系列矩阵文件（基因水平输出） =====================
class GSE85051_UnifiedPreprocessor:
    def __init__(self, raw_data_dir, gpl_dir):
        """
        raw_data_dir: 包含 GSE85051_series_matrix.txt.gz 的目录
        gpl_dir:      包含 GPL 平台文件（如 GPL17117-26578.txt）的目录
        """
        self.raw_dir = Path(raw_data_dir)
        self.gpl_dir = Path(gpl_dir)
        # ================== 完整的样本分组映射（60个样本）==================
        self.sample_mapping = {
            "GSM2257143": {"group": "Contralateral", "tissue": "LymphNode", "sample_name": "Contralateral_LymphNode_1"},
            "GSM2257144": {"group": "Contralateral", "tissue": "LymphNode", "sample_name": "Contralateral_LymphNode_2"},
            "GSM2257145": {"group": "Contralateral", "tissue": "LymphNode", "sample_name": "Contralateral_LymphNode_3"},
            "GSM2257146": {"group": "Contralateral", "tissue": "LymphNode", "sample_name": "Contralateral_LymphNode_4"},
            "GSM2257147": {"group": "Contralateral", "tissue": "LymphNode", "sample_name": "Contralateral_LymphNode_5"},
            "GSM2257148": {"group": "Contralateral", "tissue": "LymphNode", "sample_name": "Contralateral_LymphNode_6"},
            "GSM2257149": {"group": "Contralateral", "tissue": "LymphNode", "sample_name": "Contralateral_LymphNode_7"},
            "GSM2257150": {"group": "Contralateral", "tissue": "LymphNode", "sample_name": "Contralateral_LymphNode_8"},
            "GSM2257151": {"group": "Contralateral", "tissue": "LymphNode", "sample_name": "Contralateral_LymphNode_9"},
            "GSM2257152": {"group": "Contralateral", "tissue": "LymphNode", "sample_name": "Contralateral_LymphNode_10"},
            "GSM2257153": {"group": "Contralateral", "tissue": "LymphNode", "sample_name": "Contralateral_LymphNode_11"},
            "GSM2257154": {"group": "Injured", "tissue": "LymphNode", "sample_name": "Injured_LymphNode_1"},
            "GSM2257155": {"group": "Injured", "tissue": "LymphNode", "sample_name": "Injured_LymphNode_2"},
            "GSM2257156": {"group": "Injured", "tissue": "LymphNode", "sample_name": "Injured_LymphNode_3"},
            "GSM2257157": {"group": "Injured", "tissue": "LymphNode", "sample_name": "Injured_LymphNode_4"},
            "GSM2257158": {"group": "Injured", "tissue": "LymphNode", "sample_name": "Injured_LymphNode_5"},
            "GSM2257159": {"group": "Injured", "tissue": "LymphNode", "sample_name": "Injured_LymphNode_6"},
            "GSM2257160": {"group": "Injured", "tissue": "LymphNode", "sample_name": "Injured_LymphNode_7"},
            "GSM2257161": {"group": "Injured", "tissue": "LymphNode", "sample_name": "Injured_LymphNode_8"},
            "GSM2257162": {"group": "Injured", "tissue": "LymphNode", "sample_name": "Injured_LymphNode_9"},
            "GSM2257163": {"group": "Injured", "tissue": "LymphNode", "sample_name": "Injured_LymphNode_10"},
            "GSM2257164": {"group": "Injured", "tissue": "LymphNode", "sample_name": "Injured_LymphNode_11"},
            "GSM2257165": {"group": "Contralateral", "tissue": "SynovialTissue", "sample_name": "Contralateral_Synovial_1"},
            "GSM2257166": {"group": "Contralateral", "tissue": "SynovialTissue", "sample_name": "Contralateral_Synovial_2"},
            "GSM2257167": {"group": "Contralateral", "tissue": "SynovialTissue", "sample_name": "Contralateral_Synovial_3"},
            "GSM2257168": {"group": "Contralateral", "tissue": "SynovialTissue", "sample_name": "Contralateral_Synovial_4"},
            "GSM2257169": {"group": "Contralateral", "tissue": "SynovialTissue", "sample_name": "Contralateral_Synovial_5"},
            "GSM2257170": {"group": "Contralateral", "tissue": "SynovialTissue", "sample_name": "Contralateral_Synovial_6"},
            "GSM2257171": {"group": "Contralateral", "tissue": "SynovialTissue", "sample_name": "Contralateral_Synovial_7"},
            "GSM2257172": {"group": "Contralateral", "tissue": "SynovialTissue", "sample_name": "Contralateral_Synovial_8"},
            "GSM2257173": {"group": "Contralateral", "tissue": "SynovialTissue", "sample_name": "Contralateral_Synovial_9"},
            "GSM2257174": {"group": "Contralateral", "tissue": "SynovialTissue", "sample_name": "Contralateral_Synovial_10"},
            "GSM2257175": {"group": "Contralateral", "tissue": "SynovialTissue", "sample_name": "Contralateral_Synovial_11"},
            "GSM2257176": {"group": "Injured", "tissue": "SynovialTissue", "sample_name": "Injured_Synovial_1"},
            "GSM2257177": {"group": "Injured", "tissue": "SynovialTissue", "sample_name": "Injured_Synovial_2"},
            "GSM2257178": {"group": "Injured", "tissue": "SynovialTissue", "sample_name": "Injured_Synovial_3"},
            "GSM2257179": {"group": "Injured", "tissue": "SynovialTissue", "sample_name": "Injured_Synovial_4"},
            "GSM2257180": {"group": "Injured", "tissue": "SynovialTissue", "sample_name": "Injured_Synovial_5"},
            "GSM2257181": {"group": "Injured", "tissue": "SynovialTissue", "sample_name": "Injured_Synovial_6"},
            "GSM2257182": {"group": "Injured", "tissue": "SynovialTissue", "sample_name": "Injured_Synovial_7"},
            "GSM2257183": {"group": "Injured", "tissue": "SynovialTissue", "sample_name": "Injured_Synovial_8"},
            "GSM2257184": {"group": "Injured", "tissue": "SynovialTissue", "sample_name": "Injured_Synovial_9"},
            "GSM2257185": {"group": "Injured", "tissue": "SynovialTissue", "sample_name": "Injured_Synovial_10"},
            "GSM2257186": {"group": "Injured", "tissue": "SynovialTissue", "sample_name": "Injured_Synovial_11"},
            "GSM2257187": {"group": "Control", "tissue": "LymphNode", "sample_name": "Control_LymphNode_1"},
            "GSM2257188": {"group": "Control", "tissue": "LymphNode", "sample_name": "Control_LymphNode_2"},
            "GSM2257189": {"group": "Control", "tissue": "LymphNode", "sample_name": "Control_LymphNode_3"},
            "GSM2257190": {"group": "Control", "tissue": "LymphNode", "sample_name": "Control_LymphNode_4"},
            "GSM2257191": {"group": "Control", "tissue": "LymphNode", "sample_name": "Control_LymphNode_5"},
            "GSM2257192": {"group": "Control", "tissue": "LymphNode", "sample_name": "Control_LymphNode_6"},
            "GSM2257193": {"group": "Control", "tissue": "LymphNode", "sample_name": "Control_LymphNode_7"},
            "GSM2257194": {"group": "Control", "tissue": "LymphNode", "sample_name": "Control_LymphNode_8"},
            "GSM2257195": {"group": "Control", "tissue": "LymphNode", "sample_name": "Control_LymphNode_9"},
            "GSM2257196": {"group": "Control", "tissue": "SynovialTissue", "sample_name": "Control_Synovial_1"},
            "GSM2257197": {"group": "Control", "tissue": "SynovialTissue", "sample_name": "Control_Synovial_2"},
            "GSM2257198": {"group": "Control", "tissue": "SynovialTissue", "sample_name": "Control_Synovial_3"},
            "GSM2257199": {"group": "Control", "tissue": "SynovialTissue", "sample_name": "Control_Synovial_4"},
            "GSM2257200": {"group": "Control", "tissue": "SynovialTissue", "sample_name": "Control_Synovial_5"},
            "GSM2257201": {"group": "Control", "tissue": "SynovialTissue", "sample_name": "Control_Synovial_6"},
            "GSM2257202": {"group": "Control", "tissue": "SynovialTissue", "sample_name": "Control_Synovial_7"}
        }

    def read_series_matrix(self):
        """读取 GEO 系列矩阵文件，返回探针水平表达矩阵（已数值化，列名为 GSM ID）"""
        matrix_path = self.raw_dir / "GSE85051_series_matrix.txt.gz"
        if not matrix_path.exists():
            raise FileNotFoundError(f"未找到核心文件：{matrix_path}")

        df = pd.read_csv(matrix_path, sep="\t", compression="gzip", comment="!", header=0)
        df = df.set_index(df.columns[0])  # 第一列为探针 ID
        df = df.apply(pd.to_numeric, errors="coerce")
        df = df.dropna(how='all')

        new_cols = []
        for col in df.columns:
            if isinstance(col, str) and "." in col:
                new_cols.append(col.split(".")[0])
            else:
                new_cols.append(str(col))
        df.columns = new_cols
        return df

    def load_gpl_mapping(self):
        """
        加载平台文件，返回探针映射字典
        key: 探针ID
        value: (Ensembl基因ID, Gene Symbol)
        从 mrna_assignment 提取 ENSRNOG 基因ID，从 gene_assignment 提取 Symbol
        """
        # 查找GPL文件
        gpl_candidates = [
            self.gpl_dir / "GPL17117-26578.txt",
            self.gpl_dir / "GPL16686.txt",
            self.gpl_dir / "GPL16686-26578.txt"
        ]
        gpl_path = None
        for cand in gpl_candidates:
            if cand.exists():
                gpl_path = cand
                break
        if gpl_path is None:
            raise FileNotFoundError(f"在目录 {self.gpl_dir} 中未找到 GPL 平台文件")

        print(f"使用平台文件：{gpl_path}")
        gpl = pd.read_csv(gpl_path, sep="\t", comment="#", header=0, low_memory=False, dtype=str)
        
        # 定位核心列
        id_col = gpl.columns[0]  # 第一列为探针ID
        gene_col = "gene_assignment"
        mrna_col = "mrna_assignment"
        
        # 正则匹配 Ensembl 基因ID (ENSRNOG开头)
        ensembl_pattern = re.compile(r'gene:(ENSRNOG\d+)')

        mapping = {}
        for idx, row in gpl.iterrows():
            probe = str(row[id_col]).strip()
            gene_field = row[gene_col]
            mrna_field = row[mrna_col]

            # 1. 提取 Gene Symbol
            symbol = "NA"
            if pd.notna(gene_field) and gene_field != '---':
                first_assignment = str(gene_field).split('///')[0].strip()
                parts = [p.strip() for p in first_assignment.split('//')]
                symbol = parts[1] if len(parts) >= 2 else parts[0]
                symbol = symbol.split()[0] if symbol not in ('---', '', 'NA', 'nan') else "NA"

            # 2. 提取 Ensembl 基因ID
            ensembl_id = "NA"
            if pd.notna(mrna_field):
                match = ensembl_pattern.search(str(mrna_field))
                if match:
                    ensembl_id = match.group(1)

            # 仅保留有有效注释的探针
            if ensembl_id != "NA" or symbol != "NA":
                mapping[probe] = (ensembl_id, symbol)

        print(f"✅ 成功加载探针注释：{len(mapping)} 个探针包含 Ensembl ID / Gene Symbol")
        return mapping

    def run(self):
        print("🔍 读取 GSE85051 官方系列矩阵文件...")
        exp_matrix = self.read_series_matrix()
        print(f"✅ 原始矩阵：{exp_matrix.shape[0]} 探针 × {exp_matrix.shape[1]} 样本")

        # 筛选有效样本
        valid_gsm = [gsm for gsm in exp_matrix.columns if gsm in self.sample_mapping]
        if len(valid_gsm) == 0:
            raise ValueError("未找到任何匹配的样本，请检查 sample_mapping 是否包含正确的 GSM ID")
        exp_filtered_samples = exp_matrix[valid_gsm]
        print(f"✅ 保留 {len(valid_gsm)} 个样本")

        # 数据校验
        if exp_filtered_samples.empty or exp_filtered_samples.isnull().all().all():
            raise ValueError("表达矩阵无有效数值，请检查数据文件或样本映射")

        # log2转换判断
        data_vals = exp_filtered_samples.dropna().values.flatten()
        max_val = np.max(data_vals)
        min_val = np.min(data_vals)
        print(f"📊 表达值范围：min={min_val:.2f}, max={max_val:.2f}")
        need_log2 = max_val > 50
        print("⚠️ 未 log2 转换，将执行 log2(expression+1)") if need_log2 else print("✅ 已 log2 转换，不再重复")

        # 低表达过滤
        min_exp_for_filter = 1.0
        min_samples = max(3, int(exp_filtered_samples.shape[1] * 0.1))
        keep_rows = (exp_filtered_samples > min_exp_for_filter).sum(axis=1) >= min_samples
        exp_filtered = exp_filtered_samples.loc[keep_rows]
        print(f"✅ 低表达过滤后保留 {exp_filtered.shape[0]} 个探针")

        # 标准化
        exp_final_probe = np.log2(exp_filtered + 1) if need_log2 else exp_filtered
        exp_final_probe.index = exp_final_probe.index.astype(str)

        # 探针 → Ensembl ID + Symbol 映射
        print("\n🔍 加载平台文件，进行探针注释映射...")
        probe_to_info = self.load_gpl_mapping()
        common_probes = [p for p in exp_final_probe.index if p in probe_to_info]
        
        print(f"🔍 表达矩阵探针总数：{len(exp_final_probe.index)}")
        print(f"🔍 成功匹配注释探针数：{len(common_probes)}")

        if len(common_probes) == 0:
            raise ValueError("⚠️ 无匹配探针，请检查探针ID格式")

        # 构建带注释的表达矩阵
        exp_mapped = exp_final_probe.loc[common_probes].copy()
        # 添加两列：Ensembl_ID 和 Gene_Symbol
        ensembl_list = []
        symbol_list = []
        for probe in exp_mapped.index:
            ensembl, symbol = probe_to_info[probe]
            ensembl_list.append(ensembl)
            symbol_list.append(symbol)
        
        exp_mapped["Ensembl_ID"] = ensembl_list
        exp_mapped["Gene_Symbol"] = symbol_list

        # 按 Ensembl_ID 聚合（唯一标识），表达值取均值，保留对应Symbol
        exp_gene = exp_mapped.groupby("Ensembl_ID").agg({
            **{col: 'mean' for col in exp_final_probe.columns},
            "Gene_Symbol": "first"  # 一个Ensembl ID对应唯一Symbol
        }).reset_index()

        # 调整列顺序：Ensembl_ID → Gene_Symbol → 表达值
        cols = ["Ensembl_ID", "Gene_Symbol"] + [col for col in exp_final_probe.columns]
        exp_gene = exp_gene[cols]
        print(f"✅ 基因水平表达矩阵：{exp_gene.shape[0]} 基因 × {exp_gene.shape[1]-2} 样本")

        # 重命名样本列
        rename_dict = {g: self.sample_mapping[g]['sample_name'] for g in valid_gsm}
        exp_gene = exp_gene.rename(columns=rename_dict)

        # 样本信息表
        sample_info = pd.DataFrame([
            {"sample_name": self.sample_mapping[g]['sample_name'],
             "group": self.sample_mapping[g]['group'],
             "tissue": self.sample_mapping[g]['tissue'],
             "gsm_id": g} for g in valid_gsm
        ])

        # 保存文件
        processed_dir = Path(DATA_PROCESSED) / "GSE85051"
        processed_dir.mkdir(parents=True, exist_ok=True)
        output_dir = Path(DATA_PROCESSED).parent / "output" / "GSE85051"
        output_dir.mkdir(parents=True, exist_ok=True)

        # 核心输出：包含Ensembl ID + Symbol的表达矩阵
        exp_gene.to_csv(processed_dir / "gene_expression_matrix.csv", index=False)
        sample_info.to_csv(processed_dir / "sample_info.csv", index=False)
        exp_final_probe.to_csv(processed_dir / "probe_expression_matrix.csv")

        print(f"💾 数据已保存至 {processed_dir}")
        return exp_gene, sample_info


# ===================== 一键执行 =====================
if __name__ == "__main__":
    processor = GSE85051_UnifiedPreprocessor(
        raw_data_dir=MOUSE_HA_VS_OS_GSE85051,
        gpl_dir=MOUSE_HA_VS_OS_GSE85051_RAW
    )
    GSE85051_gene_expr, sample_info = processor.run()

    print("\n" + "="*60)
    print("📊 GSE85051 预处理完成！")
    print(f"📦 最终数据维度：{GSE85051_gene_expr.shape[0]} 基因 × {GSE85051_gene_expr.shape[1]-2} 样本")
    print("\n🔍 表达矩阵预览（前5行）：")
    print(GSE85051_gene_expr.head().round(2))
    print("\n📄 样本分组统计：")
    print(sample_info.groupby(["group", "tissue"]).size())

In [ ]:
# ==============================================
# GSE85051 差异分析（limma 双因素交互模型）
# 读取：data/processed/GSE85051/ 下的基因表达矩阵和样本信息
# 输出：output/GSE85051_diff/ 下的差异结果和可视化
# 适配：Ensembl ID 分析 + Gene Symbol 绘图
# ==============================================
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap

# rpy2 相关
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri, StrVector
from rpy2.robjects.conversion import localconverter
from rpy2.robjects.packages import importr, isinstalled

# 导入 config
from config import DATA_PROCESSED, RESULT_DIR

# 安装/加载 limma
utils = importr('utils')
if not isinstalled('limma'):
    print("正在安装 limma ...")
    utils.install_packages('BiocManager', quiet=True)
    ro.r('BiocManager::install("limma", update=FALSE, ask=FALSE)')
limma = importr('limma')
stats = importr('stats')

# ===================== 路径配置 =====================
# 从 processed 读取表达矩阵和样本信息
PROCESSED_DIR = Path(DATA_PROCESSED) / "GSE85051"
EXPR_FILE = PROCESSED_DIR / "gene_expression_matrix.csv"
SAMPLE_FILE = PROCESSED_DIR / "sample_info.csv"

# 差异分析结果输出到 output
OUTPUT_DIR = Path(RESULT_DIR) / "GSE85051_diff"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ===================== 1. 读取数据（适配新格式：Ensembl_ID + Gene_Symbol） =====================
print("="*50)
print("读取表达矩阵和样本信息...")
print("="*50)

# 读取带注释的表达矩阵（Ensembl_ID, Gene_Symbol, 样本表达值）
expr_annot = pd.read_csv(EXPR_FILE)
# 1. 构建表达矩阵：行=Ensembl_ID（用于差异分析），列=样本
expr_matrix = expr_annot.set_index('Ensembl_ID').iloc[:, 1:].copy()
# 2. 构建基因映射表：Ensembl_ID ↔ Gene_Symbol（用于绘图替换）
gene_mapping = expr_annot[['Ensembl_ID', 'Gene_Symbol']].set_index('Ensembl_ID')

# 读取样本信息
sample_info = pd.read_csv(SAMPLE_FILE)
print(f"✅ 表达矩阵：{expr_matrix.shape[0]} 基因 × {expr_matrix.shape[1]} 样本")
print(f"✅ 样本信息：{sample_info.shape[0]} 个样本")

# 确保样本顺序一致（关键：表达矩阵列 = 样本信息行）
sample_info = sample_info.set_index('sample_name').loc[expr_matrix.columns].reset_index()
print("\n样本分组统计：")
print(sample_info.groupby(['group', 'tissue']).size())

# ===================== 2. 准备数据并转换到 R =====================
group_vector = StrVector(sample_info['group'].tolist())
tissue_vector = StrVector(sample_info['tissue'].tolist())

# 传入 R 的矩阵：行=Ensembl_ID，列=样本
with localconverter(ro.default_converter + pandas2ri.converter):
    expr_r = ro.conversion.py2rpy(expr_matrix)

ro.globalenv['expr_mat'] = ro.r('as.matrix')(expr_r)
ro.globalenv['group'] = group_vector
ro.globalenv['tissue'] = tissue_vector

# ===================== 3. 运行 limma 线性模型 =====================
ro.r('''
    library(limma)
    group <- factor(group)
    tissue <- factor(tissue)
    # 打印因子水平，确认参考组
    cat("group 因子水平：\n"); print(levels(group))
    cat("tissue 因子水平：\n"); print(levels(tissue))
    
    design <- model.matrix(~ group * tissue)
    colnames(design) <- make.names(colnames(design))
    # 打印设计矩阵列名（核心调试）
    cat("\n设计矩阵列名：\n"); print(colnames(design))
    
    fit <- lmFit(expr_mat, design)
    fit <- eBayes(fit)
''')

# ===================== 4. 构建对比矩阵 =====================
ro.r('''
    cont.matrix <- makeContrasts(
        # 淋巴结 (LN) 对比：Contralateral为参考组，直接用0替代
        Injured_vs_Control_LN = groupInjured - groupControl,
        Contralateral_vs_Control_LN = 0 - groupControl,
        
        # 滑膜组织 (ST) 对比
        Injured_vs_Control_ST = (groupInjured + groupInjured.tissueSynovialTissue) - (groupControl + groupControl.tissueSynovialTissue),
        Contralateral_vs_Control_ST = 0 - (groupControl + groupControl.tissueSynovialTissue),
        
        # 交互效应
        Interaction_Injured_STvsLN = groupInjured.tissueSynovialTissue - groupControl.tissueSynovialTissue,
        
        levels = design
    )
    fit2 <- contrasts.fit(fit, cont.matrix)
    fit2 <- eBayes(fit2)
''')

# ===================== 5. 提取结果函数（合并Gene Symbol） =====================
def get_results(contrast_name):
    with localconverter(ro.default_converter + pandas2ri.converter):
        ro.globalenv['contrast'] = contrast_name
        ro.r(f'''
            res <- topTable(fit2, coef="{contrast_name}", number=Inf, sort.by="none")
            res_df <- as.data.frame(res)
        ''')
        res_pd = ro.globalenv['res_df']
    # 重置索引：Gene列 = Ensembl_ID
    res_pd = res_pd.reset_index().rename(columns={'index': 'Ensembl_ID'})
    # 合并 Gene Symbol
    res_pd = res_pd.merge(gene_mapping, on='Ensembl_ID', how='left')
    # 调整列顺序：ID + Symbol + 差异结果
    cols = ['Ensembl_ID', 'Gene_Symbol'] + [col for col in res_pd.columns if col not in ['Ensembl_ID', 'Gene_Symbol']]
    res_pd = res_pd[cols]
    return res_pd

contrasts_list = [
    "Injured_vs_Control_LN",
    "Contralateral_vs_Control_LN",
    "Injured_vs_Control_ST",
    "Contralateral_vs_Control_ST",
    "Interaction_Injured_STvsLN"
]

for cname in contrasts_list:
    print(f"正在提取对比：{cname}")
    df_res = get_results(cname)
    df_res.to_csv(OUTPUT_DIR / f"{cname}_results.csv", index=False)
    # 筛选显著差异基因
    sig = (df_res['adj.P.Val'] < 0.05) & (abs(df_res['logFC']) > 1)
    up = df_res[sig & (df_res['logFC'] > 0)]
    down = df_res[sig & (df_res['logFC'] < -1)]
    print(f"  显著上调: {len(up)}, 显著下调: {len(down)}")
    up.to_csv(OUTPUT_DIR / f"{cname}_upregulated.csv", index=False)
    down.to_csv(OUTPUT_DIR / f"{cname}_downregulated.csv", index=False)

# ===================== 7. 热图（极简兼容版，零报错+彻底无重叠）=====================
print("\n" + "="*50)
print("绘制热图（终极兼容修复版）...")
print("="*50)
# 获取Top50差异基因（Ensembl_ID）
res_ln = pd.read_csv(OUTPUT_DIR / "Injured_vs_Control_LN_results.csv")
top_genes = res_ln.sort_values('adj.P.Val').head(50)['Ensembl_ID'].tolist()
# 获取对应的 Gene Symbol
top_symbols = res_ln.set_index('Ensembl_ID').loc[top_genes, 'Gene_Symbol'].fillna(pd.Series(top_genes, index=top_genes)).tolist()

# 提取热图数据
heat_data = expr_matrix.loc[top_genes, :].copy()
# 过滤标准差为0的基因
heat_data = heat_data[heat_data.std(axis=1) > 1e-5]
top_symbols = [top_symbols[i] for i in range(len(heat_data.index))]

# 替换行索引为 Gene Symbol
heat_data.index = top_symbols

# Z-score标准化
heat_z = heat_data.sub(heat_data.mean(axis=1), axis=0)
heat_z = heat_z.div(heat_data.std(axis=1), axis=0)
heat_z = heat_z.replace([np.inf, -np.inf], np.nan).fillna(0)
heat_z = heat_z.clip(-3, 3)

# 绘图基础设置
cmap = LinearSegmentedColormap.from_list('custom', ['#3b76af', 'white', '#e37a57'])
fig_height = max(14, len(top_symbols)*0.35)
fig_width = 18

# 核心：极简参数，100%兼容所有旧版本库
g = sns.clustermap(
    heat_z,
    cmap=cmap,
    center=0,
    row_cluster=False,
    col_cluster=True,
    dendrogram_ratio=(0.05, 0.1),
    # 色条放在右侧，彻底避免重叠
    cbar_pos=(0.88, 0.8, 0.02, 0.12),
    # 仅保留基础cbar参数，无任何报错
    cbar_kws={'label': 'Z-score'},
    yticklabels=True,
    xticklabels=True,
    linewidths=0.5,
    figsize=(fig_width, fig_height)
)

# 手动设置标签（通用写法，无报错）
g.ax_heatmap.set_xticklabels(g.ax_heatmap.get_xticklabels(), rotation=90, fontsize=7)
g.ax_heatmap.set_yticklabels(g.ax_heatmap.get_yticklabels(), fontsize=8)

# 标题
g.fig.suptitle('Top 50 Differentially Expressed Genes', y=1.05, fontsize=14, fontweight='bold')

# 超大边距，彻底杜绝重叠
g.fig.subplots_adjust(left=0.2, right=0.8, top=0.93, bottom=0.28)

# 保存
g.savefig(OUTPUT_DIR / "Heatmap_Final.png", dpi=300, bbox_inches='tight', pad_inches=0.5)
plt.close()

print("✅ 热图保存成功！")
print("\n🎉 全部分析完成！")

In [ ]:
import pandas as pd
import numpy as np
import os
from pathlib import Path
import gzip

# 导入 config 路径配置
from config import DATA_PROCESSED, MOUSE_HA_VS_OS_GSE85051, MOUSE_HA_VS_OS_GSE85051_RAW

# ===================== 统一预处理类：适配 GSE85051 系列矩阵文件（基因水平输出） =====================
class GSE85051_UnifiedPreprocessor:
    def __init__(self, raw_data_dir, gpl_dir):
        """
        raw_data_dir: 包含 GSE85051_series_matrix.txt.gz 的目录
        gpl_dir:      包含 GPL 平台文件（如 GPL17117-26578.txt）的目录
        """
        self.raw_dir = Path(raw_data_dir)
        self.gpl_dir = Path(gpl_dir)
        # ================== 完整的样本分组映射（60个样本）==================
        self.sample_mapping = {
            "GSM2257143": {"group": "Contralateral", "tissue": "LymphNode", "sample_name": "Contralateral_LymphNode_1"},
            "GSM2257144": {"group": "Contralateral", "tissue": "LymphNode", "sample_name": "Contralateral_LymphNode_2"},
            "GSM2257145": {"group": "Contralateral", "tissue": "LymphNode", "sample_name": "Contralateral_LymphNode_3"},
            "GSM2257146": {"group": "Contralateral", "tissue": "LymphNode", "sample_name": "Contralateral_LymphNode_4"},
            "GSM2257147": {"group": "Contralateral", "tissue": "LymphNode", "sample_name": "Contralateral_LymphNode_5"},
            "GSM2257148": {"group": "Contralateral", "tissue": "LymphNode", "sample_name": "Contralateral_LymphNode_6"},
            "GSM2257149": {"group": "Contralateral", "tissue": "LymphNode", "sample_name": "Contralateral_LymphNode_7"},
            "GSM2257150": {"group": "Contralateral", "tissue": "LymphNode", "sample_name": "Contralateral_LymphNode_8"},
            "GSM2257151": {"group": "Contralateral", "tissue": "LymphNode", "sample_name": "Contralateral_LymphNode_9"},
            "GSM2257152": {"group": "Contralateral", "tissue": "LymphNode", "sample_name": "Contralateral_LymphNode_10"},
            "GSM2257153": {"group": "Contralateral", "tissue": "LymphNode", "sample_name": "Contralateral_LymphNode_11"},
            "GSM2257154": {"group": "Injured", "tissue": "LymphNode", "sample_name": "Injured_LymphNode_1"},
            "GSM2257155": {"group": "Injured", "tissue": "LymphNode", "sample_name": "Injured_LymphNode_2"},
            "GSM2257156": {"group": "Injured", "tissue": "LymphNode", "sample_name": "Injured_LymphNode_3"},
            "GSM2257157": {"group": "Injured", "tissue": "LymphNode", "sample_name": "Injured_LymphNode_4"},
            "GSM2257158": {"group": "Injured", "tissue": "LymphNode", "sample_name": "Injured_LymphNode_5"},
            "GSM2257159": {"group": "Injured", "tissue": "LymphNode", "sample_name": "Injured_LymphNode_6"},
            "GSM2257160": {"group": "Injured", "tissue": "LymphNode", "sample_name": "Injured_LymphNode_7"},
            "GSM2257161": {"group": "Injured", "tissue": "LymphNode", "sample_name": "Injured_LymphNode_8"},
            "GSM2257162": {"group": "Injured", "tissue": "LymphNode", "sample_name": "Injured_LymphNode_9"},
            "GSM2257163": {"group": "Injured", "tissue": "LymphNode", "sample_name": "Injured_LymphNode_10"},
            "GSM2257164": {"group": "Injured", "tissue": "LymphNode", "sample_name": "Injured_LymphNode_11"},
            "GSM2257165": {"group": "Contralateral", "tissue": "SynovialTissue", "sample_name": "Contralateral_Synovial_1"},
            "GSM2257166": {"group": "Contralateral", "tissue": "SynovialTissue", "sample_name": "Contralateral_Synovial_2"},
            "GSM2257167": {"group": "Contralateral", "tissue": "SynovialTissue", "sample_name": "Contralateral_Synovial_3"},
            "GSM2257168": {"group": "Contralateral", "tissue": "SynovialTissue", "sample_name": "Contralateral_Synovial_4"},
            "GSM2257169": {"group": "Contralateral", "tissue": "SynovialTissue", "sample_name": "Contralateral_Synovial_5"},
            "GSM2257170": {"group": "Contralateral", "tissue": "SynovialTissue", "sample_name": "Contralateral_Synovial_6"},
            "GSM2257171": {"group": "Contralateral", "tissue": "SynovialTissue", "sample_name": "Contralateral_Synovial_7"},
            "GSM2257172": {"group": "Contralateral", "tissue": "SynovialTissue", "sample_name": "Contralateral_Synovial_8"},
            "GSM2257173": {"group": "Contralateral", "tissue": "SynovialTissue", "sample_name": "Contralateral_Synovial_9"},
            "GSM2257174": {"group": "Contralateral", "tissue": "SynovialTissue", "sample_name": "Contralateral_Synovial_10"},
            "GSM2257175": {"group": "Contralateral", "tissue": "SynovialTissue", "sample_name": "Contralateral_Synovial_11"},
            "GSM2257176": {"group": "Injured", "tissue": "SynovialTissue", "sample_name": "Injured_Synovial_1"},
            "GSM2257177": {"group": "Injured", "tissue": "SynovialTissue", "sample_name": "Injured_Synovial_2"},
            "GSM2257178": {"group": "Injured", "tissue": "SynovialTissue", "sample_name": "Injured_Synovial_3"},
            "GSM2257179": {"group": "Injured", "tissue": "SynovialTissue", "sample_name": "Injured_Synovial_4"},
            "GSM2257180": {"group": "Injured", "tissue": "SynovialTissue", "sample_name": "Injured_Synovial_5"},
            "GSM2257181": {"group": "Injured", "tissue": "SynovialTissue", "sample_name": "Injured_Synovial_6"},
            "GSM2257182": {"group": "Injured", "tissue": "SynovialTissue", "sample_name": "Injured_Synovial_7"},
            "GSM2257183": {"group": "Injured", "tissue": "SynovialTissue", "sample_name": "Injured_Synovial_8"},
            "GSM2257184": {"group": "Injured", "tissue": "SynovialTissue", "sample_name": "Injured_Synovial_9"},
            "GSM2257185": {"group": "Injured", "tissue": "SynovialTissue", "sample_name": "Injured_Synovial_10"},
            "GSM2257186": {"group": "Injured", "tissue": "SynovialTissue", "sample_name": "Injured_Synovial_11"},
            "GSM2257187": {"group": "Control", "tissue": "LymphNode", "sample_name": "Control_LymphNode_1"},
            "GSM2257188": {"group": "Control", "tissue": "LymphNode", "sample_name": "Control_LymphNode_2"},
            "GSM2257189": {"group": "Control", "tissue": "LymphNode", "sample_name": "Control_LymphNode_3"},
            "GSM2257190": {"group": "Control", "tissue": "LymphNode", "sample_name": "Control_LymphNode_4"},
            "GSM2257191": {"group": "Control", "tissue": "LymphNode", "sample_name": "Control_LymphNode_5"},
            "GSM2257192": {"group": "Control", "tissue": "LymphNode", "sample_name": "Control_LymphNode_6"},
            "GSM2257193": {"group": "Control", "tissue": "LymphNode", "sample_name": "Control_LymphNode_7"},
            "GSM2257194": {"group": "Control", "tissue": "LymphNode", "sample_name": "Control_LymphNode_8"},
            "GSM2257195": {"group": "Control", "tissue": "LymphNode", "sample_name": "Control_LymphNode_9"},
            "GSM2257196": {"group": "Control", "tissue": "SynovialTissue", "sample_name": "Control_Synovial_1"},
            "GSM2257197": {"group": "Control", "tissue": "SynovialTissue", "sample_name": "Control_Synovial_2"},
            "GSM2257198": {"group": "Control", "tissue": "SynovialTissue", "sample_name": "Control_Synovial_3"},
            "GSM2257199": {"group": "Control", "tissue": "SynovialTissue", "sample_name": "Control_Synovial_4"},
            "GSM2257200": {"group": "Control", "tissue": "SynovialTissue", "sample_name": "Control_Synovial_5"},
            "GSM2257201": {"group": "Control", "tissue": "SynovialTissue", "sample_name": "Control_Synovial_6"},
            "GSM2257202": {"group": "Control", "tissue": "SynovialTissue", "sample_name": "Control_Synovial_7"}
        }

    def read_series_matrix(self):
        """读取 GEO 系列矩阵文件，返回探针水平表达矩阵（已数值化，列名为 GSM ID）"""
        matrix_path = self.raw_dir / "GSE85051_series_matrix.txt.gz"
        if not matrix_path.exists():
            raise FileNotFoundError(f"未找到核心文件：{matrix_path}")

        df = pd.read_csv(matrix_path, sep="\t", compression="gzip", comment="!", header=0)
        df = df.set_index(df.columns[0])  # 第一列为探针 ID
        df = df.apply(pd.to_numeric, errors="coerce")
        df = df.dropna(how='all')

        new_cols = []
        for col in df.columns:
            if isinstance(col, str) and "." in col:
                new_cols.append(col.split(".")[0])
            else:
                new_cols.append(str(col))
        df.columns = new_cols
        return df

    def load_gpl_mapping(self):
        """加载平台文件，返回探针到基因符号的映射字典（探针ID -> 基因符号）"""
        # 在 GPL 目录中查找平台文件（支持多种可能的文件名）
        gpl_candidates = [
            self.gpl_dir / "GPL17117-26578.txt",
            self.gpl_dir / "GPL16686.txt",
            self.gpl_dir / "GPL16686-26578.txt"
        ]
        gpl_path = None
        for cand in gpl_candidates:
            if cand.exists():
                gpl_path = cand
                break
        if gpl_path is None:
            raise FileNotFoundError(f"在目录 {self.gpl_dir} 中未找到 GPL 平台文件")

        print(f"使用平台文件：{gpl_path}")
        gpl = pd.read_csv(gpl_path, sep="\t", comment="#", header=0, low_memory=False, dtype=str)
        # 寻找探针ID列
        id_col = None
        for col in gpl.columns:
            if col.upper() == 'ID':
                id_col = col
                break
        if id_col is None:
            id_col = gpl.columns[0]
        # 寻找 gene_assignment 列
        gene_col = None
        for col in gpl.columns:
            if 'gene_assignment' in col.lower():
                gene_col = col
                break
        if gene_col is None:
            raise ValueError("未找到 'gene_assignment' 列")

        mapping = {}
        for idx, row in gpl.iterrows():
            probe = str(row[id_col]).strip()
            gene_field = row[gene_col]
            if pd.isna(gene_field) or gene_field == '---':
                continue
            first_assignment = str(gene_field).split('///')[0].strip()
            parts = [p.strip() for p in first_assignment.split('//')]
            gene_symbol = parts[1] if len(parts) >= 2 else parts[0]
            if gene_symbol and gene_symbol not in ('---', '', 'NA', 'nan'):
                gene_symbol = gene_symbol.split()[0]
                mapping[probe] = gene_symbol
        print(f"✅ 成功加载探针到基因映射：{len(mapping)} 个探针对应非空基因符号")
        return mapping

    def run(self):
        print("🔍 读取 GSE85051 官方系列矩阵文件...")
        exp_matrix = self.read_series_matrix()
        print(f"✅ 原始矩阵：{exp_matrix.shape[0]} 探针 × {exp_matrix.shape[1]} 样本")

        # 筛选有效样本
        valid_gsm = [gsm for gsm in exp_matrix.columns if gsm in self.sample_mapping]
        if len(valid_gsm) == 0:
            raise ValueError("未找到任何匹配的样本，请检查 sample_mapping 是否包含正确的 GSM ID")
        exp_filtered_samples = exp_matrix[valid_gsm]
        print(f"✅ 保留 {len(valid_gsm)} 个样本")

        # 检查表达矩阵是否全为 NaN
        if exp_filtered_samples.empty or exp_filtered_samples.isnull().all().all():
            raise ValueError("表达矩阵无有效数值，请检查数据文件或样本映射")

        # 检查 log2 转换
        data_vals = exp_filtered_samples.values.flatten()
        data_vals = data_vals[~np.isnan(data_vals)]
        if data_vals.size == 0:
            raise ValueError("表达矩阵所有值为 NaN，无法继续")
        max_val = np.max(data_vals)
        min_val = np.min(data_vals)
        print(f"📊 表达值范围：min={min_val:.2f}, max={max_val:.2f}")
        if max_val > 50:
            need_log2 = True
            print("⚠️ 未 log2 转换，将执行 log2(expression+1)")
        else:
            need_log2 = False
            print("✅ 已 log2 转换，不再重复")

        # 低表达过滤
        min_exp_for_filter = 1.0
        min_samples = max(3, int(exp_filtered_samples.shape[1] * 0.1))
        keep_rows = (exp_filtered_samples > min_exp_for_filter).sum(axis=1) >= min_samples
        exp_filtered = exp_filtered_samples.loc[keep_rows]
        print(f"✅ 低表达过滤后保留 {exp_filtered.shape[0]} 个探针")

        # 标准化
        if need_log2:
            exp_final_probe = np.log2(exp_filtered + 1)
        else:
            exp_final_probe = exp_filtered

        # 关键修复：将探针索引转换为字符串
        exp_final_probe.index = exp_final_probe.index.astype(str)

        # 探针到基因映射
        print("\n🔍 加载平台文件，进行探针到基因的映射...")
        probe_to_gene = self.load_gpl_mapping()

        common_probes = [p for p in exp_final_probe.index if p in probe_to_gene]
        print(f"🔍 表达矩阵中的探针总数：{len(exp_final_probe.index)}")
        print(f"🔍 映射字典中的探针数：{len(probe_to_gene)}")
        print(f"🔍 成功匹配的探针数：{len(common_probes)}")
        if len(common_probes) == 0:
            print("⚠️ 警告：没有匹配到任何探针，请检查探针 ID 格式是否一致")
            print("示例探针（前5个）：", list(exp_final_probe.index[:5]))
            print("示例映射键（前5个）：", list(probe_to_gene.keys())[:5])
            exp_gene = pd.DataFrame(index=[], columns=exp_final_probe.columns)
        else:
            exp_mapped = exp_final_probe.loc[common_probes].copy()
            exp_mapped['GeneSymbol'] = [probe_to_gene[p] for p in exp_mapped.index]
            exp_gene = exp_mapped.groupby('GeneSymbol').mean()
            exp_gene = exp_gene.drop(columns=['GeneSymbol'], errors='ignore')
            print(f"✅ 基因水平表达矩阵：{exp_gene.shape[0]} 基因 × {exp_gene.shape[1]} 样本")

        # 重命名样本列
        rename_dict = {g: self.sample_mapping[g]['sample_name'] for g in valid_gsm}
        exp_gene = exp_gene.rename(columns=rename_dict)

        # 样本信息表
        sample_info = pd.DataFrame([
            {"sample_name": self.sample_mapping[g]['sample_name'],
             "group": self.sample_mapping[g]['group'],
             "tissue": self.sample_mapping[g]['tissue'],
             "gsm_id": g} for g in valid_gsm
        ])

        # ===================== 输出目录配置 =====================
        # 表达矩阵和样本信息 → data/processed/GSE85051/
        processed_dir = Path(DATA_PROCESSED) / "GSE85051"
        processed_dir.mkdir(parents=True, exist_ok=True)

        # 同时创建 output/GSE85051/ 目录（供后续差异分析使用）
        output_dir = Path(DATA_PROCESSED).parent / "output" / "GSE85051"
        output_dir.mkdir(parents=True, exist_ok=True)

        # 保存文件
        exp_gene.to_csv(processed_dir / "gene_expression_matrix.csv")
        sample_info.to_csv(processed_dir / "sample_info.csv", index=False)
        exp_final_probe.to_csv(processed_dir / "probe_expression_matrix.csv")

        print(f"💾 数据已保存至 {processed_dir}")
        return exp_gene, sample_info


# ===================== 一键执行 =====================
if __name__ == "__main__":
    # 使用 config 中定义的路径
    # MOUSE_HA_VS_OS_GSE85051 应包含 GSE85051_series_matrix.txt.gz
    # MOUSE_HA_VS_OS_GSE85051_RAW 应包含 GPL 平台文件
    processor = GSE85051_UnifiedPreprocessor(
        raw_data_dir=MOUSE_HA_VS_OS_GSE85051,
        gpl_dir=MOUSE_HA_VS_OS_GSE85051_RAW
    )
    GSE85051_gene_expr, sample_info = processor.run()

    print("\n" + "="*60)
    print("📊 GSE85051 预处理完成！")
    print(f"📦 最终数据维度：{GSE85051_gene_expr.shape[0]} 基因 × {GSE85051_gene_expr.shape[1]} 样本")
    print("\n🔍 表达矩阵预览（前5行×5列）：")
    if GSE85051_gene_expr.shape[0] > 0:
        print(GSE85051_gene_expr.iloc[:5, :5].round(2))
    else:
        print("空矩阵（无基因）")
    print("\n📄 样本分组统计：")
    print(sample_info.groupby(["group", "tissue"]).size())

In [ ]:
# ==============================================
# GSE85051 差异分析（limma 双因素交互模型）
# 读取：data/processed/GSE85051/ 下的基因表达矩阵和样本信息
# 输出：output/GSE85051_diff/ 下的差异结果和可视化
# ==============================================
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap

# rpy2 相关
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri, StrVector
from rpy2.robjects.conversion import localconverter
from rpy2.robjects.packages import importr, isinstalled

# 导入 config
from config import DATA_PROCESSED, RESULT_DIR

# 安装/加载 limma
utils = importr('utils')
if not isinstalled('limma'):
    print("正在安装 limma ...")
    utils.install_packages('BiocManager', quiet=True)
    ro.r('BiocManager::install("limma", update=FALSE, ask=FALSE)')
limma = importr('limma')
stats = importr('stats')

# ===================== 路径配置 =====================
# 从 processed 读取表达矩阵和样本信息
PROCESSED_DIR = Path(DATA_PROCESSED) / "GSE85051"
EXPR_FILE = PROCESSED_DIR / "gene_expression_matrix.csv"
SAMPLE_FILE = PROCESSED_DIR / "sample_info.csv"

# 差异分析结果输出到 output
OUTPUT_DIR = Path(RESULT_DIR) / "GSE85051_diff"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ===================== 1. 读取数据 =====================
print("="*50)
print("读取表达矩阵和样本信息...")
print("="*50)
expr_df = pd.read_csv(EXPR_FILE, index_col=0)          # 基因 × 样本（正确格式，无需转置）
sample_info = pd.read_csv(SAMPLE_FILE)                 # sample_name, group, tissue
print(f"✅ 表达矩阵：{expr_df.shape[0]} 基因 × {expr_df.shape[1]} 样本")
print(f"✅ 样本信息：{sample_info.shape[0]} 个样本")

# 确保样本顺序一致（关键：表达矩阵列 = 样本信息行）
sample_info = sample_info.set_index('sample_name').loc[expr_df.columns].reset_index()
print("\n样本分组统计：")
print(sample_info.groupby(['group', 'tissue']).size())

# ===================== 2. 准备数据并转换到 R =====================
group_vector = StrVector(sample_info['group'].tolist())
tissue_vector = StrVector(sample_info['tissue'].tolist())

# 直接使用 基因×样本 的原始矩阵
with localconverter(ro.default_converter + pandas2ri.converter):
    expr_r = ro.conversion.py2rpy(expr_df)

ro.globalenv['expr_mat'] = ro.r('as.matrix')(expr_r)
ro.globalenv['group'] = group_vector
ro.globalenv['tissue'] = tissue_vector

# ===================== 3. 运行 limma 线性模型 =====================
ro.r('''
    library(limma)
    group <- factor(group)
    tissue <- factor(tissue)
    # 打印因子水平，确认参考组
    cat("group 因子水平：\n"); print(levels(group))
    cat("tissue 因子水平：\n"); print(levels(tissue))
    
    design <- model.matrix(~ group * tissue)
    colnames(design) <- make.names(colnames(design))
    # 打印设计矩阵列名（核心调试）
    cat("\n设计矩阵列名：\n"); print(colnames(design))
    
    fit <- lmFit(expr_mat, design)
    fit <- eBayes(fit)
''')

# ===================== 4. 构建对比矩阵 =====================
# 修正原因：Contralateral 是参考组，无groupContralateral列，系数=0
ro.r('''
    cont.matrix <- makeContrasts(
        # 淋巴结 (LN) 对比：Contralateral为参考组，直接用0替代
        Injured_vs_Control_LN = groupInjured - groupControl,
        Contralateral_vs_Control_LN = 0 - groupControl,
        
        # 滑膜组织 (ST) 对比
        Injured_vs_Control_ST = (groupInjured + groupInjured.tissueSynovialTissue) - (groupControl + groupControl.tissueSynovialTissue),
        Contralateral_vs_Control_ST = 0 - (groupControl + groupControl.tissueSynovialTissue),
        
        # 交互效应
        Interaction_Injured_STvsLN = groupInjured.tissueSynovialTissue - groupControl.tissueSynovialTissue,
        
        levels = design
    )
    fit2 <- contrasts.fit(fit, cont.matrix)
    fit2 <- eBayes(fit2)
''')

# ===================== 5. 提取结果函数 =====================
def get_results(contrast_name):
    with localconverter(ro.default_converter + pandas2ri.converter):
        ro.globalenv['contrast'] = contrast_name
        ro.r(f'''
            res <- topTable(fit2, coef="{contrast_name}", number=Inf, sort.by="none")
            res_df <- as.data.frame(res)
        ''')
        res_pd = ro.globalenv['res_df']
    res_pd = res_pd.reset_index().rename(columns={'index': 'Gene'})
    return res_pd

contrasts_list = [
    "Injured_vs_Control_LN",
    "Contralateral_vs_Control_LN",
    "Injured_vs_Control_ST",
    "Contralateral_vs_Control_ST",
    "Interaction_Injured_STvsLN"
]

for cname in contrasts_list:
    print(f"正在提取对比：{cname}")
    df_res = get_results(cname)
    df_res.to_csv(OUTPUT_DIR / f"{cname}_results.csv", index=False)
    sig = (df_res['adj.P.Val'] < 0.05) & (abs(df_res['logFC']) > 1)
    up = df_res[sig & (df_res['logFC'] > 0)]
    down = df_res[sig & (df_res['logFC'] < 0)]
    print(f"  显著上调: {len(up)}, 显著下调: {len(down)}")
    up.to_csv(OUTPUT_DIR / f"{cname}_upregulated.csv", index=False)
    down.to_csv(OUTPUT_DIR / f"{cname}_downregulated.csv", index=False)

# ===================== 6. 火山图（以 Injured_vs_Control_LN 为例）=====================
print("\n" + "="*50)
print("绘制火山图...")
print("="*50)
res_ln = pd.read_csv(OUTPUT_DIR / "Injured_vs_Control_LN_results.csv")
res_ln['-log10(P.Value)'] = -np.log10(res_ln['P.Value'])
res_ln['Sig'] = 'Not significant'
res_ln.loc[(res_ln['adj.P.Val'] < 0.05) & (res_ln['logFC'] > 1), 'Sig'] = 'Up-regulated'
res_ln.loc[(res_ln['adj.P.Val'] < 0.05) & (res_ln['logFC'] < -1), 'Sig'] = 'Down-regulated'

fig, ax = plt.subplots(figsize=(10, 8))
colors = {'Up-regulated': '#e64343', 'Down-regulated': '#436eee', 'Not significant': 'lightgray'}
for s, c in colors.items():
    subset = res_ln[res_ln['Sig'] == s]
    ax.scatter(subset['logFC'], subset['-log10(P.Value)'], c=c, s=15, alpha=0.7, label=s)
ax.axvline(x=1, linestyle='--', color='black', lw=1)
ax.axvline(x=-1, linestyle='--', color='black', lw=1)
ax.axhline(y=-np.log10(0.05), linestyle='--', color='black', lw=1)
ax.set_xlabel('log₂(Fold Change)', fontsize=12)
ax.set_ylabel('-log₁₀(P-value)', fontsize=12)
ax.set_title('GSE85051 Volcano Plot (Injured vs Control, LymphNode)', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "Volcano_Injured_vs_Control_LN.png", dpi=300)
plt.close()
print("✅ 火山图已保存")

# ===================== 7. 热图（Top 50 差异基因）=====================
print("\n" + "="*50)
print("绘制热图...")
print("="*50)
top_genes = res_ln.sort_values('adj.P.Val').head(50)['Gene'].tolist()
# 重新读取原始表达矩阵（用于热图）
expr_df_orig = pd.read_csv(EXPR_FILE, index_col=0)
heat_data = expr_df_orig.loc[expr_df_orig.index.isin(top_genes), :].copy()
heat_data = heat_data.reindex(top_genes)
# Z-score 标准化
heat_z = heat_data.sub(heat_data.mean(axis=1), axis=0)
heat_z = heat_z.div(heat_data.std(axis=1), axis=0).fillna(0)
# 绘制
cmap = LinearSegmentedColormap.from_list('custom', ['#3b76af', 'white', '#e37a57'])
fig_height = max(10, len(top_genes)*0.3)
fig, ax = plt.subplots(figsize=(14, fig_height))
sns.heatmap(heat_z, cmap=cmap, center=0, xticklabels=True, yticklabels=True,
            cbar_kws={'label': 'Z-score'}, ax=ax)
ax.set_title('GSE85051 Top 50 Differentially Expressed Genes (Injured vs Control, LymphNode)', fontweight='bold')
plt.xticks(rotation=90, fontsize=8)
plt.yticks(fontsize=8)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "Heatmap_Top50_Injured_LN.png", dpi=300)
plt.close()
print("✅ 热图已保存")

print("\n🎉 差异分析及可视化全部完成！")
print(f"📁 结果保存至: {OUTPUT_DIR}")



### 小鼠关键基因和韦恩图



In [ ]:
# ==============================================
# 两个小鼠数据集差异基因交集分析与韦恩图（增强版）
# 适配：GSE234863(DESeq2) + GSE85051(limma) 标准输出文件
# 核心：使用 Ensembl ID 唯一匹配，计算交集/韦恩图
# 输出：交集基因列表、统计表格、Top20基因、韦恩图
# ==============================================
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
from matplotlib_venn import venn2

# 导入 config
from config import RESULT_DIR

# ===================== 路径配置（完全适配你的输出路径） =====================
GSE234863_DIR = os.path.join(RESULT_DIR, "GSE234863")
GSE85051_DIR = os.path.join(RESULT_DIR, "GSE85051_diff")
OUTPUT_DIR = os.path.join(RESULT_DIR, "mouse_common_genes")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ===================== 1. 读取 GSE234863 差异结果（适配列名：ensembl_id, Symbol） =====================
print("="*50)
print("读取 GSE234863 完整差异结果...")
print("="*50)

# 读取DESeq2完整结果
gse234863_full = pd.read_csv(os.path.join(GSE234863_DIR, "GSE234863_DESeq2_full_results.csv"))
# 保留核心列：EnsemblID + 基因名 + 差异统计
gse234863_full = gse234863_full[['ensembl_id', 'Symbol', 'log2FoldChange', 'padj']].copy()
# 统一列名，方便后续匹配
gse234863_full.rename(columns={
    'ensembl_id': 'Ensembl',
    'log2FoldChange': 'log2FC_GSE234863',
    'padj': 'padj_GSE234863'
}, inplace=True)
print(f"✅ GSE234863 总基因数: {len(gse234863_full)}")

# 筛选显著差异基因（padj<0.05，|log2FC|>1）→ 基于Ensembl ID
sig_234863 = gse234863_full[
    (gse234863_full['padj_GSE234863'] < 0.05) &
    (abs(gse234863_full['log2FC_GSE234863']) > 1)
]
genes_all_234863 = set(sig_234863['Ensembl'])
print(f"✅ GSE234863 显著差异基因数: {len(genes_all_234863)}")

# ===================== 2. 读取 GSE85051 差异结果（适配列名：Ensembl_ID, Gene_Symbol） =====================
print("\n" + "="*50)
print("读取 GSE85051 完整差异结果 (Injured_vs_Control_LN)...")
print("="*50)

contrast_name = "Injured_vs_Control_LN"
gse85051_full = pd.read_csv(os.path.join(GSE85051_DIR, f"{contrast_name}_results.csv"))
# 保留核心列：EnsemblID + 基因名 + 差异统计
gse85051_full = gse85051_full[['Ensembl_ID', 'Gene_Symbol', 'logFC', 'adj.P.Val']].copy()
# 统一列名，方便后续匹配
gse85051_full.rename(columns={
    'Ensembl_ID': 'Ensembl',
    'Gene_Symbol': 'Symbol',
    'logFC': 'logFC_GSE85051',
    'adj.P.Val': 'padj_GSE85051'
}, inplace=True)
print(f"✅ GSE85051 总基因数: {len(gse85051_full)}")

# 筛选显著差异基因（padj<0.05，|logFC|>1）→ 基于Ensembl ID
sig_85051 = gse85051_full[
    (gse85051_full['padj_GSE85051'] < 0.05) &
    (abs(gse85051_full['logFC_GSE85051']) > 1)
]
genes_all_85051 = set(sig_85051['Ensembl'])
print(f"✅ GSE85051 显著差异基因数: {len(genes_all_85051)}")

# ===================== 3. 计算共同差异基因（基于 Ensembl ID 唯一匹配） =====================
common_ensembl = genes_all_234863.intersection(genes_all_85051)
print(f"\n✅ 两个数据集共同显著差异基因数 (Ensembl ID): {len(common_ensembl)}")

# 保存共同基因ID列表
pd.DataFrame({"Ensembl": sorted(common_ensembl)}).to_csv(
    os.path.join(OUTPUT_DIR, "common_diff_genes_Ensembl.csv"), index=False
)

# ===================== 4. 合并双数据集完整统计信息 =====================
# 基于Ensembl ID合并两个数据集的差异结果
common_df = gse234863_full[gse234863_full['Ensembl'].isin(common_ensembl)].copy()
common_df = common_df.merge(
    gse85051_full[['Ensembl', 'logFC_GSE85051', 'padj_GSE85051']],
    on='Ensembl',
    how='left'
)

# 按显著性排序并保存
common_df_sorted = common_df.sort_values('padj_GSE234863')
common_df_sorted.to_csv(os.path.join(OUTPUT_DIR, "common_genes_with_stats.csv"), index=False)

# ===================== 5. 提取 Top20 核心交集基因 =====================
top20_df = common_df_sorted.head(20).copy()
top20_df.to_csv(os.path.join(OUTPUT_DIR, "top20_common_genes_with_stats.csv"), index=False)

# 保存基因名（用于KEGG/GO富集分析）
top20_gene_names = top20_df['Symbol'].dropna().tolist()
pd.DataFrame({"GeneSymbol": top20_gene_names}).to_csv(
    os.path.join(OUTPUT_DIR, "top20_common_genes_names.csv"), index=False
)
print(f"\n✅ Top {len(top20_gene_names)} 交集基因已保存")

# ===================== 6. 绘制韦恩图（基于 Ensembl ID 统计） =====================
plt.figure(figsize=(8, 6))
venn2(
    subsets=(
        len(genes_all_234863 - common_ensembl),
        len(genes_all_85051 - common_ensembl),
        len(common_ensembl)
    ),
    set_labels=('GSE234863', 'GSE85051')
)
plt.title('Mouse Differential Expressed Genes Overlap (Ensembl ID)')
plt.savefig(
    os.path.join(OUTPUT_DIR, "Venn_diagram_diff_genes.png"),
    dpi=300,
    bbox_inches='tight'
)
plt.close()

# ===================== 完成输出 =====================
print(f"✅ 韦恩图已保存至 {OUTPUT_DIR}")
print("\n🎉 分析完成！所有输出文件：")
print(" 1. common_diff_genes_Ensembl.csv        → 共同差异基因EnsemblID")
print(" 2. common_genes_with_stats.csv          → 共同基因完整差异统计")
print(" 3. top20_common_genes_with_stats.csv    → Top20显著共同基因")
print(" 4. top20_common_genes_names.csv         → 基因名(富集分析专用)")
print(" 5. Venn_diagram_diff_genes.png          → 韦恩图")

In [ ]:
# ==============================================
# 两个小鼠数据集差异基因交集分析与韦恩图（增强版）
# 输入：
#   GSE234863 (DESeq2) 完整差异结果CSV（位于 output/GSE234863/）
#   GSE85051 (limma) 完整差异结果CSV（位于 output/GSE85051_diff/）
# 输出：
#   每个数据集的差异基因并集CSV，交集CSV（含统计信息），top20交集基因CSV，韦恩图
# ==============================================
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
from matplotlib_venn import venn2

# 导入 config
from config import RESULT_DIR

# ===================== 路径配置 =====================
GSE234863_DIR = os.path.join(RESULT_DIR, "GSE234863")
GSE85051_DIR = os.path.join(RESULT_DIR, "GSE85051_diff")
OUTPUT_DIR = os.path.join(RESULT_DIR, "mouse_common_genes")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ===================== 1. 读取 GSE234863 差异结果（完整） =====================
print("="*50)
print("读取 GSE234863 完整差异结果...")
print("="*50)

# 读取完整差异结果（包含所有基因的统计信息）
gse234863_full = pd.read_csv(os.path.join(GSE234863_DIR, "GSE234863_DESeq2_full_results.csv"))
# 保留必要列，基因名统一大写
gse234863_full['gene_upper'] = gse234863_full['Symbol'].astype(str).str.upper()
gse234863_full = gse234863_full[['gene_upper', 'Symbol', 'log2FoldChange', 'padj']].copy()
gse234863_full.rename(columns={'log2FoldChange': 'log2FC_GSE234863', 'padj': 'padj_GSE234863'}, inplace=True)
print(f"✅ GSE234863 总基因数: {len(gse234863_full)}")

# 提取所有差异基因（显著且|log2FC|>1）的集合（用于韦恩图，与原逻辑一致）
sig_234863 = gse234863_full[(gse234863_full['padj_GSE234863'] < 0.05) & (abs(gse234863_full['log2FC_GSE234863']) > 1)]
genes_all_234863 = set(sig_234863['gene_upper'])
print(f"✅ GSE234863 显著差异基因数: {len(genes_all_234863)}")

# ===================== 2. 读取 GSE85051 差异结果（完整） =====================
print("\n" + "="*50)
print("读取 GSE85051 完整差异结果 (Injured_vs_Control_LN)...")
print("="*50)

contrast_name = "Injured_vs_Control_LN"
gse85051_full = pd.read_csv(os.path.join(GSE85051_DIR, f"{contrast_name}_results.csv"))
# 保留必要列，基因名统一大写
gse85051_full['gene_upper'] = gse85051_full['Gene'].astype(str).str.upper()
gse85051_full = gse85051_full[['gene_upper', 'Gene', 'logFC', 'adj.P.Val']].copy()
gse85051_full.rename(columns={'logFC': 'logFC_GSE85051', 'adj.P.Val': 'padj_GSE85051'}, inplace=True)
print(f"✅ GSE85051 总基因数: {len(gse85051_full)}")

# 提取所有差异基因（显著且|logFC|>1）的集合（用于韦恩图）
sig_85051 = gse85051_full[(gse85051_full['padj_GSE85051'] < 0.05) & (abs(gse85051_full['logFC_GSE85051']) > 1)]
genes_all_85051 = set(sig_85051['gene_upper'])
print(f"✅ GSE85051 显著差异基因数: {len(genes_all_85051)}")

# ===================== 3. 取两个数据集交集（显著差异基因） =====================
common_genes = genes_all_234863.intersection(genes_all_85051)
print(f"\n✅ 两个数据集共同显著差异基因数: {len(common_genes)}")

# 保存简单基因名列表
pd.DataFrame({"Gene": sorted(common_genes)}).to_csv(
    os.path.join(OUTPUT_DIR, "common_diff_genes_between_GSE234863_and_GSE85051.csv"), index=False
)

# ===================== 4. 合并两个数据集的统计信息（交集基因） =====================
# 从GSE234863中提取交集基因的详细信息
common_df = gse234863_full[gse234863_full['gene_upper'].isin(common_genes)].copy()
# 合并GSE85051的信息
common_df = common_df.merge(
    gse85051_full[['gene_upper', 'logFC_GSE85051', 'padj_GSE85051']],
    on='gene_upper', how='left'
)
# 移除辅助列
common_df = common_df.drop(columns=['gene_upper'])
# 按GSE234863的padj升序排序（最显著在前）
common_df_sorted = common_df.sort_values('padj_GSE234863')
# 保存完整交集统计信息
common_df_sorted.to_csv(os.path.join(OUTPUT_DIR, "common_genes_with_stats.csv"), index=False)

# ===================== 5. 提取 top20 交集基因（按GSE234863显著性，不足20则全取） =====================
top20_df = common_df_sorted.head(20).copy()
top20_df.to_csv(os.path.join(OUTPUT_DIR, "top20_common_genes_with_stats.csv"), index=False)

# 保存纯基因名列表（供KEGG分析使用）
top20_gene_names = top20_df['Symbol'].tolist()
pd.DataFrame({"Gene": top20_gene_names}).to_csv(
    os.path.join(OUTPUT_DIR, "top20_common_genes_names.csv"), index=False
)

print(f"\n✅ Top {len(top20_gene_names)} 交集基因（按GSE234863 padj升序）已保存")
print("基因列表：", top20_gene_names)

# ===================== 6. 绘制韦恩图（基于全部显著差异基因） =====================
plt.figure(figsize=(8, 6))
venn2(subsets=(len(genes_all_234863 - common_genes),
               len(genes_all_85051 - common_genes),
               len(common_genes)),
      set_labels=('GSE234863', 'GSE85051'))
plt.title('Mouse Differential Expressed Genes Overlap')
plt.savefig(os.path.join(OUTPUT_DIR, "Venn_diagram_diff_genes.png"), dpi=300, bbox_inches='tight')
plt.close()
print(f"✅ 韦恩图已保存至 {os.path.join(OUTPUT_DIR, 'Venn_diagram_diff_genes.png')}")

print("\n🎉 分析完成！所有输出文件保存在：", OUTPUT_DIR)
print("文件列表：")
print("  - GSE234863_all_diff_genes.csv          (仅基因名)")
print("  - GSE85051_all_diff_genes.csv           (仅基因名)")
print("  - common_diff_genes_between_GSE234863_and_GSE85051.csv  (交集基因名)")
print("  - common_genes_with_stats.csv           (交集基因完整统计信息)")
print("  - top20_common_genes_with_stats.csv     (top20交集基因统计信息)")
print("  - top20_common_genes_names.csv          (top20基因名，供KEGG分析)")
print("  - Venn_diagram_diff_genes.png")

In [ ]:
# ==============================================
# 小鼠交集基因的 KEGG 富集分析（全部基因）+ PPI 边列表生成
# 输入：output/mouse_common_genes/top20_common_genes_with_stats.csv
# 输出：KEGG 富集结果表（top20 通路可视化），PPI 边列表
# ==============================================

import os
import sys
import subprocess
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import requests
from pathlib import Path

# 导入 config
from config import RESULT_DIR

# 设置输出目录（小鼠）
OUTPUT_DIR = Path(RESULT_DIR) / "mouse_common_genes"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 输入文件（包含所有小鼠交集基因及其统计信息）
GENE_STATS_FILE = OUTPUT_DIR / "top20_common_genes_with_stats.csv"

# ===================== 辅助函数：自动安装缺失的 Python 包 =====================
def install_pip_package(package):
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])
        print(f"✅ {package} 安装成功")
        return True
    except Exception as e:
        print(f"❌ {package} 安装失败: {e}")
        return False

# ===================== 1. 读取基因列表（全部小鼠交集基因）=====================
print("=" * 50)
print("读取小鼠全部交集基因...")
print("=" * 50)
gene_stats_df = pd.read_csv(GENE_STATS_FILE)
# 假设基因名列是 'Symbol'（小鼠 top20 文件中列名可能是 'Symbol'）
if 'Symbol' in gene_stats_df.columns:
    gene_list = gene_stats_df['Symbol'].dropna().astype(str).tolist()
elif 'Gene' in gene_stats_df.columns:
    gene_list = gene_stats_df['Gene'].dropna().astype(str).tolist()
else:
    # 尝试第一列
    gene_list = gene_stats_df.iloc[:, 0].dropna().astype(str).tolist()
print(f"待分析基因数: {len(gene_list)}")
print(f"基因列表（前10个）: {gene_list[:10]}")

# ===================== 2. 确保 gseapy 已安装（作为备选） =====================
try:
    from gseapy import enrichr
    gseapy_available = True
except ImportError:
    print("⚠️ gseapy 未安装，正在尝试自动安装...")
    if install_pip_package("gseapy"):
        from gseapy import enrichr
        gseapy_available = True
    else:
        gseapy_available = False
        print("❌ gseapy 安装失败，将无法使用备选方案。")

# ===================== 3. 尝试使用 clusterProfiler (via rpy2) =====================
print("\n" + "=" * 50)
print("开始 KEGG 富集分析...")
print("=" * 50)

use_clusterProfiler = False
result_df = pd.DataFrame()

# 尝试导入 rpy2
try:
    import rpy2.robjects as ro
    from rpy2.robjects import pandas2ri
    from rpy2.robjects.conversion import localconverter
    from rpy2.robjects.packages import importr, isinstalled
    rpy2_available = True
except ImportError:
    rpy2_available = False
    print("⚠️ rpy2 未安装，无法使用 clusterProfiler。将直接使用 gseapy。")

if rpy2_available:
    # 检查小鼠相关的 R 包
    if isinstalled('clusterProfiler') and isinstalled('org.Mm.eg.db'):
        use_clusterProfiler = True
        print("✅ 检测到 clusterProfiler 和 org.Mm.eg.db，将使用 R 版本进行富集分析。")
    else:
        print("⚠️ clusterProfiler 或 org.Mm.eg.db 未安装。")
        print("请手动在 R 环境中执行以下命令安装（建议在终端中启动 R）：")
        print("    install.packages('BiocManager')")
        print("    BiocManager::install(c('clusterProfiler', 'org.Mm.eg.db'))")
        print("将改用 gseapy 进行富集分析。")

# ---------------------- 使用 clusterProfiler（小鼠）---------------------
if use_clusterProfiler:
    print("正在使用 clusterProfiler 进行小鼠 KEGG 富集分析...")
    ro.r('options(stringsAsFactors = FALSE)')
    clusterProfiler = importr('clusterProfiler')
    org_Mm_eg_db = importr('org.Mm.eg.db')
    gene_vector = ro.StrVector(gene_list)

    try:
        enrich_result = clusterProfiler.enrichKEGG(
            gene=gene_vector,
            organism='mmu',          # 小鼠代码
            keyType='kegg',
            pvalueCutoff=0.05,
            qvalueCutoff=0.05
        )
        if enrich_result is not None and len(enrich_result) > 0:
            with localconverter(ro.default_converter + pandas2ri.converter):
                result_df = ro.conversion.rpy2py(enrich_result).copy()
            result_df.to_csv(OUTPUT_DIR / "kegg_enrichment_results_full.csv", index=False)
            print(f"✅ KEGG 富集结果已保存，共有 {len(result_df)} 条显著通路")
        else:
            result_df = pd.DataFrame()
            print("⚠️ 未发现显著富集的通路")
    except Exception as e:
        print(f"❌ clusterProfiler 分析出错: {e}")
        result_df = pd.DataFrame()

# ---------------------- 使用 gseapy (备选，小鼠) ----------------------
if not use_clusterProfiler and gseapy_available:
    print("正在使用 gseapy 进行小鼠 KEGG 富集分析...")
    try:
        enr = enrichr(gene_list=gene_list,
                      gene_sets='KEGG_2019_Mouse',   # 小鼠数据库
                      organism='mouse',
                      outdir=None,
                      cutoff=0.05)
        if enr.results.empty:
            result_df = pd.DataFrame()
            print("⚠️ 未发现显著富集的通路")
        else:
            result_df = enr.results.copy()
            print("gseapy 返回的列名:", result_df.columns.tolist())
            
            rename_dict = {}
            if 'Term' in result_df.columns:
                rename_dict['Term'] = 'Description'
            if 'Adjusted P-value' in result_df.columns:
                rename_dict['Adjusted P-value'] = 'p.adjust'
            if 'Genes' in result_df.columns:
                rename_dict['Genes'] = 'geneID'
            elif 'Overlap' in result_df.columns:
                rename_dict['Overlap'] = 'geneID'
            result_df.rename(columns=rename_dict, inplace=True)
            
            if 'geneID' not in result_df.columns:
                result_df['geneID'] = ''
            result_df['Count'] = result_df['geneID'].apply(lambda x: len(x.split(';')) if isinstance(x, str) and x else 0)
            
            result_df.to_csv(OUTPUT_DIR / "kegg_enrichment_results_full.csv", index=False)
            print(f"✅ KEGG 富集结果已保存，共有 {len(result_df)} 条显著通路")
    except Exception as e:
        print(f"❌ gseapy 分析出错: {e}")
        result_df = pd.DataFrame()

elif not use_clusterProfiler and not gseapy_available:
    print("❌ 既无法使用 clusterProfiler 也无法使用 gseapy。")
    print("请手动安装 gseapy: pip install gseapy")
    print("或者手动安装 R 包后重试。")

# ===================== 4. 可视化 KEGG 结果（显示 top20 通路） =====================
if not result_df.empty:
    # 按 p.adjust 排序，取前20（若不足20则全部）
    result_df_sorted = result_df.sort_values('p.adjust')
    top_n = min(20, len(result_df_sorted))
    plot_df = result_df_sorted.head(top_n).copy().sort_values('Count')
    
    # 条形图
    fig, ax = plt.subplots(figsize=(10, 8))
    ax.barh(plot_df['Description'], plot_df['Count'], color='steelblue')
    ax.set_xlabel('Gene Count')
    ax.set_ylabel('Pathway')
    ax.set_title(f'Top {top_n} KEGG Pathways Enriched (Mouse)')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "kegg_barplot.png", dpi=300, bbox_inches='tight')
    plt.close()
    print("✅ KEGG 条形图已保存: kegg_barplot.png")

    # 气泡图
    fig, ax = plt.subplots(figsize=(12, 8))
    if 'GeneRatio' in plot_df.columns:
        x_vals = plot_df['GeneRatio'].apply(lambda x: eval(x) if isinstance(x, str) else x)
    else:
        x_vals = plot_df['Count'] / plot_df['Count'].max()
    scatter = ax.scatter(x_vals, plot_df['Description'],
                         s=plot_df['Count'] * 20,
                         c=-np.log10(plot_df['p.adjust']),
                         cmap='viridis', alpha=0.7)
    ax.set_xlabel('Gene Ratio')
    ax.set_ylabel('Pathway')
    ax.set_title(f'Top {top_n} KEGG Pathways (Bubble Chart)')
    plt.colorbar(scatter, ax=ax, label='-log10(p.adjust)')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "kegg_bubble.png", dpi=300, bbox_inches='tight')
    plt.close()
    print("✅ KEGG 气泡图已保存: kegg_bubble.png")

    # 保存通路基因列表
    with open(OUTPUT_DIR / "pathway_genes.txt", 'w') as f:
        for idx, row in plot_df.iterrows():
            genes = row['geneID'].split(';') if ';' in row['geneID'] else row['geneID'].split('/')
            f.write(f"{row['Description']}\t{','.join(genes)}\n")
    print("✅ 通路基因列表已保存: pathway_genes.txt")
else:
    print("⚠️ 没有显著通路，跳过可视化。")

# ===================== 5. PPI 网络表格生成（使用全部交集基因，小鼠）=====================
print("\n" + "=" * 50)
print("生成 PPI 网络边列表（通过 STRING API）...")
print("=" * 50)

def get_string_interactions(gene_list, species=10090, score_threshold=400):
    identifiers = "\n".join(gene_list)
    url = "https://string-db.org/api/tsv/network"
    params = {
        "identifiers": identifiers,
        "species": species,
        "required_score": score_threshold,
        "add_annotation_nodes": 0
    }
    response = requests.post(url, data=params)
    if response.status_code != 200:
        print(f"API 请求失败，状态码 {response.status_code}")
        return pd.DataFrame()
    
    lines = response.text.strip().split('\n')
    if len(lines) <= 1:
        print("未找到相互作用数据")
        return pd.DataFrame()
    
    header = lines[0].split('\t')
    print(f"API 返回的列名: {header[:10]}")
    
    def find_col(possible_names):
        for name in possible_names:
            if name in header:
                return header.index(name)
        return None
    
    idx_a = find_col(['preferredName_A', 'node1'])
    idx_b = find_col(['preferredName_B', 'node2'])
    idx_score = find_col(['combined_score', 'score'])
    
    if idx_a is None or idx_b is None or idx_score is None:
        print(f"无法找到必要的列。尝试的列名: preferredName_A/preferredName_B/combined_score 不在 {header}")
        print("前3行数据:")
        for i in range(1, min(4, len(lines))):
            print(lines[i])
        return pd.DataFrame()
    
    edges = []
    for line in lines[1:]:
        row = line.split('\t')
        if len(row) > max(idx_a, idx_b, idx_score):
            edges.append({
                'node1': row[idx_a],
                'node2': row[idx_b],
                'score': float(row[idx_score])
            })
    return pd.DataFrame(edges)

# 使用全部交集基因，小鼠物种代码 10090
ppi_edges = get_string_interactions(gene_list, species=10090, score_threshold=400)

if not ppi_edges.empty:
    ppi_edges.to_csv(OUTPUT_DIR / "ppi_edges.csv", index=False)
    print(f"✅ PPI 边列表已保存，共 {len(ppi_edges)} 条边")
    nodes = set(ppi_edges['node1']).union(set(ppi_edges['node2']))
    pd.DataFrame({'Gene': sorted(nodes)}).to_csv(OUTPUT_DIR / "ppi_nodes.csv", index=False)
    print(f"✅ PPI 节点列表已保存，共 {len(nodes)} 个节点")
else:
    print("⚠️ 未获取到 PPI 数据，可能原因：")
    print("  1. 基因列表在 STRING 中无相互作用")
    print("  2. API 返回格式异常（已打印列名，请检查）")
    pd.DataFrame({'Gene': gene_list}).to_csv(OUTPUT_DIR / "gene_list_for_ppi.csv", index=False)
    print("✅ 已将基因列表保存为 gene_list_for_ppi.csv，可提交至 STRING 在线工具手动查询。")

print("\n🎉 分析完成！所有输出文件保存在：", OUTPUT_DIR)
print("文件列表：")
print("  - kegg_enrichment_results_full.csv   (KEGG 完整富集结果)")
print("  - kegg_barplot.png                    (KEGG 条形图，top20通路)")
print("  - kegg_bubble.png                     (KEGG 气泡图，top20通路)")
print("  - pathway_genes.txt                   (显著通路包含的基因)")
print("  - ppi_edges.csv                       (PPI 网络边列表，用于 Cytoscape)")
print("  - ppi_nodes.csv                       (PPI 网络节点列表)")
print("  - gene_list_for_ppi.csv               (基因列表，供备用)")

In [ ]:
# ==============================================
# 小鼠 PPI 全网络分析：基于全部交集基因的 PPI 边列表
# 输入：output/mouse_common_genes/ppi_edges.csv
# 输出：节点指标表，核心基因列表，完整网络图，GraphML 文件
# ==============================================

import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from pathlib import Path
from config import RESULT_DIR

# 设置输出目录（小鼠）
OUTPUT_DIR = Path(RESULT_DIR) / "mouse_common_genes"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 输入文件（由前面的代码生成，包含全部交集基因的 PPI 边）
EDGES_FILE = OUTPUT_DIR / "ppi_edges.csv"

# ===================== 1. 读取边列表并构建网络 =====================
print("=" * 50)
print("读取小鼠 PPI 边列表（全部节点）...")
print("=" * 50)

edges_df = pd.read_csv(EDGES_FILE)
print(f"原始边数: {len(edges_df)}")
print(edges_df.head())

# 创建无向图
G = nx.Graph()
for idx, row in edges_df.iterrows():
    G.add_edge(row['node1'], row['node2'], weight=row['score'])

print(f"网络节点数: {G.number_of_nodes()}")
print(f"网络边数: {G.number_of_edges()}")

# ===================== 2. 计算拓扑指标 =====================
print("\n" + "=" * 50)
print("计算节点拓扑指标...")
print("=" * 50)

degree_centrality = nx.degree_centrality(G)
betweenness_centrality = nx.betweenness_centrality(G, normalized=True)
closeness_centrality = nx.closeness_centrality(G)
clustering_coeff = nx.clustering(G)
degree_raw = dict(G.degree())

node_metrics = []
for node in G.nodes():
    node_metrics.append({
        'Gene': node,
        'Degree': degree_raw[node],
        'Degree_Centrality': degree_centrality[node],
        'Betweenness_Centrality': betweenness_centrality[node],
        'Closeness_Centrality': closeness_centrality[node],
        'Clustering_Coefficient': clustering_coeff[node]
    })

metrics_df = pd.DataFrame(node_metrics).sort_values('Degree_Centrality', ascending=False)
metrics_df.to_csv(OUTPUT_DIR / "ppi_node_metrics_full.csv", index=False)
print(f"✅ 节点指标已保存: ppi_node_metrics_full.csv")
print(metrics_df.head(10))

# ===================== 3. 输出核心基因列表（Top 10 by Degree） =====================
top_genes = metrics_df.head(10)['Gene'].tolist()
pd.DataFrame({'Gene': top_genes}).to_csv(OUTPUT_DIR / "top10_hub_genes_full.csv", index=False)
print(f"✅ Top 10 核心基因已保存: top10_hub_genes_full.csv")
print("核心基因:", top_genes)

# ===================== 4. 绘制完整网络图（所有节点） =====================
print("\n" + "=" * 50)
print("绘制完整 PPI 网络图（所有节点）...")
print("=" * 50)

num_nodes = G.number_of_nodes()
num_edges = G.number_of_edges()

# 动态调整图形大小和节点大小
fig_size = max(10, min(30, num_nodes // 20))   # 节点越多图越大
node_size = max(30, min(200, 5000 // max(1, num_nodes // 50)))
font_size = max(4, min(10, 200 // max(1, num_nodes // 30)))

plt.figure(figsize=(fig_size, fig_size))

# 使用 spring 布局，迭代次数增加以提高布局质量
pos = nx.spring_layout(G, k=2 / num_nodes**0.3, iterations=100, seed=42)

# 节点颜色按度中心性映射
node_colors = [degree_centrality[node] for node in G.nodes()]
# 边宽度按权重映射（权重范围 0-1000）
max_weight = max([G[u][v]['weight'] for u, v in G.edges()]) if G.edges() else 1
edge_widths = [G[u][v]['weight'] / max_weight * 2 for u, v in G.edges()]

nx.draw_networkx_nodes(G, pos, node_size=node_size, node_color=node_colors, cmap='coolwarm', alpha=0.9)
nx.draw_networkx_edges(G, pos, width=edge_widths, alpha=0.5, edge_color='gray')

# 标签：如果节点太多，只标注度数最高的前50个基因，避免重叠
if num_nodes <= 100:
    nx.draw_networkx_labels(G, pos, font_size=font_size, font_family='sans-serif')
else:
    top_degree_nodes = sorted(G.degree, key=lambda x: x[1], reverse=True)[:50]
    top_nodes = [node for node, deg in top_degree_nodes]
    labels = {node: node for node in top_nodes}
    nx.draw_networkx_labels(G, pos, labels=labels, font_size=font_size, font_family='sans-serif')

plt.title(f'Mouse PPI Network (All {num_nodes} nodes, {num_edges} edges)', fontsize=14)
plt.axis('off')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "ppi_network_full_all_nodes.png", dpi=300, bbox_inches='tight')
plt.close()
print("✅ 完整网络图已保存: ppi_network_full_all_nodes.png")

# ===================== 5. 保存完整网络文件（GraphML 格式，可导入 Cytoscape） =====================
nx.write_graphml(G, OUTPUT_DIR / "ppi_network_full.graphml")
print("✅ 完整网络文件已保存: ppi_network_full.graphml (可导入 Cytoscape 进行高级分析)")

print("\n🎉 小鼠 PPI 全网络分析完成！所有输出文件保存在：", OUTPUT_DIR)
print("文件列表：")
print("  - ppi_node_metrics_full.csv        (所有节点的拓扑指标)")
print("  - top10_hub_genes_full.csv         (Top10 核心基因)")
print("  - ppi_network_full_all_nodes.png   (完整网络可视化图，包含所有节点)")
print("  - ppi_network_full.graphml         (完整网络文件，用于 Cytoscape)")